In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

# Existing project root
ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

# Notebook 06 gets its own output directory.
COMBINED_DIR = (
    ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

# Existing dataset audit notebooks
AUDIT_NOTEBOOKS = {
    "HF_Lung_V1": ROOT / "Notebooks 2"
        / "01_HF_Lung_V1_Dataset_Audit.ipynb",
    "Fraiwan": ROOT / "Notebooks 2"
        / "01_Fraiwan_Dataset_Audit.ipynb",
}

print("Project root:", ROOT)
print("Project root exists:", ROOT.exists())
print("Notebook 06 output directory:", COMBINED_DIR)
print("\nAudit notebook checks:")

for dataset, path in AUDIT_NOTEBOOKS.items():
    print(f"{dataset}: {path.exists()} | {path}")

print("\nExisting eight-class disease labels:")
CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]
print(CLASS_NAMES)


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Project root exists: True
Notebook 06 output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1

Audit notebook checks:
HF_Lung_V1: True | /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2/01_HF_Lung_V1_Dataset_Audit.ipynb
Fraiwan: True | /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2/01_Fraiwan_Dataset_Audit.ipynb

Existing eight-class disease labels:
['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']


In [2]:

import json
import re

def inspect_audit_notebook(name, path):
    print("\n" + "=" * 80)
    print(f"AUDIT: {name}")
    print("=" * 80)

    if not path.exists():
        print("Notebook not found:", path)
        return

    with open(path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    terms = re.compile(
        r"diagnos|disease|label|class|patient|subject|metadata|"
        r"annotation|csv|xlsx|healthy|normal|asthma|copd|"
        r"pneumonia|bronch|urti|lrti|hf lung|icbhi",
        re.IGNORECASE,
    )

    for index, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))
        outputs = []

        for output in cell.get("outputs", []):
            if "text" in output:
                outputs.append("".join(output["text"]))

            data = output.get("data", {})
            if "text/plain" in data:
                outputs.append("".join(data["text/plain"]))

        # Show relevant code cells and outputs that were saved.
        relevant_source = bool(terms.search(source))
        relevant_output = any(terms.search(text) for text in outputs)

        if relevant_source or relevant_output:
            print(f"\n--- Cell {index} ({cell.get('cell_type')}) ---")

            if relevant_source:
                print("[CODE]")
                print(source[:1800])

            if relevant_output:
                print("[SAVED OUTPUT]")
                for text in outputs:
                    if terms.search(text):
                        print(text[:2500])

inspect_audit_notebook(
    "HF Lung V1",
    AUDIT_NOTEBOOKS["HF_Lung_V1"],
)

inspect_audit_notebook(
    "Fraiwan",
    AUDIT_NOTEBOOKS["Fraiwan"],
)



AUDIT: HF Lung V1

--- Cell 0 (code) ---
[CODE]

# Cell 1 — Locate HF Lung V1 within the project data directory

from pathlib import Path

PROJECT_ROOT = Path("/Users/vs/Sem 5/Respiratory-Disease-Classification")
DATA_ROOT = PROJECT_ROOT / "data" / "raw"

print("Project root exists:", PROJECT_ROOT.exists())
print("Raw data root exists:", DATA_ROOT.exists())

print("\n========== RAW DATA ROOTS ==========")
if DATA_ROOT.exists():
    for path in sorted(DATA_ROOT.iterdir()):
        kind = "DIR " if path.is_dir() else "FILE"
        print(f"{kind} {path.name}")

print("\n========== POSSIBLE HF LUNG V1 LOCATIONS ==========")
matches = []

if DATA_ROOT.exists():
    for path in DATA_ROOT.rglob("*"):
        if path.is_dir():
            name = path.name.casefold().replace("_", " ").replace("-", " ")
            if "lung" in name or name in {"hf", "hf lung v1"}:
                matches.append(path)

if matches:
    for path in matches:
        print(path)
else:
    print("No likely HF Lung 

In [3]:

from pathlib import Path
import re

ICBHI_ROOT = ROOT / "data" / "raw" / "ICBHI"

print("ICBHI root:", ICBHI_ROOT)
print("Root exists:", ICBHI_ROOT.exists())

if not ICBHI_ROOT.exists():
    print("\nSearching for ICBHI-related directories...")
    for p in (ROOT / "data" / "raw").rglob("*"):
        if p.is_dir() and "icbhi" in p.name.casefold():
            print(p)
else:
    print("\nTop-level contents:")
    for p in sorted(ICBHI_ROOT.iterdir()):
        print(("DIR  " if p.is_dir() else "FILE "), p.name)

    print("\nPotential metadata and diagnosis files:")
    patterns = re.compile(
        r"diagnos|patient|subject|metadata|annotation|"
        r"label|demographic|recording|\.csv$|\.txt$|\.xlsx$|\.tsv$",
        re.IGNORECASE,
    )

    matches = []
    for p in ICBHI_ROOT.rglob("*"):
        if not p.is_file():
            continue

        # Skip audio files and hidden system files.
        if p.name.startswith(".") or p.suffix.lower() in {
            ".wav", ".flac", ".mp3", ".DS_Store".lower()
        }:
            continue

        if patterns.search(p.name):
            matches.append(p)

    for p in matches[:150]:
        print(p.relative_to(ICBHI_ROOT))

    print("\nTotal candidate files:", len(matches))


ICBHI root: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI
Root exists: True

Top-level contents:
FILE  .DS_Store
DIR   .ipynb_checkpoints
FILE  101_1b1_Al_sc_Meditron.txt
FILE  101_1b1_Al_sc_Meditron.wav
FILE  101_1b1_Pr_sc_Meditron.txt
FILE  101_1b1_Pr_sc_Meditron.wav
FILE  102_1b1_Ar_sc_Meditron.txt
FILE  102_1b1_Ar_sc_Meditron.wav
FILE  103_2b2_Ar_mc_LittC2SE.txt
FILE  103_2b2_Ar_mc_LittC2SE.wav
FILE  104_1b1_Al_sc_Litt3200.txt
FILE  104_1b1_Al_sc_Litt3200.wav
FILE  104_1b1_Ar_sc_Litt3200.txt
FILE  104_1b1_Ar_sc_Litt3200.wav
FILE  104_1b1_Ll_sc_Litt3200.txt
FILE  104_1b1_Ll_sc_Litt3200.wav
FILE  104_1b1_Lr_sc_Litt3200.txt
FILE  104_1b1_Lr_sc_Litt3200.wav
FILE  104_1b1_Pl_sc_Litt3200.txt
FILE  104_1b1_Pl_sc_Litt3200.wav
FILE  104_1b1_Pr_sc_Litt3200.txt
FILE  104_1b1_Pr_sc_Litt3200.wav
FILE  105_1b1_Tc_sc_Meditron.txt
FILE  105_1b1_Tc_sc_Meditron.wav
FILE  106_2b1_Pl_mc_LittC2SE.txt
FILE  106_2b1_Pl_mc_LittC2SE.wav
FILE  106_2b1_Pr_mc_LittC2SE.txt
FILE  106_2b1_Pr_

In [4]:
from pathlib import Path
import pandas as pd

ICBHI_ROOT = ROOT / "data" / "raw" / "ICBHI"
METADATA_ROOT = ICBHI_ROOT / "metadata"

print("Metadata folder exists:", METADATA_ROOT.exists())

if METADATA_ROOT.exists():
    for path in sorted(METADATA_ROOT.rglob("*")):
        if path.is_file() and not path.name.startswith("."):
            print(
                f"{path.relative_to(ICBHI_ROOT)} "
                f"({path.stat().st_size:,} bytes)"
            )

print("\n--- ICBHI patient diagnosis summary ---")

# The standard ICBHI dataset commonly includes
# a patient-level diagnostic metadata file.
candidate_files = [
    ICBHI_ROOT / "patient_diagnosis.txt",
    METADATA_ROOT / "patient_diagnosis.txt",
]

diagnosis_file = next(
    (path for path in candidate_files if path.exists()),
    None,
)

if diagnosis_file is None:
    print("patient_diagnosis.txt not found in expected locations.")
    print("We will inspect the metadata files listed above.")
else:
    diagnosis_df = pd.read_csv(
        diagnosis_file,
        sep=r"\s+",
        header=None,
        names=["patient_id", "diagnosis"],
        dtype={"patient_id": str, "diagnosis": str},
    )

    print("Diagnosis file:", diagnosis_file.relative_to(ICBHI_ROOT))
    print("Patient records:", len(diagnosis_df))
    print("Unique patients:", diagnosis_df["patient_id"].nunique())

    print("\nDiagnosis counts:")
    print(diagnosis_df["diagnosis"].value_counts(dropna=False).to_string())

    print("\nPatient diagnosis preview:")
    display(diagnosis_df.head(15))

Metadata folder exists: True
metadata/.ipynb_checkpoints/demographic-checkpoint.csv (3,389 bytes)
metadata/.ipynb_checkpoints/diagnosis-checkpoint.csv (1,389 bytes)
metadata/.ipynb_checkpoints/official_split-checkpoint.csv (26,318 bytes)
metadata/demographic.csv (3,389 bytes)
metadata/diagnosis.csv (1,389 bytes)
metadata/official_split.csv (26,318 bytes)

--- ICBHI patient diagnosis summary ---
patient_diagnosis.txt not found in expected locations.
We will inspect the metadata files listed above.


In [5]:
import pandas as pd

METADATA_ROOT = ROOT / "data" / "raw" / "ICBHI" / "metadata"

diagnosis_path = METADATA_ROOT / "diagnosis.csv"
demographic_path = METADATA_ROOT / "demographic.csv"
split_path = METADATA_ROOT / "official_split.csv"

for name, path in [
    ("Diagnosis", diagnosis_path),
    ("Demographic", demographic_path),
    ("Official split", split_path),
]:
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("File:", path.name)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(8))

        if name == "Diagnosis" and len(df.columns) >= 2:
            print("\nDiagnosis value counts:")
            print(df.iloc[:, -1].value_counts(dropna=False).to_string())


==================== Diagnosis ====================
File: diagnosis.csv
Exists: True
Shape: (126, 2)
Columns: ['patient_id', 'diagnosis']


,patient_id,diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI
5,106,COPD
6,107,COPD
7,108,LRTI



Diagnosis value counts:
diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1

==================== Demographic ====================
File: demographic.csv
Exists: True
Shape: (126, 7)
Columns: ['patient_id', 'diagnosis', 'age', 'sex', 'adult_bmi', 'child_weight', 'child_height']


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
0,101,URTI,3.00,F,NaN,19.0,99.0
1,102,Healthy,0.75,F,NaN,9.8,73.0
2,103,Asthma,70.00,F,33.00,NaN,NaN
3,104,COPD,70.00,F,28.47,NaN,NaN
4,105,URTI,7.00,F,NaN,32.0,135.0
5,106,COPD,73.00,F,21.00,NaN,NaN
6,107,COPD,75.00,F,33.70,NaN,NaN
7,108,LRTI,3.00,M,NaN,NaN,NaN



==================== Official split ====================
File: official_split.csv
Exists: True
Shape: (920, 2)
Columns: ['recording_id', 'split']


,recording_id,split
0,101_1b1_Al_sc_Meditron,test
1,101_1b1_Pr_sc_Meditron,test
2,102_1b1_Ar_sc_Meditron,test
3,103_2b2_Ar_mc_LittC2SE,train
4,104_1b1_Al_sc_Litt3200,test
5,104_1b1_Ar_sc_Litt3200,test
6,104_1b1_Ll_sc_Litt3200,test
7,104_1b1_Lr_sc_Litt3200,test


In [6]:
import pandas as pd

ICBHI_ROOT = ROOT / "data" / "raw" / "ICBHI"
METADATA_ROOT = ICBHI_ROOT / "metadata"

diagnosis_df = pd.read_csv(METADATA_ROOT / "diagnosis.csv")
split_df = pd.read_csv(METADATA_ROOT / "official_split.csv")

# Extract patient ID from the beginning of each recording ID.
split_df["patient_id"] = (
    split_df["recording_id"]
    .astype(str)
    .str.extract(r"^(\d+)_", expand=False)
)

split_df["patient_id"] = pd.to_numeric(
    split_df["patient_id"], errors="coerce"
).astype("Int64")

diagnosis_df["patient_id"] = pd.to_numeric(
    diagnosis_df["patient_id"], errors="coerce"
).astype("Int64")

print("Official split recording counts:")
print(split_df["split"].value_counts(dropna=False).to_string())

print("\nRecording IDs without an extracted patient ID:",
      split_df["patient_id"].isna().sum())

print("Recordings whose patient is missing from diagnosis metadata:",
      (~split_df["patient_id"].isin(diagnosis_df["patient_id"])).sum())

print("\nRecordings per split and diagnosis:")
merged = split_df.merge(
    diagnosis_df[["patient_id", "diagnosis"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)

display(
    pd.crosstab(
        merged["diagnosis"],
        merged["split"],
        margins=True,
        dropna=False,
    )
)

print("\nPatients occurring in both official splits:")
splits_per_patient = merged.groupby("patient_id")["split"].nunique()
print(int((splits_per_patient > 1).sum()))

print("\nUnique patients per split:")
print(
    merged.groupby("split")["patient_id"]
    .nunique()
    .to_string()
)

Official split recording counts:
split
train    539
test     381

Recording IDs without an extracted patient ID: 0
Recordings whose patient is missing from diagnosis metadata: 0

Recordings per split and diagnosis:


split,test,train,All
diagnosis,,,
Asthma,0,1,1
Bronchiectasis,2,14,16
Bronchiolitis,6,7,13
COPD,349,444,793
Healthy,17,18,35
LRTI,0,2,2
Pneumonia,0,37,37
URTI,7,16,23
All,381,539,920



Patients occurring in both official splits:
2

Unique patients per split:
split
test     49
train    79


In [7]:
# Identify patients appearing in both official splits.

patient_split_counts = (
    merged.groupby("patient_id")["split"]
    .nunique()
)

overlap_patients = patient_split_counts[
    patient_split_counts > 1
].index.tolist()

print("Patients appearing in both train and test:",
      overlap_patients)

if overlap_patients:
    overlap_details = (
        merged[merged["patient_id"].isin(overlap_patients)]
        .sort_values(["patient_id", "split", "recording_id"])
        [["patient_id", "diagnosis", "split", "recording_id"]]
    )
    display(overlap_details)
else:
    print("No patient overlap found.")

Patients appearing in both train and test: [156, 218]


,patient_id,diagnosis,split,recording_id
362,156,COPD,test,156_2b3_Al_mc_AKGC417L
365,156,COPD,test,156_2b3_Lr_mc_AKGC417L
366,156,COPD,test,156_2b3_Pl_mc_AKGC417L
369,156,COPD,test,156_5b3_Ar_mc_AKGC417L
370,156,COPD,test,156_5b3_Ll_mc_AKGC417L
372,156,COPD,test,156_5b3_Pl_mc_AKGC417L
374,156,COPD,test,156_8b3_Al_mc_AKGC417L
377,156,COPD,test,156_8b3_Lr_mc_AKGC417L
363,156,COPD,train,156_2b3_Ar_mc_AKGC417L
364,156,COPD,train,156_2b3_Ll_mc_AKGC417L


In [8]:
# Assess a patient-independent correction without modifying the source data.

overlap_summary = (
    merged[merged["patient_id"].isin(overlap_patients)]
    .groupby(["patient_id", "split"])
    .agg(
        recordings=("recording_id", "count"),
        diagnosis=("diagnosis", "first"),
    )
    .reset_index()
)

display(overlap_summary)

print("\nOriginal recording counts:")
print(merged["split"].value_counts().to_string())

# Hypothetical correction: retain the official test assignment for
# overlapping patients and move their training recordings to test.
corrected = merged.copy()

mask = (
    corrected["patient_id"].isin(overlap_patients)
    & (corrected["split"] == "train")
)

print("\nTraining recordings that would move to test:", int(mask.sum()))

corrected.loc[mask, "split"] = "test"

print("\nHypothetical patient-independent recording counts:")
print(corrected["split"].value_counts().to_string())

print("\nRemaining patients in both splits:")
print(
    int(
        (
            corrected.groupby("patient_id")["split"].nunique() > 1
        ).sum()
    )
)

,patient_id,split,recordings,diagnosis
0,156,test,8,COPD
1,156,train,9,COPD
2,218,test,4,COPD
3,218,train,4,COPD



Original recording counts:
split
train    539
test     381

Training recordings that would move to test: 13

Hypothetical patient-independent recording counts:
split
train    526
test     394

Remaining patients in both splits:
0


In [9]:
# Build the ICBHI recording manifest with original and corrected splits.

icbhi_manifest = merged.copy()

icbhi_manifest = icbhi_manifest.rename(
    columns={"split": "official_split"}
)

icbhi_manifest["corrected_split"] = (
    icbhi_manifest["official_split"]
)

move_to_test = (
    icbhi_manifest["patient_id"].isin(overlap_patients)
    & (icbhi_manifest["official_split"] == "train")
)

icbhi_manifest.loc[move_to_test, "corrected_split"] = "test"

icbhi_manifest["source_dataset"] = "ICBHI"
icbhi_manifest["recording_path"] = icbhi_manifest[
    "recording_id"
].map(
    lambda recording_id: str(
        ICBHI_ROOT / f"{recording_id}.wav"
    )
)

# Validate recording files and patient-level separation.
from pathlib import Path

icbhi_manifest["file_exists"] = icbhi_manifest[
    "recording_path"
].map(lambda path: Path(path).is_file())

patients_per_split = (
    icbhi_manifest.groupby("patient_id")["corrected_split"]
    .nunique()
)

print("Manifest rows:", len(icbhi_manifest))
print("Unique recordings:", icbhi_manifest["recording_id"].nunique())
print("Missing audio files:", int((~icbhi_manifest["file_exists"]).sum()))
print("Patients in both corrected splits:",
      int((patients_per_split > 1).sum()))

print("\nCorrected split counts:")
print(icbhi_manifest["corrected_split"].value_counts().to_string())

print("\nDiagnosis counts by corrected split:")
display(
    pd.crosstab(
        icbhi_manifest["diagnosis"],
        icbhi_manifest["corrected_split"],
        margins=True,
    )
)

display(icbhi_manifest.head())

Manifest rows: 920
Unique recordings: 920
Missing audio files: 1
Patients in both corrected splits: 0

Corrected split counts:
corrected_split
train    526
test     394

Diagnosis counts by corrected split:


corrected_split,test,train,All
diagnosis,,,
Asthma,0,1,1
Bronchiectasis,2,14,16
Bronchiolitis,6,7,13
COPD,362,431,793
Healthy,17,18,35
LRTI,0,2,2
Pneumonia,0,37,37
URTI,7,16,23
All,394,526,920


,recording_id,official_split,patient_id,diagnosis,corrected_split,source_dataset,recording_path,file_exists
0,101_1b1_Al_sc_Meditron,test,101,URTI,test,ICBHI,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True
1,101_1b1_Pr_sc_Meditron,test,101,URTI,test,ICBHI,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True
2,102_1b1_Ar_sc_Meditron,test,102,Healthy,test,ICBHI,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True
3,103_2b2_Ar_mc_LittC2SE,train,103,Asthma,train,ICBHI,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True
4,104_1b1_Al_sc_Litt3200,test,104,COPD,test,ICBHI,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,True


In [10]:
# Identify the missing recording and check for alternative filename matches.

missing_rows = icbhi_manifest.loc[
    ~icbhi_manifest["file_exists"],
    ["recording_id", "patient_id", "diagnosis",
     "official_split", "corrected_split", "recording_path"]
]

display(missing_rows)

for recording_id in missing_rows["recording_id"]:
    print(f"\nSearching for: {recording_id}")

    matches = list(ICBHI_ROOT.rglob(f"{recording_id}.*"))

    if matches:
        for path in matches:
            print("Found:", path)
    else:
        print("No matching file found anywhere under the ICBHI directory.")

,recording_id,patient_id,diagnosis,official_split,corrected_split,recording_path
919,226_1b1_Pl_sc_Meditron,226,Pneumonia,train,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



Searching for: 226_1b1_Pl_sc_Meditron
No matching file found anywhere under the ICBHI directory.


In [11]:
# Inspect all recordings for patient 226 and summarize file availability.

patient_226 = icbhi_manifest[
    icbhi_manifest["patient_id"] == 226
][
    ["recording_id", "diagnosis", "official_split",
     "corrected_split", "file_exists", "recording_path"]
]

print("All manifest entries for patient 226:")
display(patient_226)

print("\nAudio availability:")
print(
    icbhi_manifest["file_exists"]
    .value_counts()
    .rename(index={True: "Available", False: "Missing"})
    .to_string()
)

print("\nDiagnosis of the missing recording:")
print(
    icbhi_manifest.loc[
        ~icbhi_manifest["file_exists"], "diagnosis"
    ].value_counts().to_string()
)

All manifest entries for patient 226:


,recording_id,diagnosis,official_split,corrected_split,file_exists,recording_path
917,226_1b1_Al_sc_Meditron,Pneumonia,train,train,True,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
918,226_1b1_Ll_sc_Meditron,Pneumonia,train,train,True,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
919,226_1b1_Pl_sc_Meditron,Pneumonia,train,train,False,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



Audio availability:
file_exists
Available    919
Missing        1

Diagnosis of the missing recording:
diagnosis
Pneumonia    1


In [12]:
# Create the usable ICBHI manifest without modifying raw data.

icbhi_usable = (
    icbhi_manifest.loc[icbhi_manifest["file_exists"]]
    .copy()
    .reset_index(drop=True)
)

# Final validation checks.
assert icbhi_usable["recording_id"].is_unique
assert icbhi_usable["file_exists"].all()

patient_split_counts = (
    icbhi_usable.groupby("patient_id")["corrected_split"]
    .nunique()
)

assert (patient_split_counts <= 1).all()

print("Usable ICBHI recordings:", len(icbhi_usable))
print("Unique patients:", icbhi_usable["patient_id"].nunique())
print("Missing audio files:", int((~icbhi_usable["file_exists"]).sum()))
print("Patients crossing corrected splits:",
      int((patient_split_counts > 1).sum()))

print("\nUsable recordings by split:")
print(icbhi_usable["corrected_split"].value_counts().to_string())

print("\nUsable recordings by diagnosis:")
print(icbhi_usable["diagnosis"].value_counts().to_string())

print("\nColumns:")
print(icbhi_usable.columns.tolist())

Usable ICBHI recordings: 919
Unique patients: 126
Missing audio files: 0
Patients crossing corrected splits: 0

Usable recordings by split:
corrected_split
train    525
test     394

Usable recordings by diagnosis:
diagnosis
COPD              793
Pneumonia          36
Healthy            35
URTI               23
Bronchiectasis     16
Bronchiolitis      13
LRTI                2
Asthma              1

Columns:
['recording_id', 'official_split', 'patient_id', 'diagnosis', 'corrected_split', 'source_dataset', 'recording_path', 'file_exists']


In [15]:
from pathlib import Path
import pandas as pd
import re

FRAIWAN_ROOT = ROOT / "data" / "raw" / "Fraiwan"

wav_files = sorted(FRAIWAN_ROOT.glob("*.wav"))

records = []

for path in wav_files:
    match = re.match(
        r"^([A-Za-z]+\d+)_([^,]+),(.*)$",
        path.stem
    )

    if not match:
        records.append({
            "filename": path.name,
            "patient_id": None,
            "prefix": None,
            "raw_diagnosis": None,
            "parsed": False,
        })
        continue

    patient_id = match.group(1)
    prefix_match = re.match(r"^([A-Za-z]+)", patient_id)

    records.append({
        "filename": path.name,
        "patient_id": patient_id,
        "prefix": prefix_match.group(1) if prefix_match else None,
        "raw_diagnosis": match.group(2).strip(),
        "parsed": True,
    })

fraiwan_inventory = pd.DataFrame(records)

print("Total WAV files:", len(fraiwan_inventory))
print("Successfully parsed:", int(fraiwan_inventory["parsed"].sum()))
print("Unparsed:", int((~fraiwan_inventory["parsed"]).sum()))

print("\nRecording counts by prefix:")
print(fraiwan_inventory["prefix"].value_counts(dropna=False).to_string())

print("\nDistinct diagnosis labels:")
print(
    fraiwan_inventory["raw_diagnosis"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nSubjects represented by multiple prefixes:")
prefixes_per_subject = (
    fraiwan_inventory.dropna(subset=["patient_id"])
    .groupby(fraiwan_inventory["patient_id"].str.extract(
        r"^[A-Za-z]+(\d+)$", expand=False
    ))["prefix"]
    .nunique()
)

print("Numeric subject IDs with multiple prefixes:",
      int((prefixes_per_subject > 1).sum()))

display(fraiwan_inventory.head(10))

Total WAV files: 0


KeyError: 'parsed'

In [14]:
FRAIWAN_ROOT.glob("*.wav")

In [16]:
from pathlib import Path

FRAIWAN_ROOT = ROOT / "data" / "raw" / "Fraiwan"

print("Expected path:", FRAIWAN_ROOT)
print("Path exists:", FRAIWAN_ROOT.exists())
print("Is directory:", FRAIWAN_ROOT.is_dir())

if FRAIWAN_ROOT.exists():
    print("\nTop-level entries:")
    for path in sorted(FRAIWAN_ROOT.iterdir()):
        kind = "DIR " if path.is_dir() else "FILE"
        print(f"{kind}  {path.name}")

    wav_files_recursive = list(FRAIWAN_ROOT.rglob("*.wav"))
    print("\nWAV files found recursively:", len(wav_files_recursive))

    for path in wav_files_recursive[:15]:
        print(path.relative_to(FRAIWAN_ROOT))

# Also check the duplicate uppercase directory.
duplicate_root = ROOT / "data" / "raw" / "FRAIWAN"

print("\nUppercase directory exists:", duplicate_root.exists())

if duplicate_root.exists():
    print("Uppercase WAV files found:",
          len(list(duplicate_root.rglob("*.wav"))))

Expected path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/Fraiwan
Path exists: True
Is directory: True

Top-level entries:
FILE  .DS_Store
DIR   Audio Files
FILE  Data annotation (1).xlsx
FILE  Stethoscope Files.zip

WAV files found recursively: 336
Audio Files/BP33_Asthma,E W,P R M,43,F.wav
Audio Files/DP63_COPD,E W,P R L ,58,F.wav
Audio Files/EP30_N,N,P R M,18,F.wav
Audio Files/DP85_N,N,A R U,33,M.wav
Audio Files/DP71_N,N,P R U,36,M.wav
Audio Files/BP52_Lung Fibrosis,Crep,P L L,76,F.wav
Audio Files/BP75_N,N,P L U,73,F.wav
Audio Files/BP39_heart failure,Crep,P R L ,67,M.wav
Audio Files/EP69_pneumonia,Bronchial,P R L ,64,M.wav
Audio Files/BP46_asthma,E W,P L U,41,F.wav
Audio Files/DP13_Heart Failure,C,P L L,55,M.wav
Audio Files/BP54_heart failure,Crep,P R L ,73,F.wav
Audio Files/DP41_pneumonia,Crep,P R M,51,M.wav
Audio Files/EP15_Asthma,E W,P R U,49,F.wav
Audio Files/DP99_N,N,P R M,50,M.wav

Uppercase directory exists: True
Uppercase WAV files found: 336


In [17]:
import pandas as pd
import re

FRAIWAN_ROOT = ROOT / "data" / "raw" / "Fraiwan"
FRAIWAN_AUDIO_ROOT = FRAIWAN_ROOT / "Audio Files"

wav_files = sorted(FRAIWAN_AUDIO_ROOT.rglob("*.wav"))

if not wav_files:
    raise FileNotFoundError(
        f"No WAV files found under {FRAIWAN_AUDIO_ROOT}"
    )

records = []

for path in wav_files:
    match = re.match(
        r"^([A-Za-z]+\d+)_([^,]+),(.*)$",
        path.stem
    )

    if not match:
        records.append({
            "filename": path.name,
            "recording_path": str(path),
            "patient_id": None,
            "prefix": None,
            "raw_diagnosis": None,
            "parsed": False,
        })
        continue

    patient_id = match.group(1)
    prefix_match = re.match(r"^([A-Za-z]+)", patient_id)

    records.append({
        "filename": path.name,
        "recording_path": str(path),
        "patient_id": patient_id,
        "prefix": prefix_match.group(1) if prefix_match else None,
        "raw_diagnosis": match.group(2).strip(),
        "parsed": True,
    })

fraiwan_inventory = pd.DataFrame(records)

print("Total WAV files:", len(fraiwan_inventory))
print("Successfully parsed:", int(fraiwan_inventory["parsed"].sum()))
print("Unparsed:", int((~fraiwan_inventory["parsed"]).sum()))

print("\nRecording counts by prefix:")
print(fraiwan_inventory["prefix"].value_counts(dropna=False).to_string())

print("\nDistinct diagnosis labels:")
print(
    fraiwan_inventory["raw_diagnosis"]
    .value_counts(dropna=False)
    .to_string()
)

# Compare numeric subject IDs across BP, DP and EP prefixes.
parsed = fraiwan_inventory.dropna(subset=["patient_id"]).copy()
parsed["subject_number"] = parsed["patient_id"].str.extract(
    r"^[A-Za-z]+(\d+)$", expand=False
)

prefixes_per_subject = parsed.groupby("subject_number")["prefix"].nunique()

print(
    "\nNumeric subject IDs represented by multiple prefixes:",
    int((prefixes_per_subject > 1).sum())
)

display(fraiwan_inventory.head(10))

Total WAV files: 336
Successfully parsed: 336
Unparsed: 0

Recording counts by prefix:
prefix
BP    112
DP    112
EP    112

Distinct diagnosis labels:
raw_diagnosis
N                                105
Asthma                            51
heart failure                     45
asthma                            45
COPD                              24
pneumonia                         15
Lung Fibrosis                     12
Heart Failure                      9
BRON                               9
Heart Failure + COPD               6
Plueral Effusion                   6
Asthma and lung fibrosis           3
copd                               3
Heart Failure + Lung Fibrosis      3

Numeric subject IDs represented by multiple prefixes: 112


,filename,recording_path,patient_id,prefix,raw_diagnosis,parsed
0,"BP100_N,N,P R M,70,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP100,BP,N,True
1,"BP101_Asthma,E W,P L M,12,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP101,BP,Asthma,True
2,"BP102_N,N,P L L,41,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP102,BP,N,True
3,"BP103_N,N,P R U,81,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP103,BP,N,True
4,"BP104_Asthma,E W,P L U,45,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP104,BP,Asthma,True
5,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP105,BP,Lung Fibrosis,True
6,"BP106_Asthma,E W,P L U,45,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP106,BP,Asthma,True
7,"BP107_Asthma,E W,P L U,59,F.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP107,BP,Asthma,True
8,"BP108_COPD,E W,P R L ,63,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP108,BP,COPD,True
9,"BP109_N,N,P L M,26,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,BP109,BP,N,True


In [18]:
# Compare Fraiwan recording versions for each numeric subject ID.

fraiwan_compare = fraiwan_inventory.dropna(
    subset=["patient_id"]
).copy()

fraiwan_compare["subject_number"] = (
    fraiwan_compare["patient_id"]
    .str.extract(r"^[A-Za-z]+(\d+)$", expand=False)
)

version_table = fraiwan_compare.pivot_table(
    index="subject_number",
    columns="prefix",
    values="raw_diagnosis",
    aggfunc="first",
    dropna=False,
)

print("Subjects represented:", len(version_table))
print("\nDiagnosis agreement across BP, DP and EP:")

for prefix_a, prefix_b in [("BP", "DP"), ("BP", "EP"), ("DP", "EP")]:
    if prefix_a in version_table.columns and prefix_b in version_table.columns:
        agreement = (
            version_table[prefix_a].str.casefold()
            == version_table[prefix_b].str.casefold()
        )
        print(
            f"{prefix_a} vs {prefix_b}: "
            f"{int(agreement.sum())}/{len(agreement)} agree"
        )

print("\nExamples where the labels differ:")
different = version_table[
    version_table.nunique(axis=1, dropna=True) > 1
]
display(different.head(20))

print("\nExample subject IDs and all three labels:")
display(version_table.head(15))

Subjects represented: 112

Diagnosis agreement across BP, DP and EP:
BP vs DP: 112/112 agree
BP vs EP: 112/112 agree
DP vs EP: 112/112 agree

Examples where the labels differ:


prefix,BP,DP,EP
subject_number,,,



Example subject IDs and all three labels:


prefix,BP,DP,EP
subject_number,,,
1,Asthma,Asthma,Asthma
10,Asthma,Asthma,Asthma
100,N,N,N
101,Asthma,Asthma,Asthma
102,N,N,N
103,N,N,N
104,Asthma,Asthma,Asthma
105,Lung Fibrosis,Lung Fibrosis,Lung Fibrosis
106,Asthma,Asthma,Asthma


In [19]:
import pandas as pd

FRAIWAN_EXCEL = FRAIWAN_ROOT / "Data annotation (1).xlsx"

print("Excel exists:", FRAIWAN_EXCEL.exists())

excel_file = pd.ExcelFile(FRAIWAN_EXCEL)

print("\nSheet names:")
print(excel_file.sheet_names)

for sheet in excel_file.sheet_names:
    df = pd.read_excel(FRAIWAN_EXCEL, sheet_name=sheet)

    print(f"\n{'=' * 20} Sheet: {sheet} {'=' * 20}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(10))

Excel exists: True

Sheet names:
['Sheet1']

==================== Sheet: Sheet1 ====================
Shape: (154, 10)
Columns: ['Age', 'Gender', 'Location', 'Sound type', 'Diagnosis', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M']


,Age,Gender,Location,Sound type,Diagnosis,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M"
0,70.0,M,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,"Insipratory: I, Experiatory: E, Wheezes: W, Cr..."
1,52.0,F,P L L,E W,Asthma,NaN,NaN,NaN,NaN,Crep:crepitations
2,50.0,F,P L L,I E W,Asthma,NaN,NaN,NaN,NaN,NaN
3,72.0,F,P R L,I C,Heart Failure + Lung Fibrosis,NaN,NaN,NaN,NaN,NaN
4,71.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
5,81.0,M,P R L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN
6,65.0,M,P R L,I C E W,Heart Failure + COPD,NaN,NaN,NaN,NaN,NaN
7,70.0,M,P L L,I C B,Plueral Effusion,NaN,NaN,NaN,NaN,NaN
8,59.0,M,P R L,E W,Asthma,NaN,NaN,NaN,NaN,NaN
9,59.0,M,P R U,E W,Asthma,NaN,NaN,NaN,NaN,NaN


In [20]:
# Compare Fraiwan Excel annotations against BP recording metadata.

FRAIWAN_EXCEL = FRAIWAN_ROOT / "Data annotation (1).xlsx"

excel_df = pd.read_excel(FRAIWAN_EXCEL)
annotation_df = excel_df.dropna(
    subset=["Diagnosis", "Age", "Gender"]
).copy().reset_index(drop=True)

bp_df = fraiwan_inventory[
    fraiwan_inventory["prefix"] == "BP"
].copy()

bp_df["subject_number"] = bp_df["patient_id"].str.extract(
    r"BP(\d+)", expand=False
).astype(int)

print("Excel rows:", len(excel_df))
print("Rows with diagnosis, age and gender:", len(annotation_df))
print("BP recordings:", len(bp_df))

print("\nFirst 15 annotation diagnoses:")
print(annotation_df["Diagnosis"].head(15).to_list())

print("\nLast 15 populated annotation rows:")
display(annotation_df.tail(15))

print("\nDiagnosis counts in populated Excel rows:")
print(annotation_df["Diagnosis"].value_counts(dropna=False).to_string())

print("\nBP subject-number range:")
print(
    "Minimum:", bp_df["subject_number"].min(),
    "| Maximum:", bp_df["subject_number"].max()
)

Excel rows: 154
Rows with diagnosis, age and gender: 112
BP recordings: 112

First 15 annotation diagnoses:
['Asthma', 'Asthma', 'Asthma', 'Heart Failure + Lung Fibrosis ', 'Heart Failure + COPD', 'Plueral Effusion', 'Heart Failure + COPD', 'Plueral Effusion', 'Asthma', 'Asthma', 'Heart Failure', 'Asthma', 'Heart Failure', 'Heart Failure', 'Asthma']

Last 15 populated annotation rows:


,Age,Gender,Location,Sound type,Diagnosis,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"Posterior: P Lower: L Left: L Right R, UPPER: U, ANTERIOR: A, MIDDLE: M"
97,53.0,M,P L L,N,N,NaN,NaN,NaN,NaN,NaN
98,50.0,M,P R M,N,N,NaN,NaN,NaN,NaN,NaN
99,70.0,F,P R M,N,N,NaN,NaN,NaN,NaN,NaN
100,12.0,F,P L M,E W,Asthma,NaN,NaN,NaN,NaN,NaN
101,41.0,M,P L L,N,N,NaN,NaN,NaN,NaN,NaN
102,81.0,F,P R U,N,N,NaN,NaN,NaN,NaN,NaN
103,45.0,F,P L U,E W,Asthma,NaN,NaN,NaN,NaN,NaN
104,44.0,M,A R U,Crep,Lung Fibrosis,NaN,NaN,NaN,NaN,NaN
105,45.0,F,P L U,E W,Asthma,NaN,NaN,NaN,NaN,NaN
106,59.0,F,P L U,E W,Asthma,NaN,NaN,NaN,NaN,NaN



Diagnosis counts in populated Excel rows:
Diagnosis
N                                 35
Asthma                            17
heart failure                     15
asthma                            15
COPD                               8
pneumonia                          5
Lung Fibrosis                      4
Heart Failure                      3
BRON                               3
Heart Failure + COPD               2
Plueral Effusion                   2
Heart Failure + Lung Fibrosis      1
Asthma and lung fibrosis           1
copd                               1

BP subject-number range:
Minimum: 1 | Maximum: 112


In [21]:
# Validate Excel annotation against BP filename metadata.

import re
import pandas as pd

excel_df = pd.read_excel(FRAIWAN_ROOT / "Data annotation (1).xlsx")
annotation_df = excel_df.dropna(
    subset=["Diagnosis", "Age", "Gender"]
).reset_index(drop=True)

bp_df = fraiwan_inventory[
    fraiwan_inventory["prefix"] == "BP"
].copy()

def parse_bp_metadata(filename):
    stem = Path(filename).stem
    match = re.match(
        r"^BP(\d+)_([^,]+),([^,]+),([^,]+),([^,]+)$",
        stem
    )

    if not match:
        return None

    return {
        "subject_number": int(match.group(1)),
        "filename_diagnosis": match.group(2).strip(),
        "sound_type": match.group(3).strip(),
        "location": match.group(4).strip(),
        "age": match.group(5).strip(),
    }

parsed_bp = []
for filename in bp_df["filename"]:
    result = parse_bp_metadata(filename)
    if result is not None:
        parsed_bp.append(result)

bp_metadata = pd.DataFrame(parsed_bp)

# The Excel rows were previously found to match BP subject order.
# Compare each row against the subject number inferred from the filename.
bp_metadata = bp_metadata.sort_values("subject_number").reset_index(drop=True)

comparison = pd.DataFrame({
    "subject_number": bp_metadata["subject_number"],
    "filename_diagnosis": bp_metadata["filename_diagnosis"],
    "excel_diagnosis": annotation_df["Diagnosis"].astype(str).str.strip(),
    "filename_location": bp_metadata["location"].astype(str).str.strip(),
    "excel_location": annotation_df["Location"].astype(str).str.strip(),
})

comparison["diagnosis_match"] = (
    comparison["filename_diagnosis"].str.casefold()
    == comparison["excel_diagnosis"].str.casefold()
)

comparison["location_match"] = (
    comparison["filename_location"].str.replace(" ", "", regex=False).str.casefold()
    == comparison["excel_location"].astype(str).str.replace(" ", "", regex=False).str.casefold()
)

print("Parsed BP metadata rows:", len(bp_metadata))
print("Excel annotation rows:", len(annotation_df))
print("Diagnosis matches:",
      int(comparison["diagnosis_match"].sum()), "/", len(comparison))
print("Location matches:",
      int(comparison["location_match"].sum()), "/", len(comparison))

print("\nDiagnosis mismatches:")
display(comparison.loc[~comparison["diagnosis_match"]].head(20))

print("\nLocation mismatches:")
display(comparison.loc[~comparison["location_match"]].head(20))

KeyError: 'subject_number'

In [22]:
import re
import pandas as pd
from pathlib import Path

# Rebuild BP metadata using the already-validated inventory.
bp_df = fraiwan_inventory[
    fraiwan_inventory["prefix"] == "BP"
].copy()

def parse_bp_metadata(filename):
    stem = Path(filename).stem

    # Example:
    # BP101_Asthma,E W,P L M,12,F
    match = re.match(r"^BP(\d+)_([^,]+),(.*)$", stem)

    if not match:
        return None

    subject_number = int(match.group(1))
    diagnosis = match.group(2).strip()
    remaining_fields = [x.strip() for x in match.group(3).split(",")]

    return {
        "subject_number": subject_number,
        "filename_diagnosis": diagnosis,
        "filename_fields": remaining_fields,
    }

parsed_bp = [
    parse_bp_metadata(filename)
    for filename in bp_df["filename"]
]
parsed_bp = [row for row in parsed_bp if row is not None]

bp_metadata = pd.DataFrame(parsed_bp)

print("BP files:", len(bp_df))
print("Successfully parsed:", len(bp_metadata))

if not bp_metadata.empty:
    print("Columns:", bp_metadata.columns.tolist())
    display(bp_metadata.head(10))
else:
    print("No filenames matched. Sample filenames:")
    display(bp_df[["filename"]].head(10))
    

BP files: 112
Successfully parsed: 112
Columns: ['subject_number', 'filename_diagnosis', 'filename_fields']


,subject_number,filename_diagnosis,filename_fields
0,100,N,"[N, P R M, 70, F]"
1,101,Asthma,"[E W, P L M, 12, F]"
2,102,N,"[N, P L L, 41, M]"
3,103,N,"[N, P R U, 81, F]"
4,104,Asthma,"[E W, P L U, 45, F]"
5,105,Lung Fibrosis,"[Crep, A U R, 44, M]"
6,106,Asthma,"[E W, P L U, 45, F]"
7,107,Asthma,"[E W, P L U, 59, F]"
8,108,COPD,"[E W, P R L, 63, M]"
9,109,N,"[N, P L M, 26, M]"


In [23]:
# Compare BP filename metadata with the 112 populated Excel rows.

excel_df = pd.read_excel(FRAIWAN_ROOT / "Data annotation (1).xlsx")

annotation_df = (
    excel_df.dropna(subset=["Diagnosis", "Age", "Gender"])
    .reset_index(drop=True)
    .copy()
)

bp_metadata = bp_metadata.sort_values(
    "subject_number"
).reset_index(drop=True)

bp_metadata["sound_type"] = bp_metadata["filename_fields"].map(
    lambda fields: fields[0]
)
bp_metadata["filename_location"] = bp_metadata["filename_fields"].map(
    lambda fields: fields[1]
)
bp_metadata["filename_age"] = pd.to_numeric(
    bp_metadata["filename_fields"].map(lambda fields: fields[2]),
    errors="coerce"
)
bp_metadata["filename_gender"] = bp_metadata["filename_fields"].map(
    lambda fields: fields[3].strip().upper()
)

comparison = pd.DataFrame({
    "subject_number": bp_metadata["subject_number"],
    "filename_diagnosis": bp_metadata["filename_diagnosis"],
    "excel_diagnosis": annotation_df["Diagnosis"].astype(str).str.strip(),
    "filename_age": bp_metadata["filename_age"],
    "excel_age": pd.to_numeric(annotation_df["Age"], errors="coerce"),
    "filename_gender": bp_metadata["filename_gender"],
    "excel_gender": annotation_df["Gender"].astype(str).str.strip().str.upper(),
    "filename_location": bp_metadata["filename_location"],
    "excel_location": annotation_df["Location"].astype(str).str.strip(),
})

comparison["diagnosis_match"] = (
    comparison["filename_diagnosis"].str.casefold()
    == comparison["excel_diagnosis"].str.casefold()
)
comparison["age_match"] = (
    comparison["filename_age"] == comparison["excel_age"]
)
comparison["gender_match"] = (
    comparison["filename_gender"] == comparison["excel_gender"]
)
comparison["location_match"] = (
    comparison["filename_location"].str.replace(" ", "", regex=False).str.casefold()
    == comparison["excel_location"].str.replace(" ", "", regex=False).str.casefold()
)

for column in ["diagnosis_match", "age_match", "gender_match", "location_match"]:
    print(f"{column}: {int(comparison[column].sum())}/{len(comparison)}")

print("\nDiagnosis mismatches:")
display(comparison.loc[~comparison["diagnosis_match"]].head(10))

print("\nAge mismatches:")
display(comparison.loc[~comparison["age_match"]].head(10))

print("\nGender mismatches:")
display(comparison.loc[~comparison["gender_match"]].head(10))

print("\nLocation mismatches:")
display(comparison.loc[~comparison["location_match"]].head(10))


diagnosis_match: 112/112
age_match: 112/112
gender_match: 112/112
location_match: 104/112

Diagnosis mismatches:


,subject_number,filename_diagnosis,excel_diagnosis,filename_age,excel_age,filename_gender,excel_gender,filename_location,excel_location,diagnosis_match,age_match,gender_match,location_match



Age mismatches:


,subject_number,filename_diagnosis,excel_diagnosis,filename_age,excel_age,filename_gender,excel_gender,filename_location,excel_location,diagnosis_match,age_match,gender_match,location_match



Gender mismatches:


,subject_number,filename_diagnosis,excel_diagnosis,filename_age,excel_age,filename_gender,excel_gender,filename_location,excel_location,diagnosis_match,age_match,gender_match,location_match



Location mismatches:


,subject_number,filename_diagnosis,excel_diagnosis,filename_age,excel_age,filename_gender,excel_gender,filename_location,excel_location,diagnosis_match,age_match,gender_match,location_match
1,2,Asthma,Asthma,52,52.0,F,F,P L L R,P L L,True,True,True,False
2,3,Asthma,Asthma,50,50.0,F,F,P L L R,P L L,True,True,True,False
3,4,Heart Failure + Lung Fibrosis,Heart Failure + Lung Fibrosis,72,72.0,F,F,P L R,P R L,True,True,True,False
4,5,Heart Failure + COPD,Heart Failure + COPD,71,71.0,M,M,P L R,P R L,True,True,True,False
5,6,Plueral Effusion,Plueral Effusion,81,81.0,M,M,P L R,P R L,True,True,True,False
6,7,Heart Failure + COPD,Heart Failure + COPD,65,65.0,M,M,P L R,P R L,True,True,True,False
25,26,Lung Fibrosis,Lung Fibrosis,90,90.0,F,F,P,P L L,True,True,True,False
104,105,Lung Fibrosis,Lung Fibrosis,44,44.0,M,M,A U R,A R U,True,True,True,False


In [24]:
# Summarize Fraiwan subjects eligible for direct ICBHI diagnosis mapping.

fraiwan_subjects = comparison[
    ["subject_number", "filename_diagnosis"]
].copy()

fraiwan_subjects["diagnosis_normalized"] = (
    fraiwan_subjects["filename_diagnosis"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

direct_mapping = {
    "asthma": "Asthma",
    "copd": "COPD",
    "pneumonia": "Pneumonia",
}

fraiwan_subjects["icbhi_diagnosis"] = (
    fraiwan_subjects["diagnosis_normalized"]
    .map(direct_mapping)
)

fraiwan_subjects["mapping_status"] = "Exclude pending review"
fraiwan_subjects.loc[
    fraiwan_subjects["icbhi_diagnosis"].notna(),
    "mapping_status"
] = "Direct mapping candidate"

fraiwan_subjects.loc[
    fraiwan_subjects["diagnosis_normalized"] == "n",
    "mapping_status"
] = "Verify whether N means Healthy"

print("Total unique Fraiwan subjects:", len(fraiwan_subjects))

print("\nMapping status:")
print(fraiwan_subjects["mapping_status"].value_counts().to_string())

print("\nDirect mapping candidates:")
display(
    fraiwan_subjects[
        fraiwan_subjects["mapping_status"] == "Direct mapping candidate"
    ].groupby("icbhi_diagnosis").size().rename("subjects").to_frame()
)

print("\nLabels requiring review:")
display(
    fraiwan_subjects[
        fraiwan_subjects["mapping_status"] != "Direct mapping candidate"
    ].groupby(
        ["filename_diagnosis", "mapping_status"]
    ).size().rename("subjects").reset_index()
)

Total unique Fraiwan subjects: 112

Mapping status:
mapping_status
Direct mapping candidate          46
Verify whether N means Healthy    35
Exclude pending review            31

Direct mapping candidates:


,subjects
icbhi_diagnosis,
Asthma,32
COPD,9
Pneumonia,5



Labels requiring review:


,filename_diagnosis,mapping_status,subjects
0,Asthma and lung fibrosis,Exclude pending review,1
1,BRON,Exclude pending review,3
2,Heart Failure,Exclude pending review,3
3,Heart Failure + COPD,Exclude pending review,2
4,Heart Failure + Lung Fibrosis,Exclude pending review,1
5,Lung Fibrosis,Exclude pending review,4
6,N,Verify whether N means Healthy,35
7,Plueral Effusion,Exclude pending review,2
8,heart failure,Exclude pending review,15


In [25]:
from pathlib import Path
import zipfile

FRAIWAN_ROOT = ROOT / "data" / "raw" / "Fraiwan"

print("Documentation files:")
for path in sorted(FRAIWAN_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in {
        ".txt", ".md", ".pdf", ".doc", ".docx"
    }:
        print(path.relative_to(FRAIWAN_ROOT))

zip_path = FRAIWAN_ROOT / "Stethoscope Files.zip"

print("\nZIP exists:", zip_path.exists())

if zip_path.exists():
    with zipfile.ZipFile(zip_path) as zf:
        print("\nZIP contents:")
        for item in zf.infolist():
            print(
                f"{item.filename} | "
                f"directory={item.is_dir()} | "
                f"size={item.file_size}"
            )

Documentation files:

ZIP exists: True

ZIP contents:
P101-P112.zsa | directory=False | size=3678864
P1-P8.zsa | directory=False | size=5173312
P21-P32.zsa | directory=False | size=3190240
P33-P40.zsa | directory=False | size=2133886
P41-P52.zsa | directory=False | size=3350028
P53-P64.zsa | directory=False | size=3761954
P65-P76.zsa | directory=False | size=4207774
P77-P88.zsa | directory=False | size=3440890
P89-P100.zsa | directory=False | size=3716218
P9-P20.zsa | directory=False | size=3874998
Readme.txt | directory=False | size=290


In [26]:
import zipfile

zip_path = FRAIWAN_ROOT / "Stethoscope Files.zip"

with zipfile.ZipFile(zip_path, "r") as zf:
    readme_name = next(
        (
            name for name in zf.namelist()
            if Path(name).name.casefold() == "readme.txt"
        ),
        None,
    )

    if readme_name is None:
        print("No Readme.txt found in the ZIP.")
    else:
        readme_text = zf.read(readme_name).decode(
            "utf-8", errors="replace"
        )
        print("README contents:\n")
        print(readme_text)

README contents:

1. First four recordings from the file P1-P8.zsa were used for testing the stethoscope and should be ignored.
2. The files are named by the included patient recordings and in order. For example P33-P40.zsa contains the recordings from patients P33 to P40 in that order from top to bottom. 


In [27]:
import zipfile
from pathlib import Path

zip_path = FRAIWAN_ROOT / "Stethoscope Files.zip"

with zipfile.ZipFile(zip_path, "r") as zf:
    archives = [
        item for item in zf.infolist()
        if item.filename.lower().endswith(".zsa")
    ]

    for item in archives:
        with zf.open(item) as f:
            signature = f.read(32)

        print(f"\nFile: {item.filename}")
        print("Size:", item.file_size, "bytes")
        print("First 32 bytes:", signature.hex(" "))
        print("ASCII preview:", repr(
            "".join(chr(b) if 32 <= b < 127 else "." for b in signature)
        ))


File: P101-P112.zsa
Size: 3678864 bytes
First 32 bytes: 3c 3f 78 6d 6c 20 76 65 72 73 69 6f 6e 3d 22 31 2e 30 22 20 65 6e 63 6f 64 69 6e 67 3d 22 55 54
ASCII preview: '<?xml version="1.0" encoding="UT'

File: P1-P8.zsa
Size: 5173312 bytes
First 32 bytes: 3c 3f 78 6d 6c 20 76 65 72 73 69 6f 6e 3d 22 31 2e 30 22 20 65 6e 63 6f 64 69 6e 67 3d 22 55 54
ASCII preview: '<?xml version="1.0" encoding="UT'

File: P21-P32.zsa
Size: 3190240 bytes
First 32 bytes: 3c 3f 78 6d 6c 20 76 65 72 73 69 6f 6e 3d 22 31 2e 30 22 20 65 6e 63 6f 64 69 6e 67 3d 22 55 54
ASCII preview: '<?xml version="1.0" encoding="UT'

File: P33-P40.zsa
Size: 2133886 bytes
First 32 bytes: 3c 3f 78 6d 6c 20 76 65 72 73 69 6f 6e 3d 22 31 2e 30 22 20 65 6e 63 6f 64 69 6e 67 3d 22 55 54
ASCII preview: '<?xml version="1.0" encoding="UT'

File: P41-P52.zsa
Size: 3350028 bytes
First 32 bytes: 3c 3f 78 6d 6c 20 76 65 72 73 69 6f 6e 3d 22 31 2e 30 22 20 65 6e 63 6f 64 69 6e 67 3d 22 55 54
ASCII preview: '<?xml version="1.0" encoding=

In [28]:
import zipfile
from pathlib import Path
import xml.etree.ElementTree as ET

zip_path = FRAIWAN_ROOT / "Stethoscope Files.zip"

with zipfile.ZipFile(zip_path, "r") as zf:
    archive_name = "P1-P8.zsa"

    with zf.open(archive_name) as f:
        xml_start = f.read(12000)

print("First 2,000 characters of XML:")
print(xml_start[:2000].decode("utf-8", errors="replace"))

try:
    root = ET.fromstring(xml_start)
    print("\nParsed root tag:", root.tag)
    print("Root attributes:", root.attrib)
    print("First child tags:", [child.tag for child in list(root)[:10]])
except ET.ParseError as exc:
    print("\nThe excerpt is incomplete XML, so it could not be parsed fully:")
    print(exc)

First 2,000 characters of XML:
<?xml version="1.0" encoding="UTF-8"?>
<Encounter EncounterID="7d1ef2ea435e4a57a94cc1893d25ac7e" InstallationID="22a7bdce8455f6179c3296eb9ed0f6e2" TimeStamp="2020-01-23T10:30:36.447+02:00" VersionNumber="1.0">
    <xenc:EncryptedData xmlns:xenc="http://www.w3.org/2001/04/xmlenc#" Type="http://www.w3.org/2001/04/xmlenc#Content">
        <xenc:EncryptionMethod Algorithm="http://www.w3.org/2001/04/xmlenc#aes128-cbc"/>
        <xenc:CipherData>
            <xenc:CipherValue>xhc60GvsQxNgVykxBji1wvfAUiskfEpXqvsZUwZ178bAELTtJ0UHGk07EBr+hupODBiDg8gm+jDH
ECNAvwg7tv3y0jOx9G8g5o24J5dxRXB68BCaeoOQGyDbSIzy5K+9kNHqctb2o/I7DuhXtReLKf27
ERJAlzP1mCH3oQYcGR4MIcGTIQr5XNio06KFaHPOXGU8W95S+dc6751eIs/OHPgEUcZhtlkIQmwI
n7VvK2dKpp4sPOM+qKa9zNzG7B2zfe5Xev3fFa35CLG6+tV5RX25uEfemc9N5Twf0kY/kPiTo8KF
ZIPAb9lGNjpV3PGsT35HPjN6LcR+4Pbbxll2OJ3e7XRPPOLSS83rdw+418RRFVUQhwBTivtq+/kE
Xoy277hul60+EMZrJJTYcwJwMSnErj//YBcDjEEhRqYVXterCSXTXM/5+xPGh+gkR/+k4le1nEnd
Pvl67GJd0dTWrdkrPOrbj5XfAbJEBC20

In [29]:
from pathlib import Path
from collections import defaultdict
import soundfile as sf
import numpy as np

audio_root = FRAIWAN_ROOT / "Audio Files"

audio_stats = []

for _, row in fraiwan_inventory.iterrows():
    path = Path(row["recording_path"])

    try:
        info = sf.info(path)

        audio_stats.append({
            "filename": row["filename"],
            "patient_id": row["patient_id"],
            "prefix": row["prefix"],
            "diagnosis": row["raw_diagnosis"],
            "sample_rate": info.samplerate,
            "channels": info.channels,
            "duration_seconds": info.duration,
            "frames": info.frames,
        })
    except Exception as exc:
        audio_stats.append({
            "filename": row["filename"],
            "patient_id": row["patient_id"],
            "prefix": row["prefix"],
            "diagnosis": row["raw_diagnosis"],
            "error": str(exc),
        })

fraiwan_audio_stats = pd.DataFrame(audio_stats)

print("Files inspected:", len(fraiwan_audio_stats))
print("Files with read errors:",
      int(fraiwan_audio_stats["error"].notna().sum())
      if "error" in fraiwan_audio_stats.columns else 0)

for column in ["sample_rate", "channels", "duration_seconds"]:
    if column in fraiwan_audio_stats.columns:
        print(f"\n{column} distribution:")
        print(
            fraiwan_audio_stats.groupby("prefix")[column]
            .agg(["min", "median", "max", "nunique"])
            .to_string()
        )

print("\nExample recording properties:")
display(fraiwan_audio_stats.head(12))

Files inspected: 336
Files with read errors: 0

sample_rate distribution:
         min  median   max  nunique
prefix                             
BP      4000  4000.0  4000        1
DP      4000  4000.0  4000        1
EP      4000  4000.0  4000        1

channels distribution:
        min  median  max  nunique
prefix                           
BP        1     1.0    1        1
DP        1     1.0    1        1
EP        1     1.0    1        1

duration_seconds distribution:
        min  median   max  nunique
prefix                            
BP      5.0  15.984  30.0       87
DP      5.0  15.984  30.0       87
EP      5.0  15.920  30.0       86

Example recording properties:


,filename,patient_id,prefix,diagnosis,sample_rate,channels,duration_seconds,frames
0,"BP100_N,N,P R M,70,F.wav",BP100,BP,N,4000,1,13.552,54208
1,"BP101_Asthma,E W,P L M,12,F.wav",BP101,BP,Asthma,4000,1,13.552,54208
2,"BP102_N,N,P L L,41,M.wav",BP102,BP,N,4000,1,19.056,76224
3,"BP103_N,N,P R U,81,F.wav",BP103,BP,N,4000,1,12.848,51392
4,"BP104_Asthma,E W,P L U,45,F.wav",BP104,BP,Asthma,4000,1,11.760,47040
5,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",BP105,BP,Lung Fibrosis,4000,1,25.264,101056
6,"BP106_Asthma,E W,P L U,45,F.wav",BP106,BP,Asthma,4000,1,19.824,79296
7,"BP107_Asthma,E W,P L U,59,F.wav",BP107,BP,Asthma,4000,1,12.656,50624
8,"BP108_COPD,E W,P R L ,63,M.wav",BP108,BP,COPD,4000,1,22.832,91328
9,"BP109_N,N,P L M,26,M.wav",BP109,BP,N,4000,1,13.232,52928


In [30]:
import hashlib
from pathlib import Path
import pandas as pd

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

hash_records = []

for _, row in fraiwan_inventory.iterrows():
    path = Path(row["recording_path"])

    hash_records.append({
        "patient_id": row["patient_id"],
        "prefix": row["prefix"],
        "filename": row["filename"],
        "sha256": sha256_file(path),
    })

fraiwan_hashes = pd.DataFrame(hash_records)

fraiwan_hashes["subject_number"] = (
    fraiwan_hashes["patient_id"]
    .str.extract(r"^[A-Za-z]+(\d+)$", expand=False)
)

hash_table = fraiwan_hashes.pivot(
    index="subject_number",
    columns="prefix",
    values="sha256",
)

print("Subjects checked:", len(hash_table))

for prefix_a, prefix_b in [("BP", "DP"), ("BP", "EP"), ("DP", "EP")]:
    matches = hash_table[prefix_a] == hash_table[prefix_b]
    print(
        f"{prefix_a} vs {prefix_b}: "
        f"{int(matches.sum())}/{len(matches)} exact matches"
    )

print("\nSubjects with at least two identical versions:")
identical_mask = hash_table.nunique(axis=1) < 3
display(hash_table.loc[identical_mask].head(20))

Subjects checked: 112
BP vs DP: 1/112 exact matches
BP vs EP: 0/112 exact matches
DP vs EP: 0/112 exact matches

Subjects with at least two identical versions:


prefix,BP,DP,EP
subject_number,,,
9,2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...,2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...,2e0dccb294d97ce24a196e393ffe444fec32e127da2cac...


In [31]:
# Summarize Fraiwan recordings at the subject level.

subject_summary = (
    fraiwan_hashes
    .groupby("subject_number")
    .agg(
        recording_count=("filename", "count"),
        version_count=("prefix", "nunique"),
        unique_audio_count=("sha256", "nunique"),
        prefixes=("prefix", lambda s: ", ".join(sorted(s))),
    )
    .reset_index()
)

subject_summary["has_duplicate_audio"] = (
    subject_summary["unique_audio_count"]
    < subject_summary["recording_count"]
)

print("Unique subjects:", len(subject_summary))
print(
    "Subjects with all three prefixes:",
    int((subject_summary["version_count"] == 3).sum())
)
print(
    "Subjects with duplicate audio:",
    int(subject_summary["has_duplicate_audio"].sum())
)

print("\nRecording count per subject:")
print(subject_summary["recording_count"].value_counts().sort_index().to_string())

print("\nSubjects with duplicate audio:")
display(
    subject_summary[subject_summary["has_duplicate_audio"]]
)

print("\nOverall audio duplication summary:")
print("Total recordings:", len(fraiwan_hashes))
print("Unique audio hashes:", fraiwan_hashes["sha256"].nunique())
print(
    "Duplicate recording entries beyond unique hashes:",
    len(fraiwan_hashes) - fraiwan_hashes["sha256"].nunique()
)

Unique subjects: 112
Subjects with all three prefixes: 112
Subjects with duplicate audio: 1

Recording count per subject:
recording_count
3    112

Subjects with duplicate audio:


,subject_number,recording_count,version_count,unique_audio_count,prefixes,has_duplicate_audio
101,9,3,3,2,"BP, DP, EP",True



Overall audio duplication summary:
Total recordings: 336
Unique audio hashes: 332
Duplicate recording entries beyond unique hashes: 4


In [32]:
# Identify every exact duplicate audio file across the Fraiwan inventory.

duplicate_hashes = fraiwan_hashes[
    fraiwan_hashes.duplicated("sha256", keep=False)
].sort_values(["sha256", "patient_id", "prefix"])

print("Files involved in exact duplicate groups:",
      len(duplicate_hashes))

print("\nDuplicate audio groups:")
display(
    duplicate_hashes[
        ["patient_id", "prefix", "filename", "sha256"]
    ]
)

print("\nNumber of files per duplicate hash:")
display(
    duplicate_hashes.groupby("sha256").agg(
        files=("filename", "count"),
        subjects=("patient_id", "nunique"),
        prefixes=("prefix", "nunique"),
        filenames=("filename", lambda s: list(s)),
    ).reset_index()
)

Files involved in exact duplicate groups: 8

Duplicate audio groups:


,patient_id,prefix,filename,sha256
111,BP9,BP,"BP9_Asthma,E W,P R L ,59,M.wav",2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...
223,DP9,DP,"DP9_Asthma,E W,P R L ,59,M.wav",2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...
0,BP100,BP,"BP100_N,N,P R M,70,F.wav",8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
1,BP101,BP,"BP101_Asthma,E W,P L M,12,F.wav",8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
309,EP76,EP,"EP76_N,N,A L U,31,M.wav",b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
310,EP77,EP,"EP77_Asthma,E W,A R L,40,M.wav",b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
112,DP100,DP,"DP100_N,N,P R M,70,F.wav",fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...
113,DP101,DP,"DP101_Asthma,E W,P L M,12,F.wav",fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...



Number of files per duplicate hash:


,sha256,files,subjects,prefixes,filenames
0,2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...,2,2,2,"[BP9_Asthma,E W,P R L ,59,M.wav, DP9_Asthma,E ..."
1,8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...,2,2,1,"[BP100_N,N,P R M,70,F.wav, BP101_Asthma,E W,P ..."
2,b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...,2,2,1,"[EP76_N,N,A L U,31,M.wav, EP77_Asthma,E W,A R ..."
3,fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...,2,2,1,"[DP100_N,N,P R M,70,F.wav, DP101_Asthma,E W,P ..."


In [33]:
# Inspect duplicate groups with diagnosis, subject, duration, and audio hash.

duplicate_details = (
    fraiwan_hashes[
        fraiwan_hashes["sha256"].isin(
            duplicate_hashes["sha256"].unique()
        )
    ]
    .merge(
        fraiwan_audio_stats[
            ["filename", "duration_seconds", "sample_rate", "channels"]
        ],
        on="filename",
        how="left",
        validate="one_to_one",
    )
    .merge(
        fraiwan_inventory[
            ["filename", "raw_diagnosis", "recording_path"]
        ],
        on="filename",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["sha256", "filename"])
)

display(
    duplicate_details[
        [
            "filename",
            "patient_id",
            "prefix",
            "raw_diagnosis",
            "duration_seconds",
            "sample_rate",
            "channels",
            "sha256",
        ]
    ]
)

,filename,patient_id,prefix,raw_diagnosis,duration_seconds,sample_rate,channels,sha256
2,"BP9_Asthma,E W,P R L ,59,M.wav",BP9,BP,Asthma,15.344,4000,1,2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...
5,"DP9_Asthma,E W,P R L ,59,M.wav",DP9,DP,Asthma,15.344,4000,1,2dbdf57aca466088fc45b4ce53f7f6b5a20042d0349b6b...
0,"BP100_N,N,P R M,70,F.wav",BP100,BP,N,13.552,4000,1,8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
1,"BP101_Asthma,E W,P L M,12,F.wav",BP101,BP,Asthma,13.552,4000,1,8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
6,"EP76_N,N,A L U,31,M.wav",EP76,EP,N,14.576,4000,1,b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
7,"EP77_Asthma,E W,A R L,40,M.wav",EP77,EP,Asthma,14.576,4000,1,b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
3,"DP100_N,N,P R M,70,F.wav",DP100,DP,N,13.552,4000,1,fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...
4,"DP101_Asthma,E W,P L M,12,F.wav",DP101,DP,Asthma,13.552,4000,1,fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...


In [34]:
# Build an explicit audit table for duplicate audio groups.

duplicate_audit = (
    duplicate_details[
        [
            "filename",
            "patient_id",
            "prefix",
            "raw_diagnosis",
            "sha256",
        ]
    ]
    .copy()
)

duplicate_audit["numeric_subject"] = (
    duplicate_audit["patient_id"]
    .str.extract(r"^[A-Za-z]+(\d+)$", expand=False)
)

group_info = (
    duplicate_audit.groupby("sha256")
    .agg(
        file_count=("filename", "count"),
        subject_count=("numeric_subject", "nunique"),
        diagnoses=("raw_diagnosis", lambda s: sorted(set(s))),
        filenames=("filename", lambda s: list(s)),
    )
    .reset_index()
)

group_info["decision"] = group_info.apply(
    lambda row: (
        "Same-subject duplicate: retain one copy"
        if row["subject_count"] == 1
        else "Cross-subject duplicate: quarantine pending review"
    ),
    axis=1,
)

print("Duplicate groups:", len(group_info))
display(group_info[["file_count", "subject_count", "diagnoses", "decision", "filenames"]])

print("\nFiles flagged for quarantine:")
quarantine_hashes = set(
    group_info.loc[
        group_info["subject_count"] > 1, "sha256"
    ]
)

quarantine_files = duplicate_audit[
    duplicate_audit["sha256"].isin(quarantine_hashes)
].copy()

display(
    quarantine_files[
        ["filename", "patient_id", "raw_diagnosis", "sha256"]
    ]
)

Duplicate groups: 4


,file_count,subject_count,diagnoses,decision,filenames
0,2,1,[Asthma],Same-subject duplicate: retain one copy,"[BP9_Asthma,E W,P R L ,59,M.wav, DP9_Asthma,E ..."
1,2,2,"[Asthma, N]",Cross-subject duplicate: quarantine pending re...,"[BP100_N,N,P R M,70,F.wav, BP101_Asthma,E W,P ..."
2,2,2,"[Asthma, N]",Cross-subject duplicate: quarantine pending re...,"[EP76_N,N,A L U,31,M.wav, EP77_Asthma,E W,A R ..."
3,2,2,"[Asthma, N]",Cross-subject duplicate: quarantine pending re...,"[DP100_N,N,P R M,70,F.wav, DP101_Asthma,E W,P ..."



Files flagged for quarantine:


,filename,patient_id,raw_diagnosis,sha256
0,"BP100_N,N,P R M,70,F.wav",BP100,N,8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
1,"BP101_Asthma,E W,P L M,12,F.wav",BP101,Asthma,8bd3ed489f89bd6d002bb5e2973bfb61a948df01d40954...
6,"EP76_N,N,A L U,31,M.wav",EP76,N,b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
7,"EP77_Asthma,E W,A R L,40,M.wav",EP77,Asthma,b3a241944b470c3cec59ec56040c6c2b77a54802e0810e...
3,"DP100_N,N,P R M,70,F.wav",DP100,N,fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...
4,"DP101_Asthma,E W,P L M,12,F.wav",DP101,Asthma,fad2a61d1f58276d563518ab74506a111d5128dcb3ba01...


In [35]:
# Build a provisional Fraiwan candidate manifest.

fraiwan_candidates = fraiwan_inventory.copy()

# Extract the numeric subject ID so BP/DP/EP versions stay grouped.
fraiwan_candidates["subject_number"] = (
    fraiwan_candidates["patient_id"]
    .str.extract(r"^[A-Za-z]+(\d+)$", expand=False)
    .astype(int)
)

# Map only direct diagnosis matches; do not map N to Healthy yet.
direct_mapping = {
    "asthma": "Asthma",
    "copd": "COPD",
    "pneumonia": "Pneumonia",
}

fraiwan_candidates["icbhi_diagnosis"] = (
    fraiwan_candidates["raw_diagnosis"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .map(direct_mapping)
)

# Exclude every file belonging to the three cross-subject duplicate groups.
quarantine_hashes = set(
    group_info.loc[
        group_info["subject_count"] > 1, "sha256"
    ]
)

fraiwan_candidates = fraiwan_candidates.merge(
    fraiwan_hashes[["filename", "sha256"]],
    on="filename",
    how="left",
    validate="one_to_one",
)

fraiwan_candidates["duplicate_quarantine"] = (
    fraiwan_candidates["sha256"].isin(quarantine_hashes)
)

fraiwan_candidates["eligible_for_initial_merge"] = (
    fraiwan_candidates["icbhi_diagnosis"].notna()
    & ~fraiwan_candidates["duplicate_quarantine"]
)

print("Total WAV files:", len(fraiwan_candidates))
print(
    "Direct-mapping candidate files:",
    int(fraiwan_candidates["icbhi_diagnosis"].notna().sum())
)
print(
    "Cross-subject duplicate files quarantined:",
    int(fraiwan_candidates["duplicate_quarantine"].sum())
)
print(
    "Eligible files after quarantine:",
    int(fraiwan_candidates["eligible_for_initial_merge"].sum())
)

print("\nEligible files by diagnosis:")
print(
    fraiwan_candidates.loc[
        fraiwan_candidates["eligible_for_initial_merge"]
    ]["icbhi_diagnosis"].value_counts().to_string()
)

print("\nEligible unique numeric subjects by diagnosis:")
print(
    fraiwan_candidates.loc[
        fraiwan_candidates["eligible_for_initial_merge"]
    ].groupby("icbhi_diagnosis")["subject_number"]
    .nunique()
    .to_string()
)

Total WAV files: 336
Direct-mapping candidate files: 138
Cross-subject duplicate files quarantined: 6
Eligible files after quarantine: 135

Eligible files by diagnosis:
icbhi_diagnosis
Asthma       93
COPD         27
Pneumonia    15

Eligible unique numeric subjects by diagnosis:
icbhi_diagnosis
Asthma       32
COPD          9
Pneumonia     5


In [36]:
# Inspect BP/DP/EP recording availability per eligible subject.

eligible_fraiwan = fraiwan_candidates.loc[
    fraiwan_candidates["eligible_for_initial_merge"]
].copy()

subject_versions = (
    eligible_fraiwan
    .groupby(["subject_number", "icbhi_diagnosis"])
    .agg(
        recording_count=("filename", "count"),
        prefixes=("prefix", lambda s: sorted(set(s))),
        filenames=("filename", lambda s: sorted(s)),
    )
    .reset_index()
)

print("Eligible unique subjects:", len(subject_versions))
print("\nRecording-count distribution per subject:")
print(subject_versions["recording_count"].value_counts().sort_index().to_string())

print("\nPrefix availability patterns:")
prefix_patterns = (
    subject_versions["prefixes"]
    .astype(str)
    .value_counts()
)
print(prefix_patterns.to_string())

print("\nSubjects with fewer than 3 eligible recordings:")
display(
    subject_versions.loc[
        subject_versions["recording_count"] < 3,
        ["subject_number", "icbhi_diagnosis", "recording_count", "prefixes", "filenames"],
    ].sort_values(["icbhi_diagnosis", "subject_number"])
)

Eligible unique subjects: 46

Recording-count distribution per subject:
recording_count
1     1
2     1
3    44

Prefix availability patterns:
prefixes
['BP', 'DP', 'EP']    44
['BP', 'DP']           1
['EP']                 1

Subjects with fewer than 3 eligible recordings:


,subject_number,icbhi_diagnosis,recording_count,prefixes,filenames
33,77,Asthma,2,"[BP, DP]","[BP77_Asthma,E W,A R L,40,M.wav, DP77_Asthma,E..."
39,101,Asthma,1,[EP],"[EP101_Asthma,E W,P L M,12,F.wav]"


In [37]:
from sklearn.model_selection import train_test_split

# One row per eligible numeric subject.
fraiwan_subject_table = (
    eligible_fraiwan[
        ["subject_number", "icbhi_diagnosis"]
    ]
    .drop_duplicates()
    .sort_values(["icbhi_diagnosis", "subject_number"])
    .reset_index(drop=True)
)

# Check that each numeric subject has exactly one diagnosis.
assert (
    fraiwan_subject_table.groupby("subject_number")["icbhi_diagnosis"]
    .nunique()
    .max() == 1
)

# Stratify only if every class has enough subjects for both partitions.
class_counts = fraiwan_subject_table["icbhi_diagnosis"].value_counts()
print("Subjects per class:")
print(class_counts.to_string())

if class_counts.min() < 2:
    raise ValueError(
        "At least one class has fewer than two subjects; "
        "a stratified train/test split is not possible."
    )

train_subjects, test_subjects = train_test_split(
    fraiwan_subject_table["subject_number"].to_numpy(),
    test_size=0.20,
    random_state=42,
    stratify=fraiwan_subject_table["icbhi_diagnosis"],
)

subject_split = dict.fromkeys(train_subjects, "train")
subject_split.update(dict.fromkeys(test_subjects, "test"))

eligible_fraiwan["split"] = (
    eligible_fraiwan["subject_number"].map(subject_split)
)

print("\nSubject split by diagnosis:")
display(
    fraiwan_subject_table.assign(
        split=fraiwan_subject_table["subject_number"].map(subject_split)
    )
    .groupby(["icbhi_diagnosis", "split"])
    .agg(subjects=("subject_number", "nunique"))
    .reset_index()
)

print("\nAudio-file split by diagnosis:")
display(
    eligible_fraiwan.groupby(["icbhi_diagnosis", "split"])
    .agg(
        audio_files=("filename", "count"),
        unique_subjects=("subject_number", "nunique"),
    )
    .reset_index()
)

# Verify no numeric subject appears in both partitions.
assert set(train_subjects).isdisjoint(set(test_subjects))
assert eligible_fraiwan["split"].notna().all()

print("\nVerified: no Fraiwan subject occurs in both splits.")

Subjects per class:
icbhi_diagnosis
Asthma       32
COPD          9
Pneumonia     5

Subject split by diagnosis:


,icbhi_diagnosis,split,subjects
0,Asthma,test,7
1,Asthma,train,25
2,COPD,test,2
3,COPD,train,7
4,Pneumonia,test,1
5,Pneumonia,train,4



Audio-file split by diagnosis:


,icbhi_diagnosis,split,audio_files,unique_subjects
0,Asthma,test,19,7
1,Asthma,train,74,25
2,COPD,test,6,2
3,COPD,train,21,7
4,Pneumonia,test,3,1
5,Pneumonia,train,12,4



Verified: no Fraiwan subject occurs in both splits.


In [38]:

import pandas as pd

# ---------- ICBHI manifest ----------
icbhi_combined = icbhi_usable.copy()

icbhi_combined["source_dataset"] = "ICBHI"
icbhi_combined["source_recording_id"] = (
    icbhi_combined["recording_id"].astype(str)
)
icbhi_combined["patient_id"] = (
    "ICBHI_" + icbhi_combined["patient_id"].astype(str)
)
icbhi_combined["diagnosis"] = (
    icbhi_combined["diagnosis"].astype(str).str.strip()
)
icbhi_combined["split"] = icbhi_combined["corrected_split"]

# ---------- Fraiwan manifest ----------
fraiwan_combined = eligible_fraiwan.copy()

fraiwan_combined["source_dataset"] = "Fraiwan"
fraiwan_combined["source_recording_id"] = (
    fraiwan_combined["filename"].astype(str)
)
fraiwan_combined["patient_id"] = (
    "FRAIWAN_" + fraiwan_combined["subject_number"].astype(str)
)
fraiwan_combined["diagnosis"] = fraiwan_combined["icbhi_diagnosis"]
fraiwan_combined["recording_path"] = (
    fraiwan_combined["recording_path"].astype(str)
)

# Keep only the columns needed for the shared manifest.
manifest_columns = [
    "source_dataset",
    "source_recording_id",
    "patient_id",
    "diagnosis",
    "split",
    "recording_path",
]

icbhi_part = icbhi_combined[manifest_columns].copy()
fraiwan_part = fraiwan_combined[manifest_columns].copy()

combined_manifest = pd.concat(
    [icbhi_part, fraiwan_part],
    ignore_index=True,
)

# Basic integrity checks.
assert combined_manifest["recording_path"].notna().all()
assert combined_manifest["diagnosis"].notna().all()
assert combined_manifest["split"].isin(["train", "test"]).all()

# A source recording should appear only once.
assert not combined_manifest.duplicated(
    ["source_dataset", "source_recording_id"]
).any()

# A patient must never cross train/test within its own source.
patient_split_counts = (
    combined_manifest.groupby(["source_dataset", "patient_id"])["split"]
    .nunique()
)
assert patient_split_counts.max() == 1

print("Combined manifest rows:", len(combined_manifest))
print("\nRows by source and split:")
display(
    combined_manifest.groupby(["source_dataset", "split"])
    .size()
    .rename("recordings")
    .reset_index()
)

print("\nRows by diagnosis, source, and split:")
display(
    combined_manifest.groupby(
        ["diagnosis", "source_dataset", "split"]
    )
    .size()
    .rename("recordings")
    .reset_index()
)

print("\nUnique patients by source and split:")
display(
    combined_manifest.groupby(["source_dataset", "split"])["patient_id"]
    .nunique()
    .rename("patients")
    .reset_index()
)

print("\nAll initial integrity checks passed.")


Combined manifest rows: 1054

Rows by source and split:


,source_dataset,split,recordings
0,Fraiwan,test,28
1,Fraiwan,train,107
2,ICBHI,test,394
3,ICBHI,train,525



Rows by diagnosis, source, and split:


,diagnosis,source_dataset,split,recordings
0,Asthma,Fraiwan,test,19
1,Asthma,Fraiwan,train,74
2,Asthma,ICBHI,train,1
3,Bronchiectasis,ICBHI,test,2
4,Bronchiectasis,ICBHI,train,14
5,Bronchiolitis,ICBHI,test,6
6,Bronchiolitis,ICBHI,train,7
7,COPD,Fraiwan,test,6
8,COPD,Fraiwan,train,21
9,COPD,ICBHI,test,362



Unique patients by source and split:


,source_dataset,split,patients
0,Fraiwan,test,10
1,Fraiwan,train,36
2,ICBHI,test,49
3,ICBHI,train,77



All initial integrity checks passed.


In [39]:

# Class-support audit for the proposed combined dataset.

all_classes = sorted(combined_manifest["diagnosis"].unique())
all_sources = ["ICBHI", "Fraiwan"]
all_splits = ["train", "test"]

support_rows = []

for diagnosis in all_classes:
    for source in all_sources:
        for split in all_splits:
            subset = combined_manifest[
                (combined_manifest["diagnosis"] == diagnosis)
                & (combined_manifest["source_dataset"] == source)
                & (combined_manifest["split"] == split)
            ]

            support_rows.append({
                "diagnosis": diagnosis,
                "source_dataset": source,
                "split": split,
                "recordings": len(subset),
                "patients": subset["patient_id"].nunique(),
            })

class_support_audit = pd.DataFrame(support_rows)

print("Full class support by source and split:")
display(
    class_support_audit.sort_values(
        ["diagnosis", "source_dataset", "split"]
    ).reset_index(drop=True)
)

print("\nClasses with zero test patients in ICBHI:")
display(
    class_support_audit[
        (class_support_audit["source_dataset"] == "ICBHI")
        & (class_support_audit["split"] == "test")
        & (class_support_audit["patients"] == 0)
    ][["diagnosis", "recordings", "patients"]]
)

print("\nClasses with fewer than 5 training patients across the combined data:")
combined_train_patients = (
    combined_manifest[combined_manifest["split"] == "train"]
    .groupby("diagnosis")["patient_id"]
    .nunique()
    .reindex(all_classes, fill_value=0)
    .rename("training_patients")
    .reset_index()
    .rename(columns={"index": "diagnosis"})
)

display(
    combined_train_patients[
        combined_train_patients["training_patients"] < 5
    ]
)

print("\nAudit complete. No files have been saved or changed.")


Full class support by source and split:


,diagnosis,source_dataset,split,recordings,patients
0,Asthma,Fraiwan,test,19,7
1,Asthma,Fraiwan,train,74,25
2,Asthma,ICBHI,test,0,0
3,Asthma,ICBHI,train,1,1
4,Bronchiectasis,Fraiwan,test,0,0
5,Bronchiectasis,Fraiwan,train,0,0
6,Bronchiectasis,ICBHI,test,2,1
7,Bronchiectasis,ICBHI,train,14,6
8,Bronchiolitis,Fraiwan,test,0,0
9,Bronchiolitis,Fraiwan,train,0,0



Classes with zero test patients in ICBHI:


,diagnosis,recordings,patients
1,Asthma,0,0
21,LRTI,0,0
25,Pneumonia,0,0



Classes with fewer than 5 training patients across the combined data:


,diagnosis,training_patients
2,Bronchiolitis,3
5,LRTI,2



Audit complete. No files have been saved or changed.


In [40]:

from pathlib import Path

# Experiment B output directory
output_dir = (
    project_root
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

proposed_files = [
    "experiment_b_train_manifest.csv",
    "experiment_b_icbhi_test_manifest.csv",
    "experiment_b_fraiwan_test_manifest.csv",
    "experiment_b_class_support_audit.csv",
    "experiment_b_preparation_report.txt",
]

print("Output directory:")
print(output_dir)
print("Directory exists:", output_dir.exists())

print("\nProposed output files:")
for filename in proposed_files:
    path = output_dir / filename
    print(f"{filename}: {'EXISTS — protect from overwrite' if path.exists() else 'not present'}")

print("\nExisting contents of the output directory:")
if output_dir.exists():
    existing = sorted(p.name for p in output_dir.iterdir())
    if existing:
        for name in existing:
            print(" -", name)
    else:
        print("(directory is empty)")
else:
    print("(directory does not exist yet)")

print("\nNo files were created or modified.")


NameError: name 'project_root' is not defined

In [41]:

from pathlib import Path

cwd = Path.cwd().resolve()

# Find the project root by locating its expected raw-data directory.
candidates = [cwd, *cwd.parents]

project_root_candidates = [
    p for p in candidates
    if (p / "data" / "raw" / "ICBHI").is_dir()
]

if len(project_root_candidates) == 1:
    project_root = project_root_candidates[0]
elif len(project_root_candidates) > 1:
    # Choose the nearest matching ancestor.
    project_root = project_root_candidates[0]
else:
    raise FileNotFoundError(
        f"Could not find the project root from {cwd}.\n"
        "Please check the notebook working directory."
    )

output_dir = (
    project_root
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

proposed_files = [
    "experiment_b_train_manifest.csv",
    "experiment_b_icbhi_test_manifest.csv",
    "experiment_b_fraiwan_test_manifest.csv",
    "experiment_b_class_support_audit.csv",
    "experiment_b_preparation_report.txt",
]

print("Notebook working directory:", cwd)
print("Detected project root:", project_root)
print("Output directory:", output_dir)
print("Output directory exists:", output_dir.exists())

print("\nProposed output files:")
for filename in proposed_files:
    path = output_dir / filename
    status = "EXISTS — will protect from overwrite" if path.exists() else "not present"
    print(f"{filename}: {status}")

print("\nNo files were created or modified.")


Notebook working directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2
Detected project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1
Output directory exists: True

Proposed output files:
experiment_b_train_manifest.csv: not present
experiment_b_icbhi_test_manifest.csv: not present
experiment_b_fraiwan_test_manifest.csv: not present
experiment_b_class_support_audit.csv: not present
experiment_b_preparation_report.txt: not present

No files were created or modified.


In [42]:

from pathlib import Path

notebooks_root = project_root / "Notebooks 2"

# Find notebooks that may contain the existing CNN training pipeline.
print("=== CNN / DEEP LEARNING NOTEBOOKS ===")
for path in sorted(notebooks_root.rglob("*.ipynb")):
    name = path.name.lower()
    if any(term in name for term in ["cnn", "deep", "train", "model", "experiment"]):
        print(path.relative_to(project_root))

# Find likely saved model artifacts, without opening or modifying them.
search_roots = [
    project_root / "models",
    project_root / "artifacts",
    project_root / "data" / "processed",
]

model_extensions = {".keras", ".h5", ".pt", ".pth", ".pkl", ".joblib", ".onnx"}

print("\n=== LIKELY SAVED MODEL ARTIFACTS ===")
found_any = False

for root in search_roots:
    if root.exists():
        for path in sorted(root.rglob("*")):
            if path.is_file() and path.suffix.lower() in model_extensions:
                print(f"{path.relative_to(project_root)} ({path.stat().st_size:,} bytes)")
                found_any = True

if not found_any:
    print("No model artifacts found in the searched directories.")

print("\nInspection only: no files were changed.")


=== CNN / DEEP LEARNING NOTEBOOKS ===
Notebooks 2/.ipynb_checkpoints/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease-checkpoint.ipynb
Notebooks 2/.ipynb_checkpoints/05_Lightweight_MultiFeature_CNN-checkpoint.ipynb
Notebooks 2/03_Multidataset_Phenotype_Pretraining_ICBHI_Disease.ipynb
Notebooks 2/05_Lightweight_MultiFeature_CNN.ipynb

=== LIKELY SAVED MODEL ARTIFACTS ===
models/deep_learning/cnn_lstm/best_cnn_lstm.pt (373,412 bytes)
models/deep_learning/experiment_b/experiment_b_cnn.pt (172,117 bytes)
models/deep_learning/imbalance_comparison/experiment_A_weighted_ce_best.pt (465,874 bytes)
models/deep_learning/imbalance_comparison/experiment_B_balanced_sampler_best.pt (466,269 bytes)
models/deep_learning/lightweight_mel_cnn/best_lightweight_mel_cnn.pt (310,935 bytes)
models/deep_learning/multifeature_cnn/best_multifeature_cnn.pt (464,538 bytes)
models/deep_learning/regularized_multifeature_cnn/best_regularized_multifeature_cnn.pt (465,550 bytes)
models/multidataset_pretraining/pheno

In [43]:

# Verify the current notebook state before finalizing the combined dataset.

required_variables = [
    "project_root",
    "output_dir",
    "icbhi_usable",
    "eligible_fraiwan",
    "combined_manifest",
    "class_support_audit",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Required variables are missing from this notebook kernel: "
        + ", ".join(missing_variables)
        + ". Rerun the cells that create them before continuing."
    )

print("Current notebook: 06_Combined_Dataset_Preparation.ipynb")
print("Project root:", project_root)
print("Output directory:", output_dir)

print("\nIn-memory dataset sizes:")
print("ICBHI usable recordings:", len(icbhi_usable))
print("Fraiwan eligible recordings:", len(eligible_fraiwan))
print("Combined manifest recordings:", len(combined_manifest))

print("\nCombined manifest columns:")
print(combined_manifest.columns.tolist())

print("\nRecordings by source and split:")
display(
    combined_manifest.groupby(["source_dataset", "split"])
    .size()
    .rename("recordings")
    .reset_index()
)

print("\nNo files were saved or modified.")


Current notebook: 06_Combined_Dataset_Preparation.ipynb
Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1

In-memory dataset sizes:
ICBHI usable recordings: 919
Fraiwan eligible recordings: 135
Combined manifest recordings: 1054

Combined manifest columns:
['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'split', 'recording_path']

Recordings by source and split:


,source_dataset,split,recordings
0,Fraiwan,test,28
1,Fraiwan,train,107
2,ICBHI,test,394
3,ICBHI,train,525



No files were saved or modified.


In [44]:

# Final source, path, label and split integrity checks.

from pathlib import Path

manifest_check = combined_manifest.copy()

# 1. Verify all referenced audio files exist.
manifest_check["path_exists"] = manifest_check["recording_path"].map(
    lambda p: Path(p).is_file()
)

missing_paths = manifest_check.loc[
    ~manifest_check["path_exists"],
    ["source_dataset", "source_recording_id", "recording_path"]
]

print("=== AUDIO PATH CHECK ===")
print("Total recordings:", len(manifest_check))
print("Existing audio paths:", int(manifest_check["path_exists"].sum()))
print("Missing audio paths:", len(missing_paths))

if not missing_paths.empty:
    display(missing_paths.head(20))

# 2. Verify diagnosis labels by source.
print("\n=== DIAGNOSIS LABELS BY SOURCE ===")
display(
    manifest_check.groupby(["source_dataset", "diagnosis"])
    .agg(
        recordings=("source_recording_id", "count"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
    .sort_values(["source_dataset", "diagnosis"])
)

# 3. Check whether any source-specific patient appears in both splits.
patient_split_counts = (
    manifest_check.groupby(["source_dataset", "patient_id"])["split"]
    .nunique()
)

overlapping_patients = patient_split_counts[patient_split_counts > 1]

print("\n=== PATIENT SPLIT CHECK ===")
print("Patients crossing train/test:", len(overlapping_patients))

# 4. Check recording uniqueness.
duplicate_recordings = manifest_check.duplicated(
    ["source_dataset", "source_recording_id"],
    keep=False,
)

print("\n=== RECORDING UNIQUENESS CHECK ===")
print("Duplicate recording rows:", int(duplicate_recordings.sum()))

# 5. Verify that Fraiwan contributes only the intended diagnoses.
fraiwan_labels = set(
    manifest_check.loc[
        manifest_check["source_dataset"] == "Fraiwan", "diagnosis"
    ].unique()
)
expected_fraiwan_labels = {"Asthma", "COPD", "Pneumonia"}

print("\nFraiwan labels:", sorted(fraiwan_labels))
print("Expected Fraiwan labels:", sorted(expected_fraiwan_labels))

assert missing_paths.empty, "Some referenced audio files are missing."
assert overlapping_patients.empty, "A patient crosses train/test splits."
assert not duplicate_recordings.any(), "Duplicate recording rows detected."
assert fraiwan_labels == expected_fraiwan_labels, "Unexpected Fraiwan labels."

print("\nAll integrity checks passed.")
print("No files were saved or modified.")


=== AUDIO PATH CHECK ===
Total recordings: 1054
Existing audio paths: 1054
Missing audio paths: 0

=== DIAGNOSIS LABELS BY SOURCE ===


,source_dataset,diagnosis,recordings,patients
0,Fraiwan,Asthma,93,32
1,Fraiwan,COPD,27,9
2,Fraiwan,Pneumonia,15,5
3,ICBHI,Asthma,1,1
4,ICBHI,Bronchiectasis,16,7
5,ICBHI,Bronchiolitis,13,6
6,ICBHI,COPD,793,64
7,ICBHI,Healthy,35,26
8,ICBHI,LRTI,2,2
9,ICBHI,Pneumonia,36,6



=== PATIENT SPLIT CHECK ===
Patients crossing train/test: 0

=== RECORDING UNIQUENESS CHECK ===
Duplicate recording rows: 0

Fraiwan labels: ['Asthma', 'COPD', 'Pneumonia']
Expected Fraiwan labels: ['Asthma', 'COPD', 'Pneumonia']

All integrity checks passed.
No files were saved or modified.


In [45]:

# Save validated dataset preparation outputs.
# This does NOT train a model or modify raw audio.

from pathlib import Path

output_dir.mkdir(parents=True, exist_ok=True)

experiment_b_train = combined_manifest.loc[
    combined_manifest["split"] == "train"
].copy()

experiment_b_icbhi_test = combined_manifest.loc[
    (combined_manifest["split"] == "test")
    & (combined_manifest["source_dataset"] == "ICBHI")
].copy()

experiment_b_fraiwan_test = combined_manifest.loc[
    (combined_manifest["split"] == "test")
    & (combined_manifest["source_dataset"] == "Fraiwan")
].copy()

# Include the source-specific class support for all splits.
experiment_b_class_support = (
    combined_manifest.groupby(
        ["source_dataset", "diagnosis", "split"]
    )
    .agg(
        recordings=("source_recording_id", "count"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

outputs = {
    "experiment_b_train_manifest.csv": experiment_b_train,
    "experiment_b_icbhi_test_manifest.csv": experiment_b_icbhi_test,
    "experiment_b_fraiwan_test_manifest.csv": experiment_b_fraiwan_test,
    "experiment_b_class_support_audit.csv": experiment_b_class_support,
}

report_path = output_dir / "experiment_b_preparation_report.txt"
target_paths = [output_dir / name for name in outputs] + [report_path]

existing = [path for path in target_paths if path.exists()]
if existing:
    raise FileExistsError(
        "Nothing was saved. These files already exist:\n"
        + "\n".join(str(path) for path in existing)
    )

# Validate partition sizes before writing.
assert len(experiment_b_train) == 632
assert len(experiment_b_icbhi_test) == 394
assert len(experiment_b_fraiwan_test) == 28

assert experiment_b_train["split"].eq("train").all()
assert experiment_b_icbhi_test["source_dataset"].eq("ICBHI").all()
assert experiment_b_fraiwan_test["source_dataset"].eq("Fraiwan").all()

report = f"""Combined Dataset Preparation Report
===================================

Status: Dataset preparation only; Experiment B model training is NOT complete.

Total recordings: {len(combined_manifest)}
Training recordings: {len(experiment_b_train)}
  ICBHI: {(experiment_b_train["source_dataset"] == "ICBHI").sum()}
  Fraiwan: {(experiment_b_train["source_dataset"] == "Fraiwan").sum()}

ICBHI test recordings: {len(experiment_b_icbhi_test)}
Fraiwan test recordings: {len(experiment_b_fraiwan_test)}

Included Fraiwan diagnoses: Asthma, COPD, Pneumonia.
Fraiwan diagnosis N was not mapped to Healthy.
Six cross-subject duplicate files were excluded from eligible Fraiwan candidates.
HF Lung V1 was excluded from disease classification because its available labels
describe sound events rather than patient diagnoses.

Split integrity:
- No source-specific patient crosses train/test.
- No duplicate source-recording IDs in the combined manifest.
- All 1,054 audio paths were verified to exist.

Important evaluation limitations:
- ICBHI has no Asthma, Pneumonia or LRTI test recordings in this split.
- Fraiwan's held-out test set contains only 10 subjects.
- The two source-specific test sets must be evaluated and reported separately.

No model was trained in this preparation step.
"""

# Write CSVs and report only after all pre-write checks pass.
for name, frame in outputs.items():
    frame.to_csv(output_dir / name, index=False)

report_path.write_text(report, encoding="utf-8")

print("Saved preparation outputs:")
for path in target_paths:
    print(f"- {path.name} ({path.stat().st_size:,} bytes)")

print("\nSaved row counts:")
for name, frame in outputs.items():
    print(f"- {name}: {len(frame)} rows")

print("\nDataset preparation outputs saved. Experiment B training remains pending.")


Saved preparation outputs:
- experiment_b_train_manifest.csv (94,721 bytes)
- experiment_b_icbhi_test_manifest.csv (56,150 bytes)
- experiment_b_fraiwan_test_manifest.csv (5,087 bytes)
- experiment_b_class_support_audit.csv (538 bytes)
- experiment_b_preparation_report.txt (1,070 bytes)

Saved row counts:
- experiment_b_train_manifest.csv: 632 rows
- experiment_b_icbhi_test_manifest.csv: 394 rows
- experiment_b_fraiwan_test_manifest.csv: 28 rows
- experiment_b_class_support_audit.csv: 19 rows

Dataset preparation outputs saved. Experiment B training remains pending.


In [46]:

# Read-back verification of saved Experiment B preparation outputs.

import pandas as pd

train_path = output_dir / "experiment_b_train_manifest.csv"
icbhi_test_path = output_dir / "experiment_b_icbhi_test_manifest.csv"
fraiwan_test_path = output_dir / "experiment_b_fraiwan_test_manifest.csv"
audit_path = output_dir / "experiment_b_class_support_audit.csv"

saved_train = pd.read_csv(train_path)
saved_icbhi_test = pd.read_csv(icbhi_test_path)
saved_fraiwan_test = pd.read_csv(fraiwan_test_path)
saved_support_audit = pd.read_csv(audit_path)

expected_rows = {
    "training": 632,
    "ICBHI test": 394,
    "Fraiwan test": 28,
}

actual_rows = {
    "training": len(saved_train),
    "ICBHI test": len(saved_icbhi_test),
    "Fraiwan test": len(saved_fraiwan_test),
}

print("=== SAVED FILE VERIFICATION ===")
for name, expected in expected_rows.items():
    actual = actual_rows[name]
    print(f"{name}: {actual} rows (expected {expected})")
    assert actual == expected, f"{name} row count mismatch"

# Check schema consistency.
expected_columns = [
    "source_dataset",
    "source_recording_id",
    "patient_id",
    "diagnosis",
    "split",
    "recording_path",
]

for name, frame in [
    ("training", saved_train),
    ("ICBHI test", saved_icbhi_test),
    ("Fraiwan test", saved_fraiwan_test),
]:
    assert frame.columns.tolist() == expected_columns, (
        f"Unexpected columns in {name}"
    )
    assert frame.notna().all().all(), f"Missing values in {name}"

assert saved_train["split"].eq("train").all()
assert saved_icbhi_test["split"].eq("test").all()
assert saved_icbhi_test["source_dataset"].eq("ICBHI").all()
assert saved_fraiwan_test["split"].eq("test").all()
assert saved_fraiwan_test["source_dataset"].eq("Fraiwan").all()

# Ensure no patient crosses training and test partitions within a source.
train_patients = set(
    zip(saved_train["source_dataset"], saved_train["patient_id"])
)
test_patients = set(
    zip(
        pd.concat([saved_icbhi_test, saved_fraiwan_test])["source_dataset"],
        pd.concat([saved_icbhi_test, saved_fraiwan_test])["patient_id"],
    )
)
assert train_patients.isdisjoint(test_patients)

print("\nTraining labels:")
print(saved_train["diagnosis"].value_counts().to_string())

print("\nFraiwan test labels:")
print(saved_fraiwan_test["diagnosis"].value_counts().to_string())

print("\nClass-support audit rows:", len(saved_support_audit))
print("All read-back checks passed.")
print("No files were modified by this verification.")


=== SAVED FILE VERIFICATION ===
training: 632 rows (expected 632)
ICBHI test: 394 rows (expected 394)
Fraiwan test: 28 rows (expected 28)

Training labels:
diagnosis
COPD              452
Asthma             75
Pneumonia          48
Healthy            18
URTI               16
Bronchiectasis     14
Bronchiolitis       7
LRTI                2

Fraiwan test labels:
diagnosis
Asthma       19
COPD          6
Pneumonia     3

Class-support audit rows: 19
All read-back checks passed.
No files were modified by this verification.


In [47]:

# Inspect preprocessing and architecture settings from the existing CNN notebook.
# Reference only: this does not execute or modify the notebook.

import json

reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with reference_notebook.open("r", encoding="utf-8") as f:
    reference_data = json.load(f)

keywords = [
    "sample_rate", "sampling_rate", "sr =", "n_mels", "n_fft",
    "hop_length", "n_mfcc", "duration", "resample", "MelSpectrogram",
    "MFCC", "log_mel", "feature", "class ", "def ", "Dataset",
    "DataLoader", "input_channels", "num_classes", "CrossEntropyLoss",
    "WeightedRandomSampler", "class_weights", "torch.save"
]

print("Reference notebook:", reference_notebook.name)

for i, cell in enumerate(reference_data.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()
    matched_lines = [
        (line_number + 1, line.strip())
        for line_number, line in enumerate(lines)
        if any(keyword.lower() in line.lower() for keyword in keywords)
    ]

    if matched_lines:
        print(f"\n--- Code cell {i} ---")
        for line_number, line in matched_lines[:18]:
            print(f"{line_number:>3}: {line[:180]}")

print("\nReference inspection complete; no notebook was executed or changed.")


Reference notebook: 05_Lightweight_MultiFeature_CNN.ipynb

--- Code cell 0 ---
  4: source_path = ROOT / "Notebooks" / "12_multifeature_cnn.ipynb"

--- Code cell 1 ---
 24: MULTIFEATURE_DIR = ROOT / "data/processed/deep_learning/multifeature_cnn"
 25: EXPERIMENT_12B_DIR = ROOT / "data/processed/deep_learning/multifeature_cnn_12b"
 30: for folder in [BASELINE_DIR, MULTIFEATURE_DIR, EXPERIMENT_12B_DIR]:
 45: MULTIFEATURE_DIR / "multifeature_cnn_metrics.csv",

--- Code cell 3 ---
  7: ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_training_config.json",
  8: ROOT / "data/processed/deep_learning/multifeature_cnn_12b/training_config.json",

--- Code cell 4 ---
  6: class ExistingFeatureBranch(nn.Module):
  7: def __init__(self, output_dim=64, dropout=0.30):
  9: self.features = nn.Sequential(
 30: def forward(self, x):
 31: return self.projection(self.features(x))
 34: class ExistingMultiFeatureCNN(nn.Module):
 35: def __init__(self, num_classes=8, dropout=0.30):
 37

In [48]:

import json
from pathlib import Path

config_paths = [
    project_root / "data/processed/deep_learning/lightweight_multifeature_cnn/feature_config.json",
    project_root / "data/processed/deep_learning/lightweight_multifeature_cnn/normalization_stats.json",
    project_root / "data/processed/deep_learning/multifeature_cnn_12b/training_config.json",
    project_root / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_training_config.json",
]

for config_path in config_paths:
    print(f"\n{'=' * 70}")
    print("File:", config_path.relative_to(project_root))

    if not config_path.exists():
        print("Not found")
        continue

    try:
        with config_path.open("r", encoding="utf-8") as f:
            config = json.load(f)

        print(json.dumps(config, indent=2, default=str)[:12000])
    except Exception as exc:
        print("Could not read as JSON:", repr(exc))

print("\nInspection only: no files were changed.")



File: data/processed/deep_learning/lightweight_multifeature_cnn/feature_config.json
{
  "sample_rate": 4000,
  "duration_seconds": 5.0,
  "target_samples": 20000,
  "n_fft": 512,
  "hop_length": 128,
  "n_mels": 64,
  "n_mfcc": 13,
  "n_chroma": 12,
  "fmin": 0,
  "fmax": 2000.0,
  "feature_names": [
    "logmel",
    "mfcc",
    "chroma"
  ],
  "normalization_source": "training split only",
  "training_cycles_for_normalization": 4423
}

File: data/processed/deep_learning/lightweight_multifeature_cnn/normalization_stats.json
{
  "logmel": {
    "count": 44442304,
    "mean": -66.23048066283803,
    "std": 17.66188607911125
  },
  "mfcc": {
    "count": 9027343,
    "mean": -22.509034445067375,
    "std": 137.27441615628354
  },
  "chroma": {
    "count": 8332932,
    "mean": 0.3856557201130974,
    "std": 0.38905858480534755
  }
}

File: data/processed/deep_learning/multifeature_cnn_12b/training_config.json
{
  "experiment": "12B",
  "model": "Regularized Multi-Feature CNN",
  "featur

In [49]:

# Check the libraries required for combined-dataset feature extraction.
# This cell does not extract audio features or modify files.

import importlib
import sys

required_packages = [
    "numpy",
    "pandas",
    "soundfile",
    "librosa",
    "scipy",
    "torch",
    "sklearn",
]

print("Python executable:", sys.executable)
print("\nDependency availability:")

dependency_status = {}

for package in required_packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version not reported")
        dependency_status[package] = True
        print(f"{package}: available ({version})")
    except Exception as exc:
        dependency_status[package] = False
        print(f"{package}: FAILED ({type(exc).__name__}: {exc})")

missing = [
    package for package, available in dependency_status.items()
    if not available
]

print("\nMissing or unusable dependencies:", missing)

if not missing:
    print("All required preprocessing and training libraries are available.")

print("No files were created or modified.")


Python executable: /Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/bin/python3.14

Dependency availability:
numpy: available (2.5.3)
pandas: available (3.0.6)
soundfile: available (0.14.0)
librosa: available (1.0.0)
scipy: available (1.18.1)
torch: available (2.14.1)
sklearn: available (1.9.1)

Missing or unusable dependencies: []
All required preprocessing and training libraries are available.
No files were created or modified.


In [50]:

# Inspect the existing audio loading and feature-extraction implementation.
# Reference only; no notebook execution or file modifications.

import json

reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with reference_notebook.open("r", encoding="utf-8") as f:
    reference_data = json.load(f)

for cell_index in [7, 8, 9, 10, 11, 13]:
    cell = reference_data["cells"][cell_index]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 70}")
    print(f"REFERENCE CODE CELL {cell_index}")
    print(source[:12000])

print("\nInspection only: no files were changed.")



REFERENCE CODE CELL 7

from pathlib import Path
import random
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Match the existing Notebook 12 feature setup
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12
FMIN = 0
FMAX = TARGET_SR / 2

# Select available compute device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Librosa:", librosa.__version__)
print("SoundFile:", sf.__version__)

print("\nFeature configuration:")
print("Sample rate:", TARGET_SR, "Hz")
print("Audio duration:", TARGET_DURATION, "seconds")
print("Samples per cycle:", TARGET_SAMPLES)


In [51]:

import json
import soundfile as sf
from pathlib import Path

# 1. Inspect the existing CNN's class mapping.
reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with reference_notebook.open("r", encoding="utf-8") as f:
    reference_data = json.load(f)

print("=== EXISTING CNN CLASS MAPPING ===")

for i, cell in enumerate(reference_data["cells"]):
    source = "".join(cell.get("source", []))
    if "class_names" in source.lower() or "CLASS_NAMES" in source:
        print(f"\nReference code cell {i}:")
        print(source[:7000])

# 2. Inspect sample rates and channel counts from both datasets.
print("\n=== AUDIO HEADER CHECK ===")

for source_name, frame in [
    ("ICBHI", icbhi_usable),
    ("Fraiwan", eligible_fraiwan),
]:
    print(f"\n{source_name}:")
    sample_rows = frame.sample(
        n=min(5, len(frame)),
        random_state=42,
    )

    for _, row in sample_rows.iterrows():
        path = Path(row["recording_path"])
        info = sf.info(str(path))
        print(
            f"{path.name}: "
            f"sample_rate={info.samplerate}, "
            f"channels={info.channels}, "
            f"frames={info.frames}"
        )

print("\nInspection only: no files were modified.")


=== EXISTING CNN CLASS MAPPING ===

Reference code cell 6:

import numpy as np
import pandas as pd

BASELINE_DIR = ROOT / "data/processed/deep_learning/cnn_baseline"
MAPPING_PATH = ROOT / "data/processed/deep_learning/disease_class_mapping.csv"

train_manifest = pd.read_csv(BASELINE_DIR / "train_cnn_manifest.csv")
val_manifest = pd.read_csv(BASELINE_DIR / "validation_cnn_manifest.csv")
test_manifest = pd.read_csv(BASELINE_DIR / "test_cnn_manifest.csv")
class_mapping = pd.read_csv(MAPPING_PATH)

class_names = (
    class_mapping.sort_values("disease_label")["diagnosis"].tolist()
)
num_classes = len(class_names)

print("Class mapping:")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print("\n=== Manifest validation ===")

for split_name, manifest in [
    ("Train", train_manifest),
    ("Validation", val_manifest),
    ("Test", test_manifest),
]:
    assert "audio_path" in manifest.columns
    assert "disease_label" in manifest.columns
    assert "patient_id" in mani

In [52]:

# Audit sample rates, channels, and durations for every manifest recording.

import soundfile as sf
import pandas as pd
from pathlib import Path

audio_header_records = []

for i, row in combined_manifest.iterrows():
    path = Path(row["recording_path"])
    info = sf.info(str(path))

    audio_header_records.append({
        "source_dataset": row["source_dataset"],
        "source_recording_id": row["source_recording_id"],
        "diagnosis": row["diagnosis"],
        "split": row["split"],
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "duration_seconds": info.duration,
        "frames": info.frames,
    })

    if (i + 1) % 250 == 0 or i + 1 == len(combined_manifest):
        print(f"Checked {i + 1}/{len(combined_manifest)} audio headers")

audio_header_audit = pd.DataFrame(audio_header_records)

print("\n=== SAMPLE RATE COUNTS ===")
display(
    audio_header_audit.groupby(
        ["source_dataset", "sample_rate"]
    ).size().rename("recordings").reset_index()
)

print("\n=== CHANNEL COUNTS ===")
display(
    audio_header_audit.groupby(
        ["source_dataset", "channels"]
    ).size().rename("recordings").reset_index()
)

print("\n=== DURATION SUMMARY ===")
display(
    audio_header_audit.groupby("source_dataset")["duration_seconds"]
    .describe()
    .round(3)
)

print("\n=== VALIDATION ===")
print("Total recordings checked:", len(audio_header_audit))
print("Distinct sample rates:", sorted(audio_header_audit["sample_rate"].unique()))
print("Files with multiple channels:", int((audio_header_audit["channels"] > 1).sum()))
print("Files shorter than 5 seconds:", int((audio_header_audit["duration_seconds"] < 5).sum()))
print("Files at least 5 seconds:", int((audio_header_audit["duration_seconds"] >= 5).sum()))

assert len(audio_header_audit) == len(combined_manifest)
assert audio_header_audit["sample_rate"].gt(0).all()
assert audio_header_audit["channels"].gt(0).all()

print("\nHeader audit complete. No audio files were changed.")


Checked 250/1054 audio headers
Checked 500/1054 audio headers
Checked 750/1054 audio headers
Checked 1000/1054 audio headers
Checked 1054/1054 audio headers

=== SAMPLE RATE COUNTS ===


,source_dataset,sample_rate,recordings
0,Fraiwan,4000,135
1,ICBHI,4000,90
2,ICBHI,10000,6
3,ICBHI,44100,823



=== CHANNEL COUNTS ===


,source_dataset,channels,recordings
0,Fraiwan,1,135
1,ICBHI,1,919



=== DURATION SUMMARY ===


,count,mean,std,min,25%,50%,75%,max
source_dataset,,,,,,,,
Fraiwan,135.0,17.467,6.313,5.936,12.656,15.984,21.936,30.0
ICBHI,919.0,21.494,8.312,7.856,20.000,20.000,20.000,86.2



=== VALIDATION ===
Total recordings checked: 1054
Distinct sample rates: [np.int64(4000), np.int64(10000), np.int64(44100)]
Files with multiple channels: 0
Files shorter than 5 seconds: 0
Files at least 5 seconds: 1054

Header audit complete. No audio files were changed.


In [53]:

from pathlib import Path
import soundfile as sf
import pandas as pd

processed_root = project_root / "data" / "processed"

print("Processed data root:", processed_root)
print("\nTop-level contents:")

for path in sorted(processed_root.iterdir()):
    if path.is_dir():
        print(f"[DIR]  {path.name}")
    else:
        print(f"[FILE] {path.name}")

print("\nDirectories containing audio files:")
audio_extensions = {".wav", ".flac", ".ogg"}

for folder in sorted(p for p in processed_root.rglob("*") if p.is_dir()):
    audio_files = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in audio_extensions
    ]
    if audio_files:
        print(f"{folder.relative_to(processed_root)}: {len(audio_files)} direct audio files")
        for audio_path in audio_files[:3]:
            info = sf.info(str(audio_path))
            print(
                f"  {audio_path.name} | "
                f"sr={info.samplerate} | "
                f"duration={info.duration:.2f}s | "
                f"channels={info.channels}"
            )


Processed data root: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed

Top-level contents:
[FILE] .gitkeep
[DIR]  .ipynb_checkpoints
[DIR]  analysis
[DIR]  audio
[FILE] audio_info_clean.csv
[DIR]  classical_ml
[FILE] cleaning_summary.csv
[FILE] cycles_clean.csv
[DIR]  deep_learning
[DIR]  eda
[DIR]  feature_selection
[DIR]  features
[FILE] patients_clean.csv
[DIR]  pca_clustering
[FILE] processed_cycles.csv
[FILE] processing_errors.csv
[FILE] recordings_clean.csv
[DIR]  sound_classification
[DIR]  unified_phenotype

Directories containing audio files:
audio/cycles/101: 23 direct audio files
  1b1_Al_007.wav | sr=4000 | duration=1.87s | channels=1
  1b1_Al_006.wav | sr=4000 | duration=1.76s | channels=1
  1b1_Al_012.wav | sr=4000 | duration=1.46s | channels=1
audio/cycles/101/.ipynb_checkpoints: 6 direct audio files
  1b1_Al_001-checkpoint.wav | sr=4000 | duration=0.54s | channels=1
  1b1_Al_007-checkpoint.wav | sr=4000 | duration=1.87s | channels=1
  1b1_Pr_011-checkpo

In [54]:

from pathlib import Path
import pandas as pd

train_path = output_dir / "experiment_b_train_manifest.csv"
icbhi_test_path = output_dir / "experiment_b_icbhi_test_manifest.csv"
fraiwan_test_path = output_dir / "experiment_b_fraiwan_test_manifest.csv"

train_manifest_check = pd.read_csv(train_path)
icbhi_test_check = pd.read_csv(icbhi_test_path)
fraiwan_test_check = pd.read_csv(fraiwan_test_path)

experiment_b_all = pd.concat(
    [
        train_manifest_check,
        icbhi_test_check,
        fraiwan_test_check,
    ],
    ignore_index=True,
)

expected_classes = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

print("=== MANIFEST SHAPES ===")
print("Train:", train_manifest_check.shape)
print("ICBHI test:", icbhi_test_check.shape)
print("Fraiwan test:", fraiwan_test_check.shape)
print("Combined:", experiment_b_all.shape)

print("\n=== DIAGNOSIS COUNTS BY SOURCE AND SPLIT ===")
display(
    experiment_b_all.groupby(
        ["source_dataset", "split", "diagnosis"]
    ).size().rename("recordings").reset_index()
)

print("\n=== LABEL COMPATIBILITY CHECK ===")
print("Observed labels:", sorted(experiment_b_all["diagnosis"].dropna().unique()))
print("Expected eight classes:", expected_classes)

unexpected = sorted(
    set(experiment_b_all["diagnosis"].dropna()) - set(expected_classes)
)
missing_classes = sorted(
    set(expected_classes) - set(experiment_b_all["diagnosis"].dropna())
)

print("Unexpected labels:", unexpected)
print("Classes absent from all manifests:", missing_classes)

print("\n=== FRAIWAN DIAGNOSES ===")
display(
    experiment_b_all.loc[
        experiment_b_all["source_dataset"].eq("Fraiwan"),
        ["source_dataset", "patient_id", "diagnosis", "split"],
    ]
    .groupby(["source_dataset", "diagnosis", "split"])
    .agg(recordings=("patient_id", "size"),
         patients=("patient_id", "nunique"))
    .reset_index()
)

assert len(experiment_b_all) == 1054, "Unexpected manifest row count"
assert not experiment_b_all["diagnosis"].isna().any(), "Missing diagnoses found"
assert not unexpected, f"Unexpected labels: {unexpected}"

print("\nStructural label check passed.")
print("Clinical label compatibility still requires source annotation verification.")


=== MANIFEST SHAPES ===
Train: (632, 6)
ICBHI test: (394, 6)
Fraiwan test: (28, 6)
Combined: (1054, 6)

=== DIAGNOSIS COUNTS BY SOURCE AND SPLIT ===


,source_dataset,split,diagnosis,recordings
0,Fraiwan,test,Asthma,19
1,Fraiwan,test,COPD,6
2,Fraiwan,test,Pneumonia,3
3,Fraiwan,train,Asthma,74
4,Fraiwan,train,COPD,21
5,Fraiwan,train,Pneumonia,12
6,ICBHI,test,Bronchiectasis,2
7,ICBHI,test,Bronchiolitis,6
8,ICBHI,test,COPD,362
9,ICBHI,test,Healthy,17



=== LABEL COMPATIBILITY CHECK ===
Observed labels: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Expected eight classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Unexpected labels: []
Classes absent from all manifests: []

=== FRAIWAN DIAGNOSES ===


,source_dataset,diagnosis,split,recordings,patients
0,Fraiwan,Asthma,test,19,7
1,Fraiwan,Asthma,train,74,25
2,Fraiwan,COPD,test,6,2
3,Fraiwan,COPD,train,21,7
4,Fraiwan,Pneumonia,test,3,1
5,Fraiwan,Pneumonia,train,12,4



Structural label check passed.
Clinical label compatibility still requires source annotation verification.


In [55]:

from pathlib import Path
import pandas as pd

search_roots = [
    project_root / "data" / "raw",
    project_root / "data" / "processed",
]

metadata_candidates = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*"):
            if (
                path.is_file()
                and path.suffix.lower() in {".csv", ".xlsx", ".json", ".txt"}
                and any(
                    term in path.name.lower()
                    for term in [
                        "fraiwan", "metadata", "diagnos",
                        "clinical", "label", "subject", "patient",
                        "manifest", "phenotype",
                    ]
                )
                and ".ipynb_checkpoints" not in str(path)
            ):
                metadata_candidates.append(path)

print("Candidate metadata files:", len(metadata_candidates))

for path in sorted(metadata_candidates)[:30]:
    print("\n" + "=" * 80)
    print("FILE:", path.relative_to(project_root))

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=5)
        elif path.suffix.lower() == ".xlsx":
            df = pd.read_excel(path, nrows=5)
        elif path.suffix.lower() == ".json":
            df = pd.read_json(path)
            df = df.head(5)
        else:
            with open(path, encoding="utf-8", errors="replace") as f:
                print("".join(f.readlines()[:8]))
            continue

        print("Columns:", df.columns.tolist())
        display(df.head(3))
    except Exception as exc:
        print("Could not preview:", repr(exc))


Candidate metadata files: 9797

FILE: data/processed/deep_learning/cnn_baseline/cnn_patient_disjoint_manifest.csv
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



FILE: data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path,cnn_split
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,test



FILE: data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv
Columns: ['patient_id', 'recording_id', 'chest_location', 'diagnosis', 'disease_label', 'official_split', 'audio_path', 'cnn_split']


...

,dataset,subject_id,recording_version,source_file,source_audio_path,window_index,start_seconds,end_seconds,original_window_seconds,stored_window_seconds,was_padded,sample_rate,phenotype_label,label_granularity,split,processed_audio_path,quality_status
0,Fraiwan,102,BP,"BP102_N,N,P L L,41,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,0,0.0,5.0,5.0,5.0,False,4000,Normal,recording_level_weak_label,test,data/processed/unified_phenotype/fraiwan/audio...,eligible
1,Fraiwan,102,BP,"BP102_N,N,P L L,41,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1,5.0,10.0,5.0,5.0,False,4000,Normal,recording_level_weak_label,test,data/processed/unified_phenotype/fraiwan/audio...,eligible
2,Fraiwan,102,BP,"BP102_N,N,P L L,41,M.wav",/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2,10.0,15.0,5.0,5.0,False,4000,Normal,recording_level_weak_label,test,data/processed/unified_phenotype/fraiwan/audio...,eligible



FILE: data/processed/unified_phenotype/hf_lung_v1/hf_phenotype_manifest.csv
Columns: ['dataset', 'recording_id', 'recording_date', 'split', 'source_audio_path', 'annotation_path', 'phase_index', 'phase_type', 'start_time', 'end_time', 'duration_seconds', 'sample_rate', 'crackles', 'wheezes', 'phenotype_label', 'label_interpretation', 'overlapping_event_labels', 'processed_audio_path']


,dataset,recording_id,recording_date,split,source_audio_path,annotation_path,phase_index,phase_type,start_time,end_time,duration_seconds,sample_rate,crackles,wheezes,phenotype_label,label_interpretation,overlapping_event_labels,processed_audio_path
0,HF_Lung_V1,steth_20180814_09_37_11,2018-08-14,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1,I,1.500,2.457,0.957,4000,1,0,Crackles,Annotation-derived sound phenotype,D,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,HF_Lung_V1,steth_20180814_09_37_11,2018-08-14,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2,E,2.608,3.637,1.029,4000,1,0,Crackles,Annotation-derived sound phenotype,D,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,HF_Lung_V1,steth_20180814_09_37_11,2018-08-14,train,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3,I,5.227,6.350,1.123,4000,1,0,Crackles,Annotation-derived sound phenotype,D,/Users/vs/Sem 5/Respiratory-Disease-Classifica...



FILE: data/processed/unified_phenotype/icbhi_phenotype_manifest.csv
Columns: ['chest_location', 'acquisition_mode', 'equipment', 'cycle_number', 'start_time', 'end_time', 'sound_label', 'crackles', 'wheezes', 'processed_path', 'mean', 'std', 'variance', 'rms', 'max', 'min', 'peak_to_peak', 'median', 'skewness', 'kurtosis', 'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness', 'spectral_energy', 'dominant_frequency', 'spectral_entropy', 'band_energy_0_500', 'band_energy_500_1000', 'band_energy_1000_1500', 'band_energy_1500_2000', 'mfcc_1_mean', 'mfcc_2_mean', 'mfcc_3_mean', 'mfcc_4_mean', 'mfcc_5_mean', 'mfcc_6_mean', 'mfcc_7_mean', 'mfcc_8_mean', 'mfcc_9_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'wavelet_energy', 'wavelet_entropy', 'wavelet_std', 'audio_key', 'cnn_split', 'official_split', 'patient_id', 'recording_id', 'diagnosis', 'disease_label', 'phenotype_label', 'split', 'dataset', 'label_g

,chest_location,acquisition_mode,equipment,cycle_number,start_time,end_time,sound_label,crackles,wheezes,processed_path,...,official_split,patient_id,recording_id,diagnosis,disease_label,phenotype_label,split,dataset,label_granularity,processed_audio_path
0,Al,sc,Meditron,1,0.036,0.579,Normal,0,0,audio/cycles/101/1b1_Al_001.wav,...,test,101,1b1,URTI,7,Normal,test,ICBHI,respiratory_cycle,audio/cycles/101/1b1_Al_001.wav
1,Al,sc,Meditron,2,0.579,2.450,Normal,0,0,audio/cycles/101/1b1_Al_002.wav,...,test,101,1b1,URTI,7,Normal,test,ICBHI,respiratory_cycle,audio/cycles/101/1b1_Al_002.wav
2,Al,sc,Meditron,3,2.450,3.893,Normal,0,0,audio/cycles/101/1b1_Al_003.wav,...,test,101,1b1,URTI,7,Normal,test,ICBHI,respiratory_cycle,audio/cycles/101/1b1_Al_003.wav


In [56]:

from pathlib import Path
import pandas as pd

fraiwan_window_path = (
    project_root / "data" / "processed" /
    "unified_phenotype" / "fraiwan" / "fraiwan_window_manifest.csv"
)

fraiwan_windows = pd.read_csv(fraiwan_window_path)

print("=== FRAIWAN WINDOW MANIFEST ===")
print("Rows:", len(fraiwan_windows))
print("Label granularity:")
display(fraiwan_windows["label_granularity"].value_counts(dropna=False).rename("rows"))

print("\nSound phenotype labels:")
display(
    fraiwan_windows["phenotype_label"]
    .value_counts(dropna=False)
    .rename("windows")
    .reset_index()
)

print("\n=== ORIGINAL SOURCE-FILE LABEL FIELDS ===")
source_cols = [
    col for col in [
        "source_file", "subject_id", "recording_version",
        "phenotype_label", "label_granularity", "quality_status"
    ]
    if col in fraiwan_windows.columns
]
display(fraiwan_windows[source_cols].drop_duplicates().head(20))

print("\n=== EXPERIMENT B FRAIWAN RECORDING LABELS ===")
fraiwan_manifest = pd.concat(
    [
        pd.read_csv(output_dir / "experiment_b_train_manifest.csv"),
        pd.read_csv(output_dir / "experiment_b_fraiwan_test_manifest.csv"),
    ],
    ignore_index=True,
)
fraiwan_manifest = fraiwan_manifest[
    fraiwan_manifest["source_dataset"].eq("Fraiwan")
]

display(
    fraiwan_manifest[["source_recording_id", "patient_id", "diagnosis", "split"]]
    .drop_duplicates()
    .head(30)
)

print("\n=== INTERPRETATION ===")
print(
    "The window manifest's phenotype_label describes sound phenotype, "
    "not necessarily the patient's disease diagnosis."
)
print(
    "We must verify the original clinical diagnosis field or dataset "
    "documentation before accepting the Experiment B Fraiwan mappings."
)


=== FRAIWAN WINDOW MANIFEST ===
Rows: 1086
Label granularity:


label_granularity
recording_level_weak_label    1086
Name: rows, dtype: int64


Sound phenotype labels:


,phenotype_label,windows
0,Wheezes,402
1,Normal,360
2,Crackles,294
3,Both,30



=== ORIGINAL SOURCE-FILE LABEL FIELDS ===


,source_file,subject_id,recording_version,phenotype_label,label_granularity,quality_status
0,"BP102_N,N,P L L,41,M.wav",102,BP,Normal,recording_level_weak_label,eligible
4,"BP103_N,N,P R U,81,F.wav",103,BP,Normal,recording_level_weak_label,eligible
7,"BP104_Asthma,E W,P L U,45,F.wav",104,BP,Wheezes,recording_level_weak_label,eligible
9,"BP105_Lung Fibrosis,Crep,A U R,44,M.wav",105,BP,Crackles,recording_level_weak_label,eligible
14,"BP106_Asthma,E W,P L U,45,F.wav",106,BP,Wheezes,recording_level_weak_label,eligible
18,"BP107_Asthma,E W,P L U,59,F.wav",107,BP,Wheezes,recording_level_weak_label,eligible
21,"BP108_COPD,E W,P R L ,63,M.wav",108,BP,Wheezes,recording_level_weak_label,eligible
26,"BP109_N,N,P L M,26,M.wav",109,BP,Normal,recording_level_weak_label,eligible
29,"BP10_Asthma,E W,P R U,59,M.wav",10,BP,Wheezes,recording_level_weak_label,eligible
30,"BP110_COPD,E W,P L L,62,M.wav",110,BP,Wheezes,recording_level_weak_label,eligible



=== EXPERIMENT B FRAIWAN RECORDING LABELS ===


,source_recording_id,patient_id,diagnosis,split
525,"BP104_Asthma,E W,P L U,45,F.wav",FRAIWAN_104,Asthma,train
526,"BP106_Asthma,E W,P L U,45,F.wav",FRAIWAN_106,Asthma,train
527,"BP107_Asthma,E W,P L U,59,F.wav",FRAIWAN_107,Asthma,train
528,"BP108_COPD,E W,P R L ,63,M.wav",FRAIWAN_108,COPD,train
529,"BP10_Asthma,E W,P R U,59,M.wav",FRAIWAN_10,Asthma,train
530,"BP110_COPD,E W,P L L,62,M.wav",FRAIWAN_110,COPD,train
531,"BP111_COPD,E W,P R L ,51,M.wav",FRAIWAN_111,COPD,train
532,"BP12_Asthma,E W,P L L,50,F.wav",FRAIWAN_12,Asthma,train
533,"BP17_COPD,E W,A R M,57,M.wav",FRAIWAN_17,COPD,train
534,"BP18_pneumonia,C,P R U,57,M.wav",FRAIWAN_18,Pneumonia,train



=== INTERPRETATION ===
The window manifest's phenotype_label describes sound phenotype, not necessarily the patient's disease diagnosis.
We must verify the original clinical diagnosis field or dataset documentation before accepting the Experiment B Fraiwan mappings.


In [57]:

import re
import pandas as pd

fraiwan_rows = experiment_b_all[
    experiment_b_all["source_dataset"].eq("Fraiwan")
].copy()

def parse_filename_diagnosis(filename):
    # Diagnosis is the text after the recording prefix and before the first comma.
    # Example: BP104_Asthma,E W,... -> Asthma
    name = str(filename).split("/")[-1]
    match = re.match(r"^(?:BP|DP|EP)\d+_([^,]+)", name, flags=re.IGNORECASE)

    if not match:
        return None

    raw_label = match.group(1).strip().lower()

    if raw_label in {"asthma"}:
        return "Asthma"
    if raw_label in {"copd"}:
        return "COPD"
    if raw_label in {"pneumonia"}:
        return "Pneumonia"
    if raw_label in {"n", "normal"}:
        return "Unspecified/Normal"
    return raw_label.title()

fraiwan_rows["filename_diagnosis"] = (
    fraiwan_rows["source_recording_id"].apply(parse_filename_diagnosis)
)

print("=== FILENAME DIAGNOSIS VS EXPERIMENT B LABEL ===")
display(
    fraiwan_rows.groupby(
        ["filename_diagnosis", "diagnosis"]
    ).size().rename("recordings").reset_index()
)

mismatches = fraiwan_rows[
    fraiwan_rows["filename_diagnosis"] != fraiwan_rows["diagnosis"]
]

print("\nMismatches:", len(mismatches))
if len(mismatches):
    display(
        mismatches[
            ["source_recording_id", "filename_diagnosis", "diagnosis", "split"]
        ].head(30)
    )

print("\n=== DISTINCT SOURCE DIAGNOSIS TOKENS ===")
display(
    fraiwan_rows[
        ["source_recording_id", "filename_diagnosis", "diagnosis"]
    ].drop_duplicates()
    .groupby(["filename_diagnosis", "diagnosis"])
    .size()
    .rename("distinct_recordings")
    .reset_index()
)

print(
    "\nThis verifies filename-label consistency only; it does not establish "
    "clinical validity independently of the original dataset documentation."
)


=== FILENAME DIAGNOSIS VS EXPERIMENT B LABEL ===


,filename_diagnosis,diagnosis,recordings
0,Asthma,Asthma,93
1,COPD,COPD,27
2,Pneumonia,Pneumonia,15



Mismatches: 0

=== DISTINCT SOURCE DIAGNOSIS TOKENS ===


,filename_diagnosis,diagnosis,distinct_recordings
0,Asthma,Asthma,93
1,COPD,COPD,27
2,Pneumonia,Pneumonia,15



This verifies filename-label consistency only; it does not establish clinical validity independently of the original dataset documentation.


In [58]:

from pathlib import Path
import pandas as pd
import soundfile as sf

manifest_paths = [
    output_dir / "experiment_b_train_manifest.csv",
    output_dir / "experiment_b_icbhi_test_manifest.csv",
    output_dir / "experiment_b_fraiwan_test_manifest.csv",
]

path_audit = []

for manifest_path in manifest_paths:
    manifest = pd.read_csv(manifest_path)

    for _, row in manifest.iterrows():
        audio_path = Path(row["recording_path"])
        exists = audio_path.is_file()

        record = {
            "source_dataset": row["source_dataset"],
            "split": row["split"],
            "diagnosis": row["diagnosis"],
            "recording_path": str(audio_path),
            "exists": exists,
            "sample_rate": None,
            "duration_seconds": None,
            "error": None,
        }

        if exists:
            try:
                info = sf.info(str(audio_path))
                record["sample_rate"] = info.samplerate
                record["duration_seconds"] = info.duration
            except Exception as exc:
                record["error"] = repr(exc)

        path_audit.append(record)

path_audit_df = pd.DataFrame(path_audit)

print("=== PATH VALIDATION ===")
print("Total rows:", len(path_audit_df))
print("Missing paths:", (~path_audit_df["exists"]).sum())
print("Unreadable files:", path_audit_df["error"].notna().sum())

print("\n=== SAMPLE RATE COUNTS ===")
display(
    path_audit_df.groupby(
        ["source_dataset", "sample_rate"]
    ).size().rename("recordings").reset_index()
)

print("\n=== DURATION SUMMARY ===")
display(
    path_audit_df.groupby("source_dataset")["duration_seconds"]
    .describe()
    .round(3)
)

if (~path_audit_df["exists"]).any():
    display(path_audit_df[~path_audit_df["exists"]].head(20))

if path_audit_df["error"].notna().any():
    display(path_audit_df[path_audit_df["error"].notna()].head(20))

assert path_audit_df["exists"].all(), "Some manifest audio paths are missing."
assert path_audit_df["error"].isna().all(), "Some audio files could not be read."

print("\nAll Experiment B manifest paths are present and readable.")


=== PATH VALIDATION ===
Total rows: 1054
Missing paths: 0
Unreadable files: 0

=== SAMPLE RATE COUNTS ===


,source_dataset,sample_rate,recordings
0,Fraiwan,4000,135
1,ICBHI,4000,90
2,ICBHI,10000,6
3,ICBHI,44100,823



=== DURATION SUMMARY ===


,count,mean,std,min,25%,50%,75%,max
source_dataset,,,,,,,,
Fraiwan,135.0,17.467,6.313,5.936,12.656,15.984,21.936,30.0
ICBHI,919.0,21.494,8.312,7.856,20.000,20.000,20.000,86.2



All Experiment B manifest paths are present and readable.


In [59]:

from pathlib import Path

cache_dir = output_dir / "experiment_b_feature_cache"

print("Cache directory:", cache_dir)
print("Directory exists:", cache_dir.exists())

if cache_dir.exists():
    existing_files = sorted(
        p.relative_to(cache_dir).as_posix()
        for p in cache_dir.rglob("*")
        if p.is_file()
    )
    print("Existing files:", len(existing_files))
    for name in existing_files[:50]:
        print(" -", name)
    if len(existing_files) > 50:
        print(" ... and", len(existing_files) - 50, "more")
else:
    print("No existing cache directory. Safe to plan a new cache.")


Cache directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache
Directory exists: False
No existing cache directory. Safe to plan a new cache.


In [60]:

import json
from pathlib import Path

reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with open(reference_notebook, "r", encoding="utf-8") as f:
    reference_data = json.load(f)

keywords = (
    "fix_audio_length",
    "target_samples",
    "duration_seconds",
    "segment",
    "window",
    "load_processed_audio",
    "extract_features",
)

print("Reference notebook exists:", reference_notebook.exists())

for cell_index, cell in enumerate(reference_data.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    matching_lines = [
        (line_number, line)
        for line_number, line in enumerate(source.splitlines(), start=1)
        if any(keyword in line.lower() for keyword in keywords)
    ]

    if matching_lines:
        print(f"\n--- Code cell {cell_index} ---")
        lines = source.splitlines()
        selected_numbers = {
            line_number
            for line_number, _ in matching_lines
        }

        for line_number in list(selected_numbers):
            selected_numbers.update(
                range(max(1, line_number - 2),
                      min(len(lines), line_number + 3) + 1)
            )

        previous = 0
        for line_number in sorted(selected_numbers):
            if line_number > previous + 1:
                print("...")
            print(f"{line_number:4}: {lines[line_number - 1]}")
            previous = line_number


Reference notebook exists: True

--- Code cell 7 ---
...
  17: TARGET_SR = 4000
  18: TARGET_DURATION = 5.0
  19: TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)
  20: 
  21: N_FFT = 512
  22: HOP_LENGTH = 128
...
  43: print("Sample rate:", TARGET_SR, "Hz")
  44: print("Audio duration:", TARGET_DURATION, "seconds")
  45: print("Samples per cycle:", TARGET_SAMPLES)
  46: print("Log-Mel shape:", (N_MELS, 1 + TARGET_SAMPLES // HOP_LENGTH))
  47: print("MFCC coefficients:", N_MFCC)
  48: print("Chroma bins:", N_CHROMA)
  49: print("FFT:", N_FFT)
  50: print("Hop length:", HOP_LENGTH)
  51: 
  52: assert TARGET_SAMPLES == 20000
  53: assert N_MELS == 64 and N_MFCC == 13 and N_CHROMA == 12
  54: 
  55: print("\nFeature configuration checks PASSED.")

--- Code cell 8 ---
...
   4: import librosa
   5: 
   6: def fix_audio_length(signal, target_samples=TARGET_SAMPLES):
   7:     if len(signal) < target_samples:
   8:         signal = np.pad(
   9:             signal,
  10:             (0, t

In [61]:

import json
from pathlib import Path

reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with open(reference_notebook, "r", encoding="utf-8") as f:
    reference_data = json.load(f)

for cell_index in [7, 8, 9, 10, 11, 13]:
    cell = reference_data["cells"][cell_index]
    source = "".join(cell.get("source", []))

    print(f"\n{'=' * 25} CELL {cell_index} {'=' * 25}")
    print(source)



========================= CELL 7 =========================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Match the existing Notebook 12 feature setup
TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12
FMIN = 0
FMAX = TARGET_SR / 2

# Select available compute device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Librosa:", librosa.__version__)
print("SoundFile:", sf.__version__)

print("\nFeature configuration:")
print("Sample rate:", TARGET_SR, "Hz")
print("Audio duration:", TARGET_DURATION, "seconds")
print("

In [62]:

import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path

manifest_files = {
    "train": output_dir / "experiment_b_train_manifest.csv",
    "icbhi_test": output_dir / "experiment_b_icbhi_test_manifest.csv",
    "fraiwan_test": output_dir / "experiment_b_fraiwan_test_manifest.csv",
}

TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

window_rows = []

for split_name, manifest_path in manifest_files.items():
    manifest = pd.read_csv(manifest_path)

    for _, row in manifest.iterrows():
        info = sf.info(str(row["recording_path"]))
        duration = float(info.duration)

        # Number of non-overlapping windows, padding only the final window.
        n_windows = int(np.ceil(duration / TARGET_DURATION))

        window_rows.append({
            "split": split_name,
            "source_dataset": row["source_dataset"],
            "diagnosis": row["diagnosis"],
            "patient_id": row["patient_id"],
            "recording_path": row["recording_path"],
            "original_sample_rate": int(info.samplerate),
            "duration_seconds": duration,
            "expected_windows": n_windows,
        })

window_plan = pd.DataFrame(window_rows)

print("=== RECORDING AND WINDOW COUNTS ===")
display(
    window_plan.groupby(["split", "source_dataset"])
    .agg(
        recordings=("recording_path", "size"),
        expected_windows=("expected_windows", "sum"),
        min_windows_per_recording=("expected_windows", "min"),
        max_windows_per_recording=("expected_windows", "max"),
    )
    .reset_index()
)

print("\n=== CLASS SUPPORT BY WINDOW (ESTIMATE) ===")
display(
    window_plan.groupby(["split", "diagnosis"])
    .agg(
        recordings=("recording_path", "size"),
        expected_windows=("expected_windows", "sum"),
    )
    .reset_index()
)

total_windows = int(window_plan["expected_windows"].sum())

# Float32 arrays: 64 + 13 + 12 feature values per time frame,
# with 157 frames per 5-second window, plus temporary/runtime overhead.
approx_gib_per_window = (
    (64 + 13 + 12) * 157 * 4
) / (1024 ** 3)

print("\nTotal recordings:", len(window_plan))
print("Estimated 5-second windows:", total_windows)
print(
    "Approximate uncompressed feature payload:",
    f"{total_windows * approx_gib_per_window:.2f} GiB",
)
print(
    "All durations positive:",
    bool((window_plan["duration_seconds"] > 0).all()),
)
print(
    "All windows counts positive:",
    bool((window_plan["expected_windows"] > 0).all()),
)

assert len(window_plan) == 1054
assert (window_plan["duration_seconds"] > 0).all()
assert (window_plan["expected_windows"] > 0).all()


=== RECORDING AND WINDOW COUNTS ===


,split,source_dataset,recordings,expected_windows,min_windows_per_recording,max_windows_per_recording
0,fraiwan_test,Fraiwan,28,114,3,5
1,icbhi_test,ICBHI,394,1667,2,17
2,train,Fraiwan,107,417,2,6
3,train,ICBHI,525,2343,4,18



=== CLASS SUPPORT BY WINDOW (ESTIMATE) ===


,split,diagnosis,recordings,expected_windows
0,fraiwan_test,Asthma,19,75
1,fraiwan_test,COPD,6,24
2,fraiwan_test,Pneumonia,3,15
3,icbhi_test,Bronchiectasis,2,10
4,icbhi_test,Bronchiolitis,6,24
5,icbhi_test,COPD,362,1537
6,icbhi_test,Healthy,17,68
7,icbhi_test,URTI,7,28
8,train,Asthma,75,286
9,train,Bronchiectasis,14,70



Total recordings: 1054
Estimated 5-second windows: 4541
Approximate uncompressed feature payload: 0.24 GiB
All durations positive: True
All windows counts positive: True


In [63]:

import pandas as pd

manifest_paths = {
    "train": output_dir / "experiment_b_train_manifest.csv",
    "icbhi_test": output_dir / "experiment_b_icbhi_test_manifest.csv",
    "fraiwan_test": output_dir / "experiment_b_fraiwan_test_manifest.csv",
}

split_manifests = {
    name: pd.read_csv(path)
    for name, path in manifest_paths.items()
}

CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

all_manifest = pd.concat(
    [
        df.assign(manifest_split=split_name)
        for split_name, df in split_manifests.items()
    ],
    ignore_index=True,
)

print("=== CLASS MAPPING ===")
print("Expected classes:", CLASS_NAMES)
print("Observed diagnoses:", sorted(all_manifest["diagnosis"].unique()))
print(
    "Unexpected labels:",
    sorted(set(all_manifest["diagnosis"]) - set(CLASS_NAMES)),
)

print("\n=== PATIENT OVERLAP BY SOURCE ===")
for source in sorted(all_manifest["source_dataset"].unique()):
    source_df = all_manifest[all_manifest["source_dataset"] == source]
    patient_sets = {
        split_name: set(
            source_df.loc[
                source_df["manifest_split"] == split_name,
                "patient_id",
            ].astype(str)
        )
        for split_name in manifest_paths
    }

    print(f"\n{source}")
    for first, second in [
        ("train", "icbhi_test"),
        ("train", "fraiwan_test"),
        ("icbhi_test", "fraiwan_test"),
    ]:
        overlap = patient_sets[first] & patient_sets[second]
        print(f"{first} vs {second}: {len(overlap)} shared patients")

print("\n=== DUPLICATE RECORDING REFERENCES ===")
print(
    "Duplicate source recording IDs:",
    int(
        all_manifest.duplicated(
            ["source_dataset", "source_recording_id"]
        ).sum()
    ),
)
print(
    "Duplicate paths:",
    int(all_manifest["recording_path"].duplicated().sum()),
)

print("\n=== RECORDING COUNTS BY SPLIT ===")
display(
    all_manifest.groupby(["manifest_split", "source_dataset"])
    .agg(
        recordings=("recording_path", "size"),
        patients=("patient_id", "nunique"),
        classes=("diagnosis", "nunique"),
    )
    .reset_index()
)

assert set(all_manifest["diagnosis"]).issubset(set(CLASS_NAMES))
assert not all_manifest.duplicated(
    ["source_dataset", "source_recording_id"]
).any()
assert not all_manifest["recording_path"].duplicated().any()

print("\nLabel mapping and duplicate-reference checks PASSED.")


=== CLASS MAPPING ===
Expected classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Observed diagnoses: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Unexpected labels: []

=== PATIENT OVERLAP BY SOURCE ===

Fraiwan
train vs icbhi_test: 0 shared patients
train vs fraiwan_test: 0 shared patients
icbhi_test vs fraiwan_test: 0 shared patients

ICBHI
train vs icbhi_test: 0 shared patients
train vs fraiwan_test: 0 shared patients
icbhi_test vs fraiwan_test: 0 shared patients

=== DUPLICATE RECORDING REFERENCES ===
Duplicate source recording IDs: 0
Duplicate paths: 0

=== RECORDING COUNTS BY SPLIT ===


,manifest_split,source_dataset,recordings,patients,classes
0,fraiwan_test,Fraiwan,28,10,3
1,icbhi_test,ICBHI,394,49,5
2,train,Fraiwan,107,36,3
3,train,ICBHI,525,77,8



Label mapping and duplicate-reference checks PASSED.


In [64]:

from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf

cache_dir = output_dir / "experiment_b_feature_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

window_manifest_path = cache_dir / "experiment_b_window_manifest.csv"

if window_manifest_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing file: {window_manifest_path}"
    )

manifest_paths = {
    "train": output_dir / "experiment_b_train_manifest.csv",
    "icbhi_test": output_dir / "experiment_b_icbhi_test_manifest.csv",
    "fraiwan_test": output_dir / "experiment_b_fraiwan_test_manifest.csv",
}

class_to_index = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

window_records = []

for split_name, manifest_path in manifest_paths.items():
    manifest = pd.read_csv(manifest_path)

    for _, row in manifest.iterrows():
        audio_path = str(row["recording_path"])
        info = sf.info(audio_path)

        duration = float(info.duration)
        n_windows = int(np.ceil(duration / TARGET_DURATION))

        for window_index in range(n_windows):
            start_seconds = window_index * TARGET_DURATION
            end_seconds = min(
                start_seconds + TARGET_DURATION,
                duration,
            )
            valid_duration = end_seconds - start_seconds

            window_records.append({
                "source_dataset": row["source_dataset"],
                "source_recording_id": row["source_recording_id"],
                "patient_id": row["patient_id"],
                "diagnosis": row["diagnosis"],
                "label_index": class_to_index[row["diagnosis"]],
                "split": split_name,
                "recording_path": audio_path,
                "window_index": window_index,
                "start_seconds": start_seconds,
                "end_seconds": end_seconds,
                "valid_duration_seconds": valid_duration,
                "requires_zero_padding": (
                    valid_duration < TARGET_DURATION
                ),
                "original_sample_rate": int(info.samplerate),
            })

window_manifest = pd.DataFrame(window_records)

print("Window manifest rows:", len(window_manifest))
print("Unique recordings:", window_manifest["recording_path"].nunique())
print("Padded final windows:", int(window_manifest["requires_zero_padding"].sum()))
print("Label range:", (
    int(window_manifest["label_index"].min()),
    int(window_manifest["label_index"].max()),
))

print("\n=== WINDOWS BY SPLIT ===")
display(
    window_manifest.groupby(["split", "source_dataset"])
    .agg(
        windows=("window_index", "size"),
        recordings=("recording_path", "nunique"),
        patients=("patient_id", "nunique"),
        padded_windows=("requires_zero_padding", "sum"),
    )
    .reset_index()
)

assert len(window_manifest) == 4541
assert window_manifest["recording_path"].nunique() == 1054
assert window_manifest["label_index"].between(0, 7).all()
assert (
    window_manifest.groupby("recording_path")["split"].nunique().max()
    == 1
)
assert (
    window_manifest.groupby("recording_path")["diagnosis"].nunique().max()
    == 1
)

window_manifest.to_csv(window_manifest_path, index=False)

saved_window_manifest = pd.read_csv(window_manifest_path)
assert len(saved_window_manifest) == len(window_manifest)
assert saved_window_manifest["recording_path"].nunique() == 1054

print("\nSaved and verified:", window_manifest_path)
print("Rows:", len(saved_window_manifest))


Window manifest rows: 4541
Unique recordings: 1054
Padded final windows: 233
Label range: (0, 7)

=== WINDOWS BY SPLIT ===


,split,source_dataset,windows,recordings,patients,padded_windows
0,fraiwan_test,Fraiwan,114,28,10,28
1,icbhi_test,ICBHI,1667,394,49,70
2,train,Fraiwan,417,107,36,92
3,train,ICBHI,2343,525,77,43



Saved and verified: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_window_manifest.csv
Rows: 4541


In [65]:

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

TARGET_SR = 4000
TARGET_DURATION = 5.0
TARGET_SAMPLES = int(TARGET_SR * TARGET_DURATION)

def load_audio_window(row):
    """Load, resample, and prepare one manifest window in memory."""
    audio, original_sr = sf.read(
        str(row["recording_path"]),
        start=0,
        dtype="float32",
        always_2d=False,
    )

    # Convert stereo/multichannel audio to mono.
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample the waveform, not merely its sample-rate metadata.
    if original_sr != TARGET_SR:
        audio = librosa.resample(
            audio,
            orig_sr=original_sr,
            target_sr=TARGET_SR,
        )

    start_sample = int(round(row["start_seconds"] * TARGET_SR))
    end_sample = int(round(row["end_seconds"] * TARGET_SR))

    window = audio[start_sample:end_sample]

    # Correct rounding differences at window boundaries.
    if len(window) > TARGET_SAMPLES:
        window = window[:TARGET_SAMPLES]

    if len(window) < TARGET_SAMPLES:
        window = np.pad(
            window,
            (0, TARGET_SAMPLES - len(window)),
            mode="constant",
        )

    window = np.asarray(window, dtype=np.float32)

    if window.shape != (TARGET_SAMPLES,):
        raise ValueError(f"Unexpected waveform shape: {window.shape}")

    if not np.isfinite(window).all():
        raise ValueError("Non-finite audio samples detected.")

    return window, int(original_sr)


# Select examples covering both datasets and differing sample rates.
sample_candidates = (
    window_manifest
    .sort_values(["source_dataset", "original_sample_rate"])
    .groupby(["source_dataset", "original_sample_rate"], as_index=False)
    .first()
)

print("=== AUDIO WINDOW SMOKE TEST ===")

for _, row in sample_candidates.iterrows():
    waveform, original_sr = load_audio_window(row)

    print({
        "dataset": row["source_dataset"],
        "original_sr": original_sr,
        "target_sr": TARGET_SR,
        "window_shape": waveform.shape,
        "duration_seconds": len(waveform) / TARGET_SR,
        "finite": bool(np.isfinite(waveform).all()),
        "max_abs_amplitude": float(np.max(np.abs(waveform))),
        "recording": row["source_recording_id"],
        "window_index": int(row["window_index"]),
    })

    assert waveform.shape == (20000,)
    assert np.isfinite(waveform).all()

print("\nAudio resampling and window preparation smoke test PASSED.")


=== AUDIO WINDOW SMOKE TEST ===
{'dataset': 'Fraiwan', 'original_sr': 4000, 'target_sr': 4000, 'window_shape': (20000,), 'duration_seconds': 5.0, 'finite': True, 'max_abs_amplitude': 1.0, 'recording': 'BP104_Asthma,E W,P L U,45,F.wav', 'window_index': 0}
{'dataset': 'ICBHI', 'original_sr': 4000, 'target_sr': 4000, 'window_shape': (20000,), 'duration_seconds': 5.0, 'finite': True, 'max_abs_amplitude': 0.5145263671875, 'recording': '110_1b1_Pr_sc_Meditron', 'window_index': 0}
{'dataset': 'ICBHI', 'original_sr': 10000, 'target_sr': 4000, 'window_shape': (20000,), 'duration_seconds': 5.0, 'finite': True, 'max_abs_amplitude': 1.0011868476867676, 'recording': '223_1b1_Pr_sc_Meditron', 'window_index': 0}
{'dataset': 'ICBHI', 'original_sr': 44100, 'target_sr': 4000, 'window_shape': (20000,), 'duration_seconds': 5.0, 'finite': True, 'max_abs_amplitude': 1.000124216079712, 'recording': '103_2b2_Ar_mc_LittC2SE', 'window_index': 0}

Audio resampling and window preparation smoke test PASSED.


In [66]:

import numpy as np
import librosa

N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
N_MFCC = 13
N_CHROMA = 12
FMIN = 0
FMAX = TARGET_SR / 2

def extract_window_features(waveform):
    logmel_power = librosa.feature.melspectrogram(
        y=waveform,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0,
    )

    logmel = librosa.power_to_db(logmel_power, ref=np.max)

    mfcc = librosa.feature.mfcc(
        y=waveform,
        sr=TARGET_SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
    )

    chroma = librosa.feature.chroma_stft(
        y=waveform,
        sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_chroma=N_CHROMA,
        tuning=0.0,
    )

    features = {
        "logmel": logmel.astype(np.float32),
        "mfcc": mfcc.astype(np.float32),
        "chroma": chroma.astype(np.float32),
    }

    expected_shapes = {
        "logmel": (64, 157),
        "mfcc": (13, 157),
        "chroma": (12, 157),
    }

    for name, feature in features.items():
        assert feature.shape == expected_shapes[name], (
            f"{name}: expected {expected_shapes[name]}, "
            f"got {feature.shape}"
        )
        assert np.isfinite(feature).all(), (
            f"Non-finite values in {name}"
        )

    return features


print("=== FEATURE EXTRACTION SMOKE TEST ===")

for _, row in sample_candidates.iterrows():
    waveform, original_sr = load_audio_window(row)
    features = extract_window_features(waveform)

    print(
        f"\n{row['source_dataset']} | "
        f"{original_sr} Hz -> {TARGET_SR} Hz | "
        f"{row['source_recording_id']}"
    )

    for name, feature in features.items():
        print(
            f"{name}: shape={feature.shape}, "
            f"min={feature.min():.4f}, "
            f"max={feature.max():.4f}, "
            f"finite={np.isfinite(feature).all()}"
        )

print("\nAll feature shapes and finite-value checks PASSED.")


=== FEATURE EXTRACTION SMOKE TEST ===

Fraiwan | 4000 Hz -> 4000 Hz | BP104_Asthma,E W,P L U,45,F.wav
logmel: shape=(64, 157), min=-80.0000, max=0.0000, finite=True
mfcc: shape=(13, 157), min=-542.8058, max=105.7113, finite=True
chroma: shape=(12, 157), min=0.2230, max=1.0000, finite=True

ICBHI | 4000 Hz -> 4000 Hz | 110_1b1_Pr_sc_Meditron
logmel: shape=(64, 157), min=-80.0000, max=0.0000, finite=True
mfcc: shape=(13, 157), min=-523.0038, max=148.5830, finite=True
chroma: shape=(12, 157), min=0.1937, max=1.0000, finite=True

ICBHI | 10000 Hz -> 4000 Hz | 223_1b1_Pr_sc_Meditron
logmel: shape=(64, 157), min=-80.0000, max=0.0000, finite=True
mfcc: shape=(13, 157), min=-511.9649, max=171.2933, finite=True
chroma: shape=(12, 157), min=0.0553, max=1.0000, finite=True

ICBHI | 44100 Hz -> 4000 Hz | 103_2b2_Ar_mc_LittC2SE
logmel: shape=(64, 157), min=-80.0000, max=0.0000, finite=True
mfcc: shape=(13, 157), min=-512.2887, max=130.4929, finite=True
chroma: shape=(12, 157), min=0.2853, max=1.000

In [67]:

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

cache_dir = output_dir / "experiment_b_feature_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

window_manifest_path = cache_dir / "experiment_b_window_manifest.csv"
normalization_path = cache_dir / "experiment_b_normalization_stats.json"
feature_config_path = cache_dir / "experiment_b_feature_config.json"

# Protect all existing artifacts.
protected_paths = [
    normalization_path,
    feature_config_path,
    cache_dir / "train_features.npz",
    cache_dir / "icbhi_test_features.npz",
    cache_dir / "fraiwan_test_features.npz",
]

existing_protected = [str(p) for p in protected_paths if p.exists()]
if existing_protected:
    raise FileExistsError(
        "Refusing to overwrite existing artifacts:\n"
        + "\n".join(existing_protected)
    )

# The window manifest must already exist and match the audited count.
assert window_manifest_path.is_file()
window_manifest = pd.read_csv(window_manifest_path)
assert len(window_manifest) == 4541
assert window_manifest["recording_path"].nunique() == 1054

FEATURE_CONFIG = {
    "sample_rate": TARGET_SR,
    "duration_seconds": TARGET_DURATION,
    "target_samples": TARGET_SAMPLES,
    "windowing": "non-overlapping; final partial window zero-padded",
    "resampling": "librosa.resample",
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "n_mels": N_MELS,
    "n_mfcc": N_MFCC,
    "n_chroma": N_CHROMA,
    "fmin": FMIN,
    "fmax": FMAX,
    "feature_names": ["logmel", "mfcc", "chroma"],
    "normalization_source": "Experiment B training windows only",
    "class_names": CLASS_NAMES,
    "window_manifest": window_manifest_path.name,
    "label_granularity": (
        "recording-level diagnosis inherited by each window; "
        "window labels are weak labels"
    ),
}

def extract_row_features(row):
    waveform, original_sr = load_audio_window(row)
    features = extract_window_features(waveform)
    return features

def cache_metadata_columns():
    return [
        "source_dataset",
        "source_recording_id",
        "patient_id",
        "diagnosis",
        "label_index",
        "split",
        "recording_path",
        "window_index",
        "start_seconds",
        "end_seconds",
        "valid_duration_seconds",
        "requires_zero_padding",
        "original_sample_rate",
    ]

print("Cache directory:", cache_dir)
print("Window rows:", len(window_manifest))
print("Unique recordings:", window_manifest["recording_path"].nunique())
print("Existing protected cache artifacts:", len(existing_protected))
print("Feature configuration defined.")
print("No feature files have been created yet.")


Cache directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache
Window rows: 4541
Unique recordings: 1054
Existing protected cache artifacts: 0
Feature configuration defined.
No feature files have been created yet.


In [68]:

import numpy as np
import soundfile as sf
import librosa

def load_full_recording(audio_path):
    """Load one complete recording and resample it to TARGET_SR."""
    signal, original_sr = sf.read(
        str(audio_path),
        dtype="float32",
        always_2d=False,
    )

    if signal.ndim > 1:
        signal = signal.mean(axis=1)

    if original_sr != TARGET_SR:
        signal = librosa.resample(
            signal,
            orig_sr=original_sr,
            target_sr=TARGET_SR,
        )

    signal = np.asarray(signal, dtype=np.float32)

    if len(signal) == 0 or not np.isfinite(signal).all():
        raise ValueError(f"Invalid audio signal: {audio_path}")

    return signal, int(original_sr)


def make_recording_windows(signal):
    """Create non-overlapping windows and zero-pad the final short window."""
    windows = []

    for start in range(0, len(signal), TARGET_SAMPLES):
        window = signal[start:start + TARGET_SAMPLES]
        valid_samples = len(window)

        if valid_samples < TARGET_SAMPLES:
            window = np.pad(
                window,
                (0, TARGET_SAMPLES - valid_samples),
                mode="constant",
            )

        windows.append(
            (
                np.asarray(window, dtype=np.float32),
                valid_samples / TARGET_SR,
            )
        )

    return windows


print("=== RECORDING-LEVEL WINDOW TEST ===")

for _, row in sample_candidates.iterrows():
    signal, original_sr = load_full_recording(row["recording_path"])
    windows = make_recording_windows(signal)

    expected_count = int(np.ceil(
        len(signal) / TARGET_SAMPLES
    ))

    print({
        "dataset": row["source_dataset"],
        "original_sample_rate": original_sr,
        "resampled_samples": len(signal),
        "duration_seconds": round(len(signal) / TARGET_SR, 3),
        "windows": len(windows),
        "expected_windows": expected_count,
        "first_window_shape": windows[0][0].shape,
        "last_window_valid_seconds": round(windows[-1][1], 3),
    })

    assert len(windows) == expected_count
    assert all(wave.shape == (TARGET_SAMPLES,) for wave, _ in windows)
    assert all(np.isfinite(wave).all() for wave, _ in windows)

print("\nRecording-level loading and windowing PASSED.")


=== RECORDING-LEVEL WINDOW TEST ===
{'dataset': 'Fraiwan', 'original_sample_rate': 4000, 'resampled_samples': 47040, 'duration_seconds': 11.76, 'windows': 3, 'expected_windows': 3, 'first_window_shape': (20000,), 'last_window_valid_seconds': 1.76}
{'dataset': 'ICBHI', 'original_sample_rate': 4000, 'resampled_samples': 285800, 'duration_seconds': 71.45, 'windows': 15, 'expected_windows': 15, 'first_window_shape': (20000,), 'last_window_valid_seconds': 1.45}
{'dataset': 'ICBHI', 'original_sample_rate': 10000, 'resampled_samples': 131600, 'duration_seconds': 32.9, 'windows': 7, 'expected_windows': 7, 'first_window_shape': (20000,), 'last_window_valid_seconds': 2.9}
{'dataset': 'ICBHI', 'original_sample_rate': 44100, 'resampled_samples': 80000, 'duration_seconds': 20.0, 'windows': 4, 'expected_windows': 4, 'first_window_shape': (20000,), 'last_window_valid_seconds': 5.0}

Recording-level loading and windowing PASSED.


In [69]:

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

train_cache_path = cache_dir / "train_features.npz"
train_metadata_path = cache_dir / "train_window_metadata.csv"
normalization_path = cache_dir / "experiment_b_normalization_stats.json"

for path in [train_cache_path, train_metadata_path, normalization_path]:
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing file: {path}"
        )

train_windows = window_manifest[
    window_manifest["split"] == "train"
].copy().reset_index(drop=True)

expected_train_windows = 2760
assert len(train_windows) == expected_train_windows

n = len(train_windows)
logmel_array = np.empty((n, 1, N_MELS, 157), dtype=np.float32)
mfcc_array = np.empty((n, 1, N_MFCC, 157), dtype=np.float32)
chroma_array = np.empty((n, 1, N_CHROMA, 157), dtype=np.float32)

metadata_records = []
feature_stats = {
    name: {"count": 0, "sum": 0.0, "sum_sq": 0.0}
    for name in ["logmel", "mfcc", "chroma"]
}

# Group windows by recording to avoid loading the same audio repeatedly.
start_time = time.time()
grouped = train_windows.groupby(
    ["source_dataset", "recording_path"],
    sort=False,
)

row_position = 0
n_recordings = grouped.ngroups

print(
    f"Extracting {n} training windows from "
    f"{n_recordings} recordings..."
)

for recording_number, (_, group) in enumerate(grouped, start=1):
    first_row = group.iloc[0]
    signal, original_sr = load_full_recording(
        first_row["recording_path"]
    )

    windows = make_recording_windows(signal)

    for _, row in group.iterrows():
        window_index = int(row["window_index"])

        if window_index >= len(windows):
            raise ValueError(
                f"Window index {window_index} exceeds windows available "
                f"for {row['recording_path']}"
            )

        waveform, valid_duration = windows[window_index]
        features = extract_window_features(waveform)

        logmel = features["logmel"]
        mfcc = features["mfcc"]
        chroma = features["chroma"]

        logmel_array[row_position, 0] = logmel
        mfcc_array[row_position, 0] = mfcc
        chroma_array[row_position, 0] = chroma

        for name, feature in features.items():
            values = feature.astype(np.float64)
            stats = feature_stats[name]
            stats["count"] += values.size
            stats["sum"] += values.sum()
            stats["sum_sq"] += np.square(values).sum()

        metadata_records.append({
            **row.to_dict(),
            "valid_duration_seconds": valid_duration,
            "resampled_sample_rate": TARGET_SR,
        })

        row_position += 1

    if recording_number % 25 == 0 or recording_number == n_recordings:
        print(
            f"Recordings: {recording_number}/{n_recordings} | "
            f"Windows: {row_position}/{n}"
        )

assert row_position == n

# Calculate training-only global feature statistics.
normalization_stats = {}

for name, stats in feature_stats.items():
    count = stats["count"]
    mean = stats["sum"] / count
    variance = max(stats["sum_sq"] / count - mean**2, 0.0)
    std = float(np.sqrt(variance))

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        raise ValueError(f"Invalid normalization statistics: {name}")

    normalization_stats[name] = {
        "count": int(count),
        "mean": float(mean),
        "std": std,
    }

# Normalize training features using training-only statistics.
logmel_array = (
    logmel_array - normalization_stats["logmel"]["mean"]
) / normalization_stats["logmel"]["std"]

mfcc_array = (
    mfcc_array - normalization_stats["mfcc"]["mean"]
) / normalization_stats["mfcc"]["std"]

chroma_array = (
    chroma_array - normalization_stats["chroma"]["mean"]
) / normalization_stats["chroma"]["std"]

for name, array in [
    ("logmel", logmel_array),
    ("mfcc", mfcc_array),
    ("chroma", chroma_array),
]:
    if not np.isfinite(array).all():
        raise ValueError(f"Non-finite normalized values in {name}")

labels = train_windows["label_index"].to_numpy(dtype=np.int64)
paths = train_windows["recording_path"].astype(str).to_numpy()
metadata_df = pd.DataFrame(metadata_records)

# Save caches to temporary paths, then atomically rename.
tmp_cache = cache_dir / "train_features.tmp.npz"
tmp_metadata = cache_dir / "train_window_metadata.tmp.csv"
tmp_normalization = cache_dir / "experiment_b_normalization_stats.tmp.json"

np.savez_compressed(
    tmp_cache,
    logmel=logmel_array,
    mfcc=mfcc_array,
    chroma=chroma_array,
    labels=labels,
    paths=paths,
)

metadata_df.to_csv(tmp_metadata, index=False)

with open(tmp_normalization, "w", encoding="utf-8") as f:
    json.dump(normalization_stats, f, indent=2)

tmp_cache.replace(train_cache_path)
tmp_metadata.replace(train_metadata_path)
tmp_normalization.replace(normalization_path)

print("\n=== TRAINING CACHE SUMMARY ===")
print("Windows:", n)
print("Recordings:", metadata_df["recording_path"].nunique())
print("Feature shapes:")
print("  Log-Mel:", logmel_array.shape)
print("  MFCC:", mfcc_array.shape)
print("  Chroma:", chroma_array.shape)

print("\nTraining-only normalization:")
display(pd.DataFrame(normalization_stats).T)

print("\nSaved files:")
for path in [train_cache_path, train_metadata_path, normalization_path]:
    print(path.name, f"{path.stat().st_size / (1024**2):.2f} MiB")

print(f"Elapsed: {(time.time() - start_time) / 60:.2f} minutes")
print("Training feature extraction and normalization complete.")


Extracting 2760 training windows from 632 recordings...
Recordings: 25/632 | Windows: 100/2760
Recordings: 50/632 | Windows: 253/2760
Recordings: 75/632 | Windows: 409/2760
Recordings: 100/632 | Windows: 509/2760
Recordings: 125/632 | Windows: 609/2760
Recordings: 150/632 | Windows: 709/2760
Recordings: 175/632 | Windows: 809/2760
Recordings: 200/632 | Windows: 909/2760
Recordings: 225/632 | Windows: 1009/2760
Recordings: 250/632 | Windows: 1157/2760
Recordings: 275/632 | Windows: 1257/2760
Recordings: 300/632 | Windows: 1357/2760
Recordings: 325/632 | Windows: 1502/2760
Recordings: 350/632 | Windows: 1602/2760
Recordings: 375/632 | Windows: 1702/2760
Recordings: 400/632 | Windows: 1802/2760
Recordings: 425/632 | Windows: 1903/2760
Recordings: 450/632 | Windows: 2009/2760
Recordings: 475/632 | Windows: 2109/2760
Recordings: 500/632 | Windows: 2211/2760
Recordings: 525/632 | Windows: 2343/2760
Recordings: 550/632 | Windows: 2435/2760
Recordings: 575/632 | Windows: 2538/2760
Recordings: 

,count,mean,std
logmel,27732480.0,-58.349869,17.919855
mfcc,5633160.0,-9.563521,137.709747
chroma,5199840.0,0.693031,0.246216



Saved files:
train_features.npz 118.16 MiB
train_window_metadata.csv 0.48 MiB
experiment_b_normalization_stats.json 0.00 MiB
Elapsed: 0.44 minutes
Training feature extraction and normalization complete.


In [71]:

import json
import numpy as np
import pandas as pd

train_cache_path = cache_dir / "train_features.npz"
train_metadata_path = cache_dir / "train_window_metadata.csv"
normalization_path = cache_dir / "experiment_b_normalization_stats.json"

# This file was created by our own cache-building cell.
with np.load(train_cache_path, allow_pickle=True) as cached:
    logmel = cached["logmel"]
    mfcc = cached["mfcc"]
    chroma = cached["chroma"]
    labels = cached["labels"]
    paths = cached["paths"].astype(str)

metadata = pd.read_csv(train_metadata_path)

with open(normalization_path, "r", encoding="utf-8") as f:
    saved_stats = json.load(f)

print("=== TRAIN CACHE VALIDATION ===")
print("Log-Mel:", logmel.shape, logmel.dtype)
print("MFCC:", mfcc.shape, mfcc.dtype)
print("Chroma:", chroma.shape, chroma.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Paths:", paths.shape)
print("Metadata rows:", len(metadata))
print("Unique recordings:", metadata["recording_path"].nunique())

assert logmel.shape == (2760, 1, 64, 157)
assert mfcc.shape == (2760, 1, 13, 157)
assert chroma.shape == (2760, 1, 12, 157)
assert labels.shape == (2760,)
assert len(paths) == len(metadata) == 2760
assert np.isin(labels, np.arange(8)).all()

assert np.isfinite(logmel).all()
assert np.isfinite(mfcc).all()
assert np.isfinite(chroma).all()

assert np.array_equal(
    paths,
    metadata["recording_path"].astype(str).to_numpy(),
)
assert np.array_equal(
    labels,
    metadata["label_index"].to_numpy(dtype=np.int64),
)

print("\n=== NORMALIZED TRAINING ARRAY SUMMARY ===")
for name, array in [
    ("logmel", logmel),
    ("mfcc", mfcc),
    ("chroma", chroma),
]:
    print(
        name,
        "mean =", round(float(array.mean()), 5),
        "std =", round(float(array.std()), 5),
        "finite =", bool(np.isfinite(array).all()),
    )

print("\n=== SAVED NORMALIZATION PARAMETERS ===")
display(pd.DataFrame(saved_stats).T)

print("\nTraining cache validation PASSED.")


=== TRAIN CACHE VALIDATION ===
Log-Mel: (2760, 1, 64, 157) float32
MFCC: (2760, 1, 13, 157) float32
Chroma: (2760, 1, 12, 157) float32
Labels: (2760,) int64
Paths: (2760,)
Metadata rows: 2760
Unique recordings: 632

=== NORMALIZED TRAINING ARRAY SUMMARY ===
logmel mean = -0.0 std = 1.0 finite = True
mfcc mean = -0.0 std = 1.0 finite = True
chroma mean = -0.0 std = 1.0 finite = True

=== SAVED NORMALIZATION PARAMETERS ===


,count,mean,std
logmel,27732480.0,-58.349869,17.919855
mfcc,5633160.0,-9.563521,137.709747
chroma,5199840.0,0.693031,0.246216



Training cache validation PASSED.


In [72]:

import time
import numpy as np
import pandas as pd

normalization_path = cache_dir / "experiment_b_normalization_stats.json"

with open(normalization_path, "r", encoding="utf-8") as f:
    normalization_stats = json.load(f)

test_splits = ["icbhi_test", "fraiwan_test"]

for split_name in test_splits:
    print(f"\n{'=' * 18} {split_name.upper()} {'=' * 18}")

    cache_path = cache_dir / f"{split_name}_features.npz"
    metadata_path = cache_dir / f"{split_name}_window_metadata.csv"

    if cache_path.exists() or metadata_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing files for {split_name}"
        )

    split_df = (
        window_manifest[window_manifest["split"] == split_name]
        .copy()
        .reset_index(drop=True)
    )

    n = len(split_df)
    logmel_array = np.empty((n, 1, N_MELS, 157), dtype=np.float32)
    mfcc_array = np.empty((n, 1, N_MFCC, 157), dtype=np.float32)
    chroma_array = np.empty((n, 1, N_CHROMA, 157), dtype=np.float32)

    metadata_records = []
    start_time = time.time()

    grouped = split_df.groupby(
        ["source_dataset", "recording_path"],
        sort=False,
    )

    row_position = 0

    for recording_number, (_, group) in enumerate(
        grouped, start=1
    ):
        first_row = group.iloc[0]
        signal, original_sr = load_full_recording(
            first_row["recording_path"]
        )
        windows = make_recording_windows(signal)

        for _, row in group.iterrows():
            window_index = int(row["window_index"])

            if window_index >= len(windows):
                raise ValueError(
                    f"Invalid window index for {row['recording_path']}"
                )

            waveform, valid_duration = windows[window_index]
            features = extract_window_features(waveform)

            logmel_array[row_position, 0] = (
                features["logmel"] - normalization_stats["logmel"]["mean"]
            ) / normalization_stats["logmel"]["std"]

            mfcc_array[row_position, 0] = (
                features["mfcc"] - normalization_stats["mfcc"]["mean"]
            ) / normalization_stats["mfcc"]["std"]

            chroma_array[row_position, 0] = (
                features["chroma"] - normalization_stats["chroma"]["mean"]
            ) / normalization_stats["chroma"]["std"]

            metadata_records.append({
                **row.to_dict(),
                "valid_duration_seconds": valid_duration,
                "resampled_sample_rate": TARGET_SR,
            })

            row_position += 1

        if recording_number % 25 == 0 or recording_number == grouped.ngroups:
            print(
                f"Recordings: {recording_number}/{grouped.ngroups} | "
                f"Windows: {row_position}/{n}"
            )

    assert row_position == n

    for name, array in [
        ("logmel", logmel_array),
        ("mfcc", mfcc_array),
        ("chroma", chroma_array),
    ]:
        if not np.isfinite(array).all():
            raise ValueError(
                f"Non-finite values in {split_name} {name}"
            )

    labels = split_df["label_index"].to_numpy(dtype=np.int64)
    paths = split_df["recording_path"].astype(str).to_numpy()
    metadata_df = pd.DataFrame(metadata_records)

    # Unicode paths avoid object arrays in the saved NPZ.
    unicode_paths = np.asarray(paths, dtype=str)

    tmp_cache = cache_dir / f"{split_name}_features.tmp.npz"
    tmp_metadata = cache_dir / f"{split_name}_window_metadata.tmp.csv"

    np.savez_compressed(
        tmp_cache,
        logmel=logmel_array,
        mfcc=mfcc_array,
        chroma=chroma_array,
        labels=labels,
        paths=unicode_paths,
    )
    metadata_df.to_csv(tmp_metadata, index=False)

    tmp_cache.replace(cache_path)
    tmp_metadata.replace(metadata_path)

    print("Saved cache:", cache_path.name)
    print("Saved metadata:", metadata_path.name)
    print("Windows:", n)
    print("Recordings:", metadata_df["recording_path"].nunique())
    print("Cache size:", f"{cache_path.stat().st_size / (1024**2):.2f} MiB")
    print("Elapsed:", f"{(time.time() - start_time) / 60:.2f} minutes")

print("\nBoth test feature caches created.")



================== ICBHI_TEST ==================
Recordings: 25/394 | Windows: 107/1667
Recordings: 50/394 | Windows: 199/1667
Recordings: 75/394 | Windows: 298/1667
Recordings: 100/394 | Windows: 398/1667
Recordings: 125/394 | Windows: 498/1667
Recordings: 150/394 | Windows: 598/1667
Recordings: 175/394 | Windows: 699/1667
Recordings: 200/394 | Windows: 800/1667
Recordings: 225/394 | Windows: 900/1667
Recordings: 250/394 | Windows: 1001/1667
Recordings: 275/394 | Windows: 1101/1667
Recordings: 300/394 | Windows: 1202/1667
Recordings: 325/394 | Windows: 1312/1667
Recordings: 350/394 | Windows: 1412/1667
Recordings: 375/394 | Windows: 1512/1667
Recordings: 394/394 | Windows: 1667/1667
Saved cache: icbhi_test_features.npz
Saved metadata: icbhi_test_window_metadata.csv
Windows: 1667
Recordings: 394
Cache size: 74.12 MiB
Elapsed: 0.26 minutes

================== FRAIWAN_TEST ==================
Recordings: 25/28 | Windows: 99/114
Recordings: 28/28 | Windows: 114/114
Saved cache: fraiwan_te

In [74]:

import numpy as np
import pandas as pd
import json

cache_specs = {
    "train": {
        "features": cache_dir / "train_features.npz",
        "metadata": cache_dir / "train_window_metadata.csv",
        "expected_windows": 2760,
        "expected_recordings": 632,
        "allow_pickle": True,  # Created by our own training-cache cell
    },
    "icbhi_test": {
        "features": cache_dir / "icbhi_test_features.npz",
        "metadata": cache_dir / "icbhi_test_window_metadata.csv",
        "expected_windows": 1667,
        "expected_recordings": 394,
        "allow_pickle": False,
    },
    "fraiwan_test": {
        "features": cache_dir / "fraiwan_test_features.npz",
        "metadata": cache_dir / "fraiwan_test_window_metadata.csv",
        "expected_windows": 114,
        "expected_recordings": 28,
        "allow_pickle": False,
    },
}

validated_metadata = {}

for split_name, spec in cache_specs.items():
    print(f"\n=== {split_name.upper()} ===")

    metadata = pd.read_csv(spec["metadata"])

    with np.load(
        spec["features"],
        allow_pickle=spec["allow_pickle"],
    ) as cached:
        logmel = cached["logmel"]
        mfcc = cached["mfcc"]
        chroma = cached["chroma"]
        labels = cached["labels"]
        paths = cached["paths"].astype(str)

    n = spec["expected_windows"]

    assert logmel.shape == (n, 1, 64, 157)
    assert mfcc.shape == (n, 1, 13, 157)
    assert chroma.shape == (n, 1, 12, 157)
    assert labels.shape == (n,)
    assert paths.shape == (n,)
    assert len(metadata) == n
    assert metadata["recording_path"].nunique() == spec["expected_recordings"]

    assert np.isfinite(logmel).all()
    assert np.isfinite(mfcc).all()
    assert np.isfinite(chroma).all()
    assert np.isin(labels, np.arange(8)).all()

    assert np.array_equal(
        paths,
        metadata["recording_path"].astype(str).to_numpy(),
    )
    assert np.array_equal(
        labels,
        metadata["label_index"].to_numpy(dtype=np.int64),
    )

    print("Windows:", n)
    print("Unique recordings:", metadata["recording_path"].nunique())
    print("Log-Mel:", logmel.shape)
    print("MFCC:", mfcc.shape)
    print("Chroma:", chroma.shape)
    print("All features finite: True")
    print("Metadata alignment: PASSED")

    validated_metadata[split_name] = metadata

recording_sets = {
    name: set(metadata["recording_path"].astype(str))
    for name, metadata in validated_metadata.items()
}

for first, second in [
    ("train", "icbhi_test"),
    ("train", "fraiwan_test"),
    ("icbhi_test", "fraiwan_test"),
]:
    overlap = recording_sets[first] & recording_sets[second]
    print(f"{first} vs {second}: {len(overlap)} shared recordings")
    assert len(overlap) == 0

with open(
    cache_dir / "experiment_b_normalization_stats.json",
    "r",
    encoding="utf-8",
) as f:
    stats = json.load(f)

for feature_name in ["logmel", "mfcc", "chroma"]:
    assert np.isfinite(stats[feature_name]["mean"])
    assert np.isfinite(stats[feature_name]["std"])
    assert stats[feature_name]["std"] > 0

print("\nALL THREE EXPERIMENT B CACHES VALIDATED SUCCESSFULLY.")



=== TRAIN ===
Windows: 2760
Unique recordings: 632
Log-Mel: (2760, 1, 64, 157)
MFCC: (2760, 1, 13, 157)
Chroma: (2760, 1, 12, 157)
All features finite: True
Metadata alignment: PASSED

=== ICBHI_TEST ===
Windows: 1667
Unique recordings: 394
Log-Mel: (1667, 1, 64, 157)
MFCC: (1667, 1, 13, 157)
Chroma: (1667, 1, 12, 157)
All features finite: True
Metadata alignment: PASSED

=== FRAIWAN_TEST ===
Windows: 114
Unique recordings: 28
Log-Mel: (114, 1, 64, 157)
MFCC: (114, 1, 13, 157)
Chroma: (114, 1, 12, 157)
All features finite: True
Metadata alignment: PASSED
train vs icbhi_test: 0 shared recordings
train vs fraiwan_test: 0 shared recordings
icbhi_test vs fraiwan_test: 0 shared recordings

ALL THREE EXPERIMENT B CACHES VALIDATED SUCCESSFULLY.


In [75]:

import json

reference_notebook = (
    project_root
    / "Notebooks 2"
    / "05_Lightweight_MultiFeature_CNN.ipynb"
)

with open(reference_notebook, "r", encoding="utf-8") as f:
    reference_data = json.load(f)

search_terms = [
    "class LightweightMultiFeatureCNN",
    "class_weights",
    "CrossEntropyLoss",
    "ReduceLROnPlateau",
    "patience",
    "optimizer",
    "DataLoader",
    "best_model",
    "EarlyStopping",
    "epochs",
]

for cell_index, cell in enumerate(reference_data["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if any(term.lower() in source.lower() for term in search_terms):
        print(f"\n{'=' * 22} CELL {cell_index} {'=' * 22}")
        print(source)

print("\nReference notebook inspection complete; no cells were executed.")



====================== CELL 5 ======================

import torch
import torch.nn as nn

class DepthwiseSeparableBlock(nn.Module):
    """Depthwise spatial convolution followed by pointwise channel mixing."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False,
            ),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class CompactFeatureBranch(nn.Module)

In [76]:

import pandas as pd

train_recordings = (
    window_manifest[window_manifest["split"] == "train"]
    .drop_duplicates(
        subset=["source_dataset", "source_recording_id"]
    )
    .copy()
)

print("Training recordings:", len(train_recordings))
print("Training patients:", train_recordings["patient_id"].nunique())

print("\n=== RECORDINGS BY SOURCE AND DIAGNOSIS ===")
display(
    train_recordings.groupby(["source_dataset", "diagnosis"])
    .agg(
        recordings=("source_recording_id", "size"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

print("\n=== RECORDINGS PER PATIENT ===")
display(
    train_recordings.groupby(["source_dataset", "patient_id"])
    .size()
    .describe()
    .to_frame("recordings_per_patient")
)

print("\n=== PATIENT SUPPORT PER CLASS ===")
display(
    train_recordings.groupby("diagnosis")["patient_id"]
    .nunique()
    .reindex(CLASS_NAMES, fill_value=0)
    .rename("training_patients")
    .reset_index()
    .rename(columns={"index": "diagnosis"})
)

assert len(train_recordings) == 632
print("\nReady to plan a patient-grouped validation split.")


Training recordings: 632
Training patients: 113

=== RECORDINGS BY SOURCE AND DIAGNOSIS ===


,source_dataset,diagnosis,recordings,patients
0,Fraiwan,Asthma,74,25
1,Fraiwan,COPD,21,7
2,Fraiwan,Pneumonia,12,4
3,ICBHI,Asthma,1,1
4,ICBHI,Bronchiectasis,14,6
5,ICBHI,Bronchiolitis,7,3
6,ICBHI,COPD,431,37
7,ICBHI,Healthy,18,12
8,ICBHI,LRTI,2,2
9,ICBHI,Pneumonia,36,6



=== RECORDINGS PER PATIENT ===


,recordings_per_patient
count,113.000000
mean,5.592920
std,8.656786
min,1.000000
25%,2.000000
50%,3.000000
75%,5.000000
max,66.000000



=== PATIENT SUPPORT PER CLASS ===


,diagnosis,training_patients
0,Asthma,26
1,Bronchiectasis,6
2,Bronchiolitis,3
3,COPD,44
4,Healthy,12
5,LRTI,2
6,Pneumonia,10
7,URTI,10



Ready to plan a patient-grouped validation split.


In [77]:
import pandas as pd

# Check whether patients have multiple diagnoses within a source dataset
patient_label_check = (
    train_recordings
    .groupby(["source_dataset", "patient_id"])
    .agg(
        diagnosis_count=("diagnosis", "nunique"),
        diagnoses=("diagnosis", lambda x: ", ".join(sorted(set(x)))),
        recordings=("source_recording_id", "size"),
    )
    .reset_index()
)

print("=== PATIENT DIAGNOSIS CONSISTENCY ===")
print(
    "Patients with multiple diagnoses:",
    int((patient_label_check["diagnosis_count"] > 1).sum())
)
display(
    patient_label_check[
        patient_label_check["diagnosis_count"] > 1
    ]
)

# Inspect patient-level support for each class
print("\n=== PATIENT SUPPORT BY SOURCE AND DIAGNOSIS ===")
display(
    patient_label_check
    .groupby(["source_dataset", "diagnoses"])
    .agg(
        patients=("patient_id", "nunique"),
        recordings=("recordings", "sum"),
    )
    .reset_index()
    .rename(columns={"diagnoses": "diagnosis"})
)

# Inspect window counts and recording counts per class
train_windows = window_manifest[
    window_manifest["split"] == "train"
].copy()

print("\n=== TRAINING WINDOWS BY SOURCE AND DIAGNOSIS ===")
display(
    train_windows
    .groupby(["source_dataset", "diagnosis"])
    .agg(
        windows=("source_recording_id", "size"),
        recordings=("source_recording_id", "nunique"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

print("\nInspection complete. No model training has started.")

=== PATIENT DIAGNOSIS CONSISTENCY ===
Patients with multiple diagnoses: 0


,source_dataset,patient_id,diagnosis_count,diagnoses,recordings



=== PATIENT SUPPORT BY SOURCE AND DIAGNOSIS ===


,source_dataset,diagnosis,patients,recordings
0,Fraiwan,Asthma,25,74
1,Fraiwan,COPD,7,21
2,Fraiwan,Pneumonia,4,12
3,ICBHI,Asthma,1,1
4,ICBHI,Bronchiectasis,6,14
5,ICBHI,Bronchiolitis,3,7
6,ICBHI,COPD,37,431
7,ICBHI,Healthy,12,18
8,ICBHI,LRTI,2,2
9,ICBHI,Pneumonia,6,36



=== TRAINING WINDOWS BY SOURCE AND DIAGNOSIS ===


,source_dataset,diagnosis,windows,recordings,patients
0,Fraiwan,Asthma,282,74,25
1,Fraiwan,COPD,87,21,7
2,Fraiwan,Pneumonia,48,12,4
3,ICBHI,Asthma,4,1,1
4,ICBHI,Bronchiectasis,70,14,6
5,ICBHI,Bronchiolitis,28,7,3
6,ICBHI,COPD,1953,431,37
7,ICBHI,Healthy,72,18,12
8,ICBHI,LRTI,8,2,2
9,ICBHI,Pneumonia,144,36,6



Inspection complete. No model training has started.


In [78]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

# One row per recording; all windows inherit their recording's assignment.
recordings = (
    window_manifest[window_manifest["split"] == "train"]
    .drop_duplicates(
        subset=["source_dataset", "source_recording_id"]
    )
    .copy()
    .reset_index(drop=True)
)

# Namespaced patient IDs should be unique across source datasets.
assert recordings["patient_id"].is_unique is False or True
assert recordings.groupby(
    ["source_dataset", "source_recording_id"]
).size().max() == 1

# Confirm a patient has one diagnosis within its source.
patient_label_counts = recordings.groupby(
    ["source_dataset", "patient_id"]
)["diagnosis"].nunique()

assert (patient_label_counts == 1).all(), (
    "At least one patient has multiple diagnoses; inspect before splitting."
)

# Group by patient and stratify on diagnosis.
# With very rare classes, this is a candidate split, not a guarantee that
# every class will appear in both subsets.
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

candidate_splits = list(
    splitter.split(
        X=recordings,
        y=recordings["diagnosis"],
        groups=recordings["patient_id"],
    )
)

# Select the first fold as validation (~20% of recordings).
inner_train_idx, validation_idx = candidate_splits[0]

inner_train_recordings = recordings.iloc[inner_train_idx].copy()
validation_recordings = recordings.iloc[validation_idx].copy()

train_patients = set(inner_train_recordings["patient_id"])
validation_patients = set(validation_recordings["patient_id"])

assert train_patients.isdisjoint(validation_patients)
assert len(inner_train_recordings) + len(validation_recordings) == len(recordings)

print("=== CANDIDATE SPLIT ===")
print("Inner-training recordings:", len(inner_train_recordings))
print("Validation recordings:", len(validation_recordings))
print("Inner-training patients:", inner_train_recordings["patient_id"].nunique())
print("Validation patients:", validation_recordings["patient_id"].nunique())
print("Patient overlap:", len(train_patients & validation_patients))

print("\n=== CLASS SUPPORT BY RECORDING ===")
support = pd.concat(
    [
        inner_train_recordings["diagnosis"].value_counts().rename("inner_train"),
        validation_recordings["diagnosis"].value_counts().rename("validation"),
    ],
    axis=1,
).reindex(CLASS_NAMES, fill_value=0).fillna(0).astype(int)

display(support.reset_index().rename(columns={"index": "diagnosis"}))

print("\n=== VALIDATION SUPPORT BY SOURCE ===")
display(
    validation_recordings.groupby(["source_dataset", "diagnosis"])
    .agg(
        recordings=("source_recording_id", "size"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

print("\nCandidate split inspected; no model training has started.")

=== CANDIDATE SPLIT ===
Inner-training recordings: 507
Validation recordings: 125
Inner-training patients: 93
Validation patients: 20
Patient overlap: 0

=== CLASS SUPPORT BY RECORDING ===


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


,diagnosis,inner_train,validation
0,Asthma,60,15
1,Bronchiectasis,12,2
2,Bronchiolitis,7,0
3,COPD,361,91
4,Healthy,14,4
5,LRTI,2,0
6,Pneumonia,39,9
7,URTI,12,4



=== VALIDATION SUPPORT BY SOURCE ===


,source_dataset,diagnosis,recordings,patients
0,Fraiwan,Asthma,14,5
1,Fraiwan,COPD,3,1
2,Fraiwan,Pneumonia,9,3
3,ICBHI,Asthma,1,1
4,ICBHI,Bronchiectasis,2,2
5,ICBHI,COPD,88,6
6,ICBHI,Healthy,4,1
7,ICBHI,URTI,4,1



Candidate split inspected; no model training has started.


In [79]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
recordings = (
    window_manifest[window_manifest["split"] == "train"]
    .drop_duplicates(
        subset=["source_dataset", "source_recording_id"]
    )
    .copy()
    .reset_index(drop=True)
)

# Search deterministic grouped folds, varying the seed.
# Prefer splits with support for every feasible class, then those closest
# to 20% validation recordings.
results = []

for seed in range(200):
    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    for fold, (train_idx, val_idx) in enumerate(
        splitter.split(
            X=recordings,
            y=recordings["diagnosis"],
            groups=recordings["patient_id"],
        )
    ):
        train_df = recordings.iloc[train_idx]
        val_df = recordings.iloc[val_idx]

        val_classes = set(val_df["diagnosis"])
        missing = [c for c in CLASS_NAMES if c not in val_classes]

        # Require no patient overlap.
        if set(train_df["patient_id"]) & set(val_df["patient_id"]):
            continue

        results.append({
            "seed": seed,
            "fold": fold,
            "train_idx": train_idx,
            "val_idx": val_idx,
            "n_val": len(val_df),
            "val_fraction": len(val_df) / len(recordings),
            "missing_classes": missing,
            "n_missing": len(missing),
            "n_val_patients": val_df["patient_id"].nunique(),
        })

best = sorted(
    results,
    key=lambda r: (
        r["n_missing"],
        abs(r["val_fraction"] - 0.20),
        abs(r["n_val"] - 126),
        r["seed"],
        r["fold"],
    ),
)[0]

inner_train_recordings = recordings.iloc[best["train_idx"]].copy()
validation_recordings = recordings.iloc[best["val_idx"]].copy()

print("=== BEST CANDIDATE FOUND ===")
print("Seed:", best["seed"], "| Fold:", best["fold"])
print("Inner-training recordings:", len(inner_train_recordings))
print("Validation recordings:", len(validation_recordings))
print("Validation fraction:", round(best["val_fraction"], 3))
print("Inner-training patients:", inner_train_recordings["patient_id"].nunique())
print("Validation patients:", validation_recordings["patient_id"].nunique())
print("Patient overlap:", len(
    set(inner_train_recordings["patient_id"])
    & set(validation_recordings["patient_id"])
))
print("Missing validation classes:", best["missing_classes"])

support = pd.concat(
    [
        inner_train_recordings["diagnosis"].value_counts().rename("inner_train"),
        validation_recordings["diagnosis"].value_counts().rename("validation"),
    ],
    axis=1,
).reindex(CLASS_NAMES, fill_value=0).fillna(0).astype(int)

print("\n=== CLASS SUPPORT ===")
display(support.reset_index().rename(columns={"index": "diagnosis"}))

print("\n=== VALIDATION SUPPORT BY SOURCE ===")
display(
    validation_recordings.groupby(["source_dataset", "diagnosis"])
    .agg(
        recordings=("source_recording_id", "size"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/Users/vs/Sem 5/Respiratory-Disease-Classificati

=== BEST CANDIDATE FOUND ===
Seed: 1 | Fold: 0
Inner-training recordings: 506
Validation recordings: 126
Validation fraction: 0.199
Inner-training patients: 92
Validation patients: 21
Patient overlap: 0
Missing validation classes: []

=== CLASS SUPPORT ===


,diagnosis,inner_train,validation
0,Asthma,60,15
1,Bronchiectasis,12,2
2,Bronchiolitis,5,2
3,COPD,362,90
4,Healthy,14,4
5,LRTI,1,1
6,Pneumonia,39,9
7,URTI,13,3



=== VALIDATION SUPPORT BY SOURCE ===


,source_dataset,diagnosis,recordings,patients
0,Fraiwan,Asthma,15,5
1,Fraiwan,COPD,3,1
2,ICBHI,Bronchiectasis,2,1
3,ICBHI,Bronchiolitis,2,1
4,ICBHI,COPD,87,6
5,ICBHI,Healthy,4,3
6,ICBHI,LRTI,1,1
7,ICBHI,Pneumonia,9,1
8,ICBHI,URTI,3,2


In [80]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

CACHE_DIR = (
    Path.cwd()
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
)

stats_path = CACHE_DIR / "experiment_b_normalization_stats.json"

print("Current working directory:", Path.cwd())
print("Cache directory exists:", CACHE_DIR.exists())
print("Normalization stats exist:", stats_path.exists())

assert CACHE_DIR.exists(), f"Cache directory not found: {CACHE_DIR}"
assert stats_path.exists(), f"Stats file not found: {stats_path}"

with open(stats_path, "r") as f:
    old_stats = json.load(f)

print("\n=== EXISTING NORMALIZATION STATISTICS ===")
print(json.dumps(old_stats, indent=2))

print("\n=== CACHE FILES ===")
for name in [
    "train_features.npz",
    "train_window_metadata.csv",
    "icbhi_test_features.npz",
    "icbhi_test_window_metadata.csv",
    "fraiwan_test_features.npz",
    "fraiwan_test_window_metadata.csv",
]:
    path = CACHE_DIR / name
    print(f"{name}: exists={path.exists()}", 
          f"size={path.stat().st_size / (1024**2):.2f} MB" if path.exists() else "")

assert all(
    (CACHE_DIR / name).exists()
    for name in [
        "train_features.npz",
        "train_window_metadata.csv",
        "icbhi_test_features.npz",
        "icbhi_test_window_metadata.csv",
        "fraiwan_test_features.npz",
        "fraiwan_test_window_metadata.csv",
    ]
)

print("\nCache and statistics checks passed.")

Current working directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2
Cache directory exists: False
Normalization stats exist: False


AssertionError: Cache directory not found: /Users/vs/Sem 5/Respiratory-Disease-Classification/Notebooks 2/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache

In [81]:
from pathlib import Path
import json

# Go from "Notebooks 2" to the project root
PROJECT_ROOT = Path.cwd().parent

CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
)

stats_path = CACHE_DIR / "experiment_b_normalization_stats.json"

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Cache directory:", CACHE_DIR)
print("Cache directory exists:", CACHE_DIR.exists())
print("Normalization stats exist:", stats_path.exists())

if CACHE_DIR.exists():
    print("\n=== CACHE FILES ===")
    for path in sorted(CACHE_DIR.iterdir()):
        if path.is_file():
            print(f"{path.name}: {path.stat().st_size / (1024**2):.2f} MB")

if stats_path.exists():
    with open(stats_path, "r") as f:
        old_stats = json.load(f)

    print("\n=== EXISTING NORMALIZATION STATISTICS ===")
    print(json.dumps(old_stats, indent=2))
else:
    print("\nStatistics file not found. No files were modified.")

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Project root exists: True
Cache directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache
Cache directory exists: True
Normalization stats exist: True

=== CACHE FILES ===
experiment_b_normalization_stats.json: 0.00 MB
experiment_b_window_manifest.csv: 0.78 MB
fraiwan_test_features.npz: 3.12 MB
fraiwan_test_window_metadata.csv: 0.02 MB
icbhi_test_features.npz: 74.12 MB
icbhi_test_window_metadata.csv: 0.29 MB
train_features.npz: 118.16 MB
train_window_metadata.csv: 0.48 MB

=== EXISTING NORMALIZATION STATISTICS ===
{
  "logmel": {
    "count": 27732480,
    "mean": -58.34986932952237,
    "std": 17.919855328794245
  },
  "mfcc": {
    "count": 5633160,
    "mean": -9.56352060630519,
    "std": 137.70974665089176
  },
  "chroma": {
    "count": 5199840,
    "mean": 0.6930308145310033,
    "std": 0.24621550905495546
  }
}


In [82]:
import numpy as np
import pandas as pd
from pathlib import Path

# Use the corrected project-relative path
PROJECT_ROOT = Path.cwd().parent
CACHE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
)

train_npz = np.load(CACHE_DIR / "train_features.npz", allow_pickle=True)
train_meta = pd.read_csv(CACHE_DIR / "train_window_metadata.csv")

icbhi_npz = np.load(CACHE_DIR / "icbhi_test_features.npz", allow_pickle=False)
icbhi_meta = pd.read_csv(CACHE_DIR / "icbhi_test_window_metadata.csv")

fraiwan_npz = np.load(CACHE_DIR / "fraiwan_test_features.npz", allow_pickle=False)
fraiwan_meta = pd.read_csv(CACHE_DIR / "fraiwan_test_window_metadata.csv")

print("=== ARRAY AND METADATA ALIGNMENT ===")
for name, data, meta in [
    ("train", train_npz, train_meta),
    ("ICBHI test", icbhi_npz, icbhi_meta),
    ("Fraiwan test", fraiwan_npz, fraiwan_meta),
]:
    print(f"\n{name}:")
    for feature in ["logmel", "mfcc", "chroma", "labels"]:
        print(f"  {feature}: {data[feature].shape}")
    print("  metadata rows:", len(meta))

    assert len(meta) == len(data["labels"])
    assert all(
        len(data[feature]) == len(meta)
        for feature in ["logmel", "mfcc", "chroma"]
    )

print("\n=== RECORDING ID COLUMNS ===")
print("Metadata columns:", train_meta.columns.tolist())

# Confirm that the candidate recording split is compatible with metadata.
assert "source_recording_id" in train_meta.columns
assert "patient_id" in train_meta.columns
assert "diagnosis" in train_meta.columns

inner_train_ids = set(
    zip(
        inner_train_recordings["source_dataset"],
        inner_train_recordings["source_recording_id"].astype(str),
    )
)
validation_ids = set(
    zip(
        validation_recordings["source_dataset"],
        validation_recordings["source_recording_id"].astype(str),
    )
)
cached_ids = set(
    zip(
        train_meta["source_dataset"],
        train_meta["source_recording_id"].astype(str),
    )
)

print("\nInner-training recordings:", len(inner_train_ids))
print("Validation recordings:", len(validation_ids))
print("Cached training recordings:", len(cached_ids))
print("Inner-training IDs missing from cache:", len(inner_train_ids - cached_ids))
print("Validation IDs missing from cache:", len(validation_ids - cached_ids))

assert inner_train_ids.isdisjoint(validation_ids)
assert inner_train_ids | validation_ids == cached_ids

print("\nCache alignment and split membership verified.")

=== ARRAY AND METADATA ALIGNMENT ===

train:
  logmel: (2760, 1, 64, 157)
  mfcc: (2760, 1, 13, 157)
  chroma: (2760, 1, 12, 157)
  labels: (2760,)
  metadata rows: 2760

ICBHI test:
  logmel: (1667, 1, 64, 157)
  mfcc: (1667, 1, 13, 157)
  chroma: (1667, 1, 12, 157)
  labels: (1667,)
  metadata rows: 1667

Fraiwan test:
  logmel: (114, 1, 64, 157)
  mfcc: (114, 1, 13, 157)
  chroma: (114, 1, 12, 157)
  labels: (114,)
  metadata rows: 114

=== RECORDING ID COLUMNS ===
Metadata columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'label_index', 'split', 'recording_path', 'window_index', 'start_seconds', 'end_seconds', 'valid_duration_seconds', 'requires_zero_padding', 'original_sample_rate', 'resampled_sample_rate']

Inner-training recordings: 506
Validation recordings: 126
Cached training recordings: 632
Inner-training IDs missing from cache: 0
Validation IDs missing from cache: 0

Cache alignment and split membership verified.


In [83]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Paths and reproducibility
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CACHE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
)

OUTPUT_DIR = CACHE_DIR / "experiment_b_v1_strict"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["logmel", "mfcc", "chroma"]

with open(CACHE_DIR / "experiment_b_normalization_stats.json", "r") as f:
    old_stats = json.load(f)

# Load existing feature caches and metadata
train_npz = np.load(CACHE_DIR / "train_features.npz", allow_pickle=True)
train_meta = pd.read_csv(CACHE_DIR / "train_window_metadata.csv")

icbhi_npz = np.load(CACHE_DIR / "icbhi_test_features.npz", allow_pickle=False)
icbhi_meta = pd.read_csv(CACHE_DIR / "icbhi_test_window_metadata.csv")

fraiwan_npz = np.load(CACHE_DIR / "fraiwan_test_features.npz", allow_pickle=False)
fraiwan_meta = pd.read_csv(CACHE_DIR / "fraiwan_test_window_metadata.csv")

# ---------------------------------------------------------
# 2. Identify windows by recording-level split
# ---------------------------------------------------------
def recording_keys(df):
    return list(zip(
        df["source_dataset"].astype(str),
        df["source_recording_id"].astype(str),
    ))

inner_train_ids = set(recording_keys(inner_train_recordings))
validation_ids = set(recording_keys(validation_recordings))

train_keys = recording_keys(train_meta)
is_inner_train = np.array([key in inner_train_ids for key in train_keys])
is_validation = np.array([key in validation_ids for key in train_keys])

assert is_inner_train.sum() + is_validation.sum() == len(train_meta)
assert not np.any(is_inner_train & is_validation)
assert is_inner_train.any() and is_validation.any()

print("Inner-training windows:", int(is_inner_train.sum()))
print("Validation windows:", int(is_validation.sum()))

# ---------------------------------------------------------
# 3. Reverse the previous normalization
#    This reconstructs approximate raw features.
# ---------------------------------------------------------
raw_arrays = {}

for feature in FEATURES:
    old_mean = float(old_stats[feature]["mean"])
    old_std = float(old_stats[feature]["std"])

    raw_arrays[feature] = (
        train_npz[feature].astype(np.float32) * old_std + old_mean
    )

# Reconstruct raw features for both untouched test sets too.
icbhi_raw = {
    feature: (
        icbhi_npz[feature].astype(np.float32)
        * float(old_stats[feature]["std"])
        + float(old_stats[feature]["mean"])
    )
    for feature in FEATURES
}

fraiwan_raw = {
    feature: (
        fraiwan_npz[feature].astype(np.float32)
        * float(old_stats[feature]["std"])
        + float(old_stats[feature]["mean"])
    )
    for feature in FEATURES
}

# ---------------------------------------------------------
# 4. Compute fresh statistics using inner-training windows only
# ---------------------------------------------------------
new_stats = {}

for feature in FEATURES:
    values = raw_arrays[feature][is_inner_train].astype(np.float64)

    mean = float(values.mean())
    std = float(values.std())

    assert np.isfinite(mean) and np.isfinite(std) and std > 0

    new_stats[feature] = {
        "count": int(values.size),
        "mean": mean,
        "std": std,
    }

print("\n=== NEW INNER-TRAINING NORMALIZATION ===")
for feature, stats in new_stats.items():
    print(
        f"{feature}: mean={stats['mean']:.6f}, "
        f"std={stats['std']:.6f}, count={stats['count']:,}"
    )

# ---------------------------------------------------------
# 5. Normalize all subsets using the new training-only stats
# ---------------------------------------------------------
def normalize_features(arrays):
    return {
        feature: (
            (arrays[feature] - new_stats[feature]["mean"])
            / new_stats[feature]["std"]
        ).astype(np.float32)
        for feature in FEATURES
    }

normalized_train = normalize_features(raw_arrays)
normalized_icbhi = normalize_features(icbhi_raw)
normalized_fraiwan = normalize_features(fraiwan_raw)

# ---------------------------------------------------------
# 6. Save separate train/validation/test caches
# ---------------------------------------------------------
def save_feature_cache(path, arrays, labels, paths):
    np.savez_compressed(
        path,
        logmel=arrays["logmel"],
        mfcc=arrays["mfcc"],
        chroma=arrays["chroma"],
        labels=np.asarray(labels, dtype=np.int64),
        paths=np.asarray(paths, dtype=str),
    )

train_paths = train_meta["recording_path"].astype(str).to_numpy()
train_labels = train_npz["labels"].astype(np.int64)

save_feature_cache(
    OUTPUT_DIR / "train_features.npz",
    {k: v[is_inner_train] for k, v in normalized_train.items()},
    train_labels[is_inner_train],
    train_paths[is_inner_train],
)

save_feature_cache(
    OUTPUT_DIR / "validation_features.npz",
    {k: v[is_validation] for k, v in normalized_train.items()},
    train_labels[is_validation],
    train_paths[is_validation],
)

save_feature_cache(
    OUTPUT_DIR / "icbhi_test_features.npz",
    normalized_icbhi,
    icbhi_npz["labels"],
    icbhi_meta["recording_path"].astype(str).to_numpy(),
)

save_feature_cache(
    OUTPUT_DIR / "fraiwan_test_features.npz",
    normalized_fraiwan,
    fraiwan_npz["labels"],
    fraiwan_meta["recording_path"].astype(str).to_numpy(),
)

# Save metadata aligned with the corresponding feature rows.
train_meta.loc[is_inner_train].reset_index(drop=True).to_csv(
    OUTPUT_DIR / "train_window_metadata.csv", index=False
)
train_meta.loc[is_validation].reset_index(drop=True).to_csv(
    OUTPUT_DIR / "validation_window_metadata.csv", index=False
)
icbhi_meta.to_csv(OUTPUT_DIR / "icbhi_test_window_metadata.csv", index=False)
fraiwan_meta.to_csv(OUTPUT_DIR / "fraiwan_test_window_metadata.csv", index=False)

with open(OUTPUT_DIR / "normalization_stats.json", "w") as f:
    json.dump(new_stats, f, indent=2)

# Record the exact recording split for reproducibility.
inner_train_recordings.to_csv(
    OUTPUT_DIR / "inner_train_recordings.csv", index=False
)
validation_recordings.to_csv(
    OUTPUT_DIR / "validation_recordings.csv", index=False
)

print("\nNew cache files saved to:", OUTPUT_DIR)
print("Original cache files were not overwritten.")

Inner-training windows: 2224
Validation windows: 536

=== NEW INNER-TRAINING NORMALIZATION ===
logmel: mean=-58.251983, std=18.100168, count=22,346,752
mfcc: mean=-9.335047, std=136.717893, count=4,539,184
chroma: mean=0.695039, std=0.244684, count=4,190,016

New cache files saved to: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict
Original cache files were not overwritten.


In [84]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = (
    Path.cwd().parent
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

required_files = [
    "train_features.npz",
    "validation_features.npz",
    "icbhi_test_features.npz",
    "fraiwan_test_features.npz",
    "train_window_metadata.csv",
    "validation_window_metadata.csv",
    "icbhi_test_window_metadata.csv",
    "fraiwan_test_window_metadata.csv",
    "normalization_stats.json",
    "inner_train_recordings.csv",
    "validation_recordings.csv",
]

print("Output directory:", OUTPUT_DIR)
print("Directory exists:", OUTPUT_DIR.exists())

missing = [name for name in required_files if not (OUTPUT_DIR / name).exists()]
print("Missing files:", missing)
assert not missing, "Cache-writing step did not produce all expected files."

# Verify every feature array aligns with its metadata.
for split in ["train", "validation", "icbhi_test", "fraiwan_test"]:
    with np.load(OUTPUT_DIR / f"{split}_features.npz", allow_pickle=False) as data:
        meta = pd.read_csv(OUTPUT_DIR / f"{split}_window_metadata.csv")
        print(f"\n{split}:")
        print("  metadata rows:", len(meta))

        for feature in ["logmel", "mfcc", "chroma"]:
            arr = data[feature]
            print(f"  {feature}: {arr.shape}, finite={np.isfinite(arr).all()}")
            assert len(arr) == len(meta)
            assert np.isfinite(arr).all()

        assert len(data["labels"]) == len(meta)

# Verify patient-level separation.
train_rec = pd.read_csv(OUTPUT_DIR / "inner_train_recordings.csv")
val_rec = pd.read_csv(OUTPUT_DIR / "validation_recordings.csv")

train_patients = set(train_rec["patient_id"])
val_patients = set(val_rec["patient_id"])

print("\nInner-training recordings:", len(train_rec))
print("Validation recordings:", len(val_rec))
print("Patient overlap:", len(train_patients & val_patients))

assert train_patients.isdisjoint(val_patients)

# Verify training features are approximately zero mean and unit standard deviation.
with np.load(OUTPUT_DIR / "train_features.npz", allow_pickle=False) as data:
    print("\n=== INNER-TRAINING NORMALIZED FEATURE CHECK ===")
    for feature in ["logmel", "mfcc", "chroma"]:
        arr = data[feature].astype(np.float64)
        print(f"{feature}: mean={arr.mean():.5f}, std={arr.std():.5f}")
        assert abs(arr.mean()) < 0.02
        assert abs(arr.std() - 1.0) < 0.02

with open(OUTPUT_DIR / "normalization_stats.json", "r") as f:
    stats = json.load(f)

print("\nNormalization stats saved:", list(stats.keys()))
print("\nAll cache and leakage-control checks passed.")

Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict
Directory exists: True
Missing files: []

train:
  metadata rows: 2224
  logmel: (2224, 1, 64, 157), finite=True
  mfcc: (2224, 1, 13, 157), finite=True
  chroma: (2224, 1, 12, 157), finite=True

validation:
  metadata rows: 536
  logmel: (536, 1, 64, 157), finite=True
  mfcc: (536, 1, 13, 157), finite=True
  chroma: (536, 1, 12, 157), finite=True

icbhi_test:
  metadata rows: 1667
  logmel: (1667, 1, 64, 157), finite=True
  mfcc: (1667, 1, 13, 157), finite=True
  chroma: (1667, 1, 12, 157), finite=True

fraiwan_test:
  metadata rows: 114
  logmel: (114, 1, 64, 157), finite=True
  mfcc: (114, 1, 13, 157), finite=True
  chroma: (114, 1, 12, 157), finite=True

Inner-training recordings: 506
Validation recordings: 126
Patient overlap: 0

=== INNER-TRAINING NORMALIZED FEATURE CHECK ===
logmel: mean=0.00000, std

In [85]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Experiment B strict cache
PROJECT_ROOT = Path.cwd().parent
B_CACHE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 60
PATIENCE = 10
NUM_WORKERS = 0

# Reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

train_meta = pd.read_csv(B_CACHE_DIR / "train_window_metadata.csv")
validation_meta = pd.read_csv(B_CACHE_DIR / "validation_window_metadata.csv")

with np.load(B_CACHE_DIR / "train_features.npz", allow_pickle=False) as data:
    train_labels = data["labels"].astype(np.int64)

with np.load(B_CACHE_DIR / "validation_features.npz", allow_pickle=False) as data:
    validation_labels = data["labels"].astype(np.int64)

print("=== EXPERIMENT B CNN BASELINE ===")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Training windows:", len(train_labels))
print("Validation windows:", len(validation_labels))
print("Classes:", len(CLASS_NAMES))
print("Training label range:", train_labels.min(), "to", train_labels.max())
print("Validation label range:", validation_labels.min(), "to", validation_labels.max())

assert len(train_labels) == len(train_meta)
assert len(validation_labels) == len(validation_meta)
assert set(np.unique(train_labels)).issubset(set(range(len(CLASS_NAMES))))
assert set(np.unique(validation_labels)).issubset(set(range(len(CLASS_NAMES))))

print("\n=== TRAINING SUPPORT (WINDOWS) ===")
train_support = (
    pd.Series(train_labels)
    .value_counts()
    .reindex(range(len(CLASS_NAMES)), fill_value=0)
)
display(pd.DataFrame({
    "class_index": range(len(CLASS_NAMES)),
    "diagnosis": CLASS_NAMES,
    "training_windows": train_support.values,
}))

print("\n=== VALIDATION SUPPORT (WINDOWS) ===")
validation_support = (
    pd.Series(validation_labels)
    .value_counts()
    .reindex(range(len(CLASS_NAMES)), fill_value=0)
)
display(pd.DataFrame({
    "class_index": range(len(CLASS_NAMES)),
    "diagnosis": CLASS_NAMES,
    "validation_windows": validation_support.values,
}))

print("\nEnvironment and labels verified.")

=== EXPERIMENT B CNN BASELINE ===
PyTorch: 2.14.1
Device: mps
Training windows: 2224
Validation windows: 536
Classes: 8
Training label range: 0 to 7
Validation label range: 0 to 7

=== TRAINING SUPPORT (WINDOWS) ===


,class_index,diagnosis,training_windows
0,0,Asthma,241
1,1,Bronchiectasis,60
2,2,Bronchiolitis,20
3,3,COPD,1635
4,4,Healthy,56
5,5,LRTI,4
6,6,Pneumonia,156
7,7,URTI,52



=== VALIDATION SUPPORT (WINDOWS) ===


,class_index,diagnosis,validation_windows
0,0,Asthma,45
1,1,Bronchiectasis,10
2,2,Bronchiolitis,8
3,3,COPD,405
4,4,Healthy,16
5,5,LRTI,4
6,6,Pneumonia,36
7,7,URTI,12



Environment and labels verified.


In [86]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# ---------------------------------------------------------
# 1. Reproducibility and device
# ---------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

B_CACHE_DIR = (
    Path.cwd().parent
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

CLASS_NAMES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]

# ---------------------------------------------------------
# 2. Lightweight CNN architecture from the reference
# ---------------------------------------------------------
class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, in_channels, kernel_size=3,
                stride=stride, padding=1, groups=in_channels,
                bias=False
            ),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                in_channels, out_channels, kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class CompactFeatureBranch(nn.Module):
    def __init__(self, output_dim=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            DepthwiseSeparableBlock(8, 16),
            nn.MaxPool2d(2),

            DepthwiseSeparableBlock(16, 24),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(24, output_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
        )

    def forward(self, x):
        return self.projection(self.features(x))


class LightweightMultiFeatureCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.logmel_branch = CompactFeatureBranch(16)
        self.mfcc_branch = CompactFeatureBranch(16)
        self.chroma_branch = CompactFeatureBranch(16)

        self.classifier = nn.Sequential(
            nn.Linear(48, 24),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(24, num_classes),
        )

    def forward(self, logmel, mfcc, chroma):
        features = torch.cat(
            [
                self.logmel_branch(logmel),
                self.mfcc_branch(mfcc),
                self.chroma_branch(chroma),
            ],
            dim=1,
        )
        return self.classifier(features)


# ---------------------------------------------------------
# 3. Dataset with training-only augmentation
# ---------------------------------------------------------
class RespiratoryFeatureDataset(Dataset):
    def __init__(self, npz_path, augment=False):
        self.augment = augment

        with np.load(npz_path, allow_pickle=False) as data:
            self.logmel = data["logmel"].astype(np.float32)
            self.mfcc = data["mfcc"].astype(np.float32)
            self.chroma = data["chroma"].astype(np.float32)
            self.labels = data["labels"].astype(np.int64)

        assert len(self.logmel) == len(self.labels)
        assert len(self.mfcc) == len(self.labels)
        assert len(self.chroma) == len(self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        logmel = self.logmel[idx].copy()
        mfcc = self.mfcc[idx].copy()
        chroma = self.chroma[idx].copy()

        if self.augment:
            # Shared temporal shift across all three features.
            shift = np.random.randint(-8, 9)
            logmel = np.roll(logmel, shift, axis=-1)
            mfcc = np.roll(mfcc, shift, axis=-1)
            chroma = np.roll(chroma, shift, axis=-1)

            # Small feature-space Gaussian noise.
            logmel += np.random.normal(
                0, 0.015, size=logmel.shape
            ).astype(np.float32)
            mfcc += np.random.normal(
                0, 0.010, size=mfcc.shape
            ).astype(np.float32)
            chroma += np.random.normal(
                0, 0.010, size=chroma.shape
            ).astype(np.float32)

        return (
            torch.from_numpy(logmel),
            torch.from_numpy(mfcc),
            torch.from_numpy(chroma),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


# ---------------------------------------------------------
# 4. Instantiate datasets and loaders
# ---------------------------------------------------------
train_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "train_features.npz",
    augment=True,
)

validation_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "validation_features.npz",
    augment=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

model = LightweightMultiFeatureCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

# Shape smoke test
batch = next(iter(train_loader))
with torch.no_grad():
    sample_logits = model(
        batch[0].to(DEVICE),
        batch[1].to(DEVICE),
        batch[2].to(DEVICE),
    )

print("Device:", DEVICE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Trainable parameters:", f"{parameter_count:,}")
print("Sample logits shape:", tuple(sample_logits.shape))
print("Expected shape:", (len(batch[3]), len(CLASS_NAMES)))

assert sample_logits.shape == (len(batch[3]), len(CLASS_NAMES))
print("Model and data-loader smoke test passed. Training has not started.")

Device: mps
Training batches: 35
Validation batches: 9
Trainable parameters: 5,408
Sample logits shape: (64, 8)
Expected shape: (64, 8)
Model and data-loader smoke test passed. Training has not started.


In [87]:
import copy
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import f1_score
from collections import Counter

# ---------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

# ---------------------------------------------------------
# 2. Class weights from inner-training labels only
# ---------------------------------------------------------
class_counts = np.bincount(
    train_dataset.labels,
    minlength=len(CLASS_NAMES),
).astype(np.float64)

assert np.all(class_counts > 0), (
    "A class has no inner-training examples. Inspect the split."
)

# Reference strategy: inverse square root frequency,
# normalized so the frequency-weighted average weight is 1.
class_weights_np = 1.0 / np.sqrt(class_counts)
class_weights_np /= (
    np.sum(class_counts * class_weights_np) / np.sum(class_counts)
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

print("=== TRAINING CLASS WEIGHTS ===")
for i, name in enumerate(CLASS_NAMES):
    print(
        f"{name:18s} count={int(class_counts[i]):4d} "
        f"weight={class_weights_np[i]:.4f}"
    )

# ---------------------------------------------------------
# 3. Optimizer, loss and scheduler
# ---------------------------------------------------------
MAX_EPOCHS = 60
PATIENCE = 10
MIN_DELTA = 1e-4

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

# ---------------------------------------------------------
# 4. Checkpoint and history paths
# ---------------------------------------------------------
OUTPUT_DIR = B_CACHE_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "experiment_b_lightweight_cnn_best.pt"
HISTORY_PATH = OUTPUT_DIR / "experiment_b_lightweight_cnn_history.csv"
CONFIG_PATH = OUTPUT_DIR / "experiment_b_lightweight_cnn_config.json"

training_config = {
    "experiment": "B",
    "model": "LightweightMultiFeatureCNN",
    "class_names": CLASS_NAMES,
    "seed": SEED,
    "device_used": str(DEVICE),
    "trainable_parameters": sum(
        p.numel() for p in model.parameters() if p.requires_grad
    ),
    "batch_size": 64,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_mode": "max",
    "scheduler_factor": 0.5,
    "scheduler_patience": 3,
    "minimum_learning_rate": 1e-6,
    "loss": "weighted CrossEntropyLoss",
    "class_weight_strategy": "inverse square-root frequency; weighted mean normalized to 1",
    "validation_selection_metric": "macro F1 over classes present in validation",
    "validation_aggregation": "recording-level mean of window probabilities",
    "normalization": "inner-training windows only",
    "augmentation": {
        "temporal_shift_frames": [-8, 8],
        "logmel_noise_std": 0.015,
        "mfcc_noise_std": 0.010,
        "chroma_noise_std": 0.010,
    },
}

with open(CONFIG_PATH, "w") as f:
    json.dump(training_config, f, indent=2)

# ---------------------------------------------------------
# 5. Confirm configuration is valid
# ---------------------------------------------------------
assert len(class_weights) == len(CLASS_NAMES)
assert BEST_MODEL_PATH.parent.exists()
assert criterion.weight.device == DEVICE

print("\n=== TRAINING CONFIGURATION READY ===")
print("Optimizer:", optimizer.__class__.__name__)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler:", scheduler.__class__.__name__)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Best-model path:", BEST_MODEL_PATH)
print("History path:", HISTORY_PATH)
print("Configuration saved:", CONFIG_PATH)
print("\nTraining has not started.")

=== TRAINING CLASS WEIGHTS ===
Asthma             count= 241 weight=1.4714
Bronchiectasis     count=  60 weight=2.9490
Bronchiolitis      count=  20 weight=5.1078
COPD               count=1635 weight=0.5649
Healthy            count=  56 weight=3.0525
LRTI               count=   4 weight=11.4213
Pneumonia          count= 156 weight=1.8289
URTI               count=  52 weight=3.1677


AssertionError: 

In [88]:
print("Selected device:", DEVICE)
print("Model parameter device:", next(model.parameters()).device)
print("Class-weight tensor device:", class_weights.device)
print("Loss-weight device:", criterion.weight.device)

# Verify device types instead of comparing device objects directly.
assert next(model.parameters()).device.type == DEVICE.type
assert class_weights.device.type == DEVICE.type
assert criterion.weight.device.type == DEVICE.type
assert len(class_weights) == len(CLASS_NAMES)
assert BEST_MODEL_PATH.parent.exists()

print("\n=== TRAINING CONFIGURATION READY ===")
print("Optimizer:", optimizer.__class__.__name__)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler:", scheduler.__class__.__name__)
print("Maximum epochs:", MAX_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Best-model path:", BEST_MODEL_PATH)
print("History path:", HISTORY_PATH)
print("Configuration saved:", CONFIG_PATH)
print("\nTraining has not started.")

Selected device: mps
Model parameter device: mps:0
Class-weight tensor device: mps:0
Loss-weight device: mps:0

=== TRAINING CONFIGURATION READY ===
Optimizer: AdamW
Initial learning rate: 0.001
Scheduler: ReduceLROnPlateau
Maximum epochs: 60
Early-stopping patience: 10
Best-model path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/checkpoints/experiment_b_lightweight_cnn_best.pt
History path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/checkpoints/experiment_b_lightweight_cnn_history.csv
Configuration saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/checkpoints/experiment_b_lightweight_cnn_config.json

Training has not started.


In [89]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, accuracy_score

# ---------------------------------------------------------
# 1. Load validation recording metadata
# ---------------------------------------------------------
validation_meta = pd.read_csv(
    B_CACHE_DIR / "validation_window_metadata.csv"
).reset_index(drop=True)

assert len(validation_meta) == len(validation_dataset)

# Each window belongs to a recording identified by source + recording ID.
validation_recording_keys = list(zip(
    validation_meta["source_dataset"].astype(str),
    validation_meta["source_recording_id"].astype(str),
))

# Stable integer group index for recording-level aggregation.
unique_recording_keys = list(dict.fromkeys(validation_recording_keys))
recording_to_index = {
    key: idx for idx, key in enumerate(unique_recording_keys)
}
validation_group_indices = np.array(
    [recording_to_index[key] for key in validation_recording_keys],
    dtype=np.int64,
)

validation_recording_labels = np.empty(
    len(unique_recording_keys), dtype=np.int64
)

for key, group_idx in recording_to_index.items():
    rows = np.flatnonzero(
        np.array([item == key for item in validation_recording_keys])
    )
    labels_for_recording = validation_dataset.labels[rows]
    assert np.unique(labels_for_recording).size == 1
    validation_recording_labels[group_idx] = labels_for_recording[0]

print(
    "Validation recordings for model selection:",
    len(unique_recording_keys),
)

# ---------------------------------------------------------
# 2. Validation evaluation at recording level
# ---------------------------------------------------------
@torch.no_grad()
def evaluate_recording_level(model, loader, metadata, device):
    model.eval()

    all_probs = []
    all_labels = []

    for logmel, mfcc, chroma, labels in loader:
        logmel = logmel.to(device)
        mfcc = mfcc.to(device)
        chroma = chroma.to(device)

        logits = model(logmel, mfcc, chroma)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

    window_probs = np.concatenate(all_probs, axis=0)
    window_labels = np.concatenate(all_labels, axis=0)

    assert len(window_probs) == len(metadata)
    assert np.array_equal(window_labels, validation_dataset.labels)

    # Average window probabilities for each recording.
    keys = list(zip(
        metadata["source_dataset"].astype(str),
        metadata["source_recording_id"].astype(str),
    ))
    unique_keys = list(dict.fromkeys(keys))
    key_to_idx = {key: i for i, key in enumerate(unique_keys)}

    recording_probs = np.zeros(
        (len(unique_keys), len(CLASS_NAMES)), dtype=np.float64
    )
    counts = np.zeros(len(unique_keys), dtype=np.int64)
    recording_labels = np.zeros(len(unique_keys), dtype=np.int64)

    for i, key in enumerate(keys):
        group_idx = key_to_idx[key]
        recording_probs[group_idx] += window_probs[i]
        counts[group_idx] += 1

        label = int(window_labels[i])
        if counts[group_idx] == 1:
            recording_labels[group_idx] = label
        else:
            assert recording_labels[group_idx] == label

    recording_probs /= counts[:, None]
    predictions = recording_probs.argmax(axis=1)

    supported_classes = np.unique(recording_labels)
    macro_f1 = f1_score(
        recording_labels,
        predictions,
        labels=supported_classes,
        average="macro",
        zero_division=0,
    )
    accuracy = accuracy_score(recording_labels, predictions)

    return {
        "macro_f1_recording": float(macro_f1),
        "accuracy_recording": float(accuracy),
        "recording_labels": recording_labels,
        "recording_predictions": predictions,
        "recording_probabilities": recording_probs,
        "recording_keys": unique_keys,
    }


# ---------------------------------------------------------
# 3. Training loop with best-checkpoint saving
# ---------------------------------------------------------
history = []
best_val_f1 = -np.inf
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    model.train()

    running_loss = 0.0
    total_train = 0
    train_true = []
    train_pred = []

    for logmel, mfcc, chroma, labels in train_loader:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(logmel, mfcc, chroma)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_train += batch_size

        train_true.extend(labels.detach().cpu().numpy().tolist())
        train_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    train_loss = running_loss / total_train
    train_window_macro_f1 = f1_score(
        train_true,
        train_pred,
        labels=np.arange(len(CLASS_NAMES)),
        average="macro",
        zero_division=0,
    )

    # Validation selection is recording-level, not window-level.
    val_result = evaluate_recording_level(
        model,
        validation_loader,
        validation_meta,
        DEVICE,
    )
    val_f1 = val_result["macro_f1_recording"]
    val_accuracy = val_result["accuracy_recording"]

    scheduler.step(val_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    improved = val_f1 > best_val_f1 + MIN_DELTA

    if improved:
        best_val_f1 = val_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_validation_macro_f1_recording": best_val_f1,
                "class_names": CLASS_NAMES,
                "training_config": training_config,
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_window_macro_f1": train_window_macro_f1,
        "validation_recording_macro_f1": val_f1,
        "validation_recording_accuracy": val_accuracy,
        "learning_rate": current_lr,
        "seconds": time.time() - epoch_start,
        "best_checkpoint_updated": improved,
    }
    history.append(row)

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"train-window macro-F1={train_window_macro_f1:.4f} | "
        f"val-recording macro-F1={val_f1:.4f} | "
        f"val-recording acc={val_accuracy:.4f} | "
        f"lr={current_lr:.2e} | "
        f"{row['seconds']:.1f}s"
        + (" | best checkpoint" if improved else "")
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch}; "
            f"best epoch was {best_epoch}."
        )
        break

# ---------------------------------------------------------
# 4. Restore the best validation checkpoint
# ---------------------------------------------------------
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("\n=== TRAINING COMPLETE ===")
print("Best epoch:", best_epoch)
print("Best validation recording-level macro-F1:", round(best_val_f1, 4))
print("Best model saved:", BEST_MODEL_PATH.exists())
print("History saved:", HISTORY_PATH.exists())
print("Test sets have not been evaluated.")

Validation recordings for model selection: 126
Epoch 01/60 | loss=1.9526 | train-window macro-F1=0.1089 | val-recording macro-F1=0.1042 | val-recording acc=0.7143 | lr=1.00e-03 | 11.4s | best checkpoint
Epoch 02/60 | loss=1.6593 | train-window macro-F1=0.1059 | val-recording macro-F1=0.1042 | val-recording acc=0.7143 | lr=1.00e-03 | 2.9s
Epoch 03/60 | loss=1.3955 | train-window macro-F1=0.1680 | val-recording macro-F1=0.2119 | val-recording acc=0.7857 | lr=1.00e-03 | 2.6s | best checkpoint
Epoch 04/60 | loss=1.2462 | train-window macro-F1=0.2111 | val-recording macro-F1=0.1967 | val-recording acc=0.7540 | lr=1.00e-03 | 3.5s
Epoch 05/60 | loss=1.1413 | train-window macro-F1=0.2590 | val-recording macro-F1=0.2360 | val-recording acc=0.7143 | lr=1.00e-03 | 3.1s | best checkpoint
Epoch 06/60 | loss=1.0458 | train-window macro-F1=0.2946 | val-recording macro-F1=0.2378 | val-recording acc=0.7222 | lr=1.00e-03 | 2.6s | best checkpoint
Epoch 07/60 | loss=0.9934 | train-window macro-F1=0.3380 |

In [90]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

B_CACHE_DIR = (
    Path.cwd().parent
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

OUTPUT_DIR = B_CACHE_DIR / "checkpoints"
BEST_MODEL_PATH = OUTPUT_DIR / "experiment_b_lightweight_cnn_best.pt"

CLASS_NAMES = [
    "Asthma", "Bronchiectasis", "Bronchiolitis", "COPD",
    "Healthy", "LRTI", "Pneumonia", "URTI"
]

assert BEST_MODEL_PATH.exists(), "Best checkpoint not found."

# Rebuild the architecture if the notebook state was restarted.
model = LightweightMultiFeatureCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

icbhi_test_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "icbhi_test_features.npz",
    augment=False,
)
fraiwan_test_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "fraiwan_test_features.npz",
    augment=False,
)

icbhi_test_loader = DataLoader(
    icbhi_test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)
fraiwan_test_loader = DataLoader(
    fraiwan_test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

icbhi_test_meta = pd.read_csv(
    B_CACHE_DIR / "icbhi_test_window_metadata.csv"
).reset_index(drop=True)

fraiwan_test_meta = pd.read_csv(
    B_CACHE_DIR / "fraiwan_test_window_metadata.csv"
).reset_index(drop=True)

assert len(icbhi_test_meta) == len(icbhi_test_dataset)
assert len(fraiwan_test_meta) == len(fraiwan_test_dataset)

print("Best checkpoint epoch:", checkpoint["epoch"])
print(
    "Saved validation recording macro-F1:",
    round(checkpoint["best_validation_macro_f1_recording"], 4),
)
print("\nICBHI test recordings:",
      icbhi_test_meta["source_recording_id"].nunique())
print("ICBHI test windows:", len(icbhi_test_dataset))
print("Fraiwan external-test recordings:",
      fraiwan_test_meta["source_recording_id"].nunique())
print("Fraiwan external-test windows:", len(fraiwan_test_dataset))
print("\nTest datasets are loaded; evaluation has not run yet.")

Best checkpoint epoch: 15
Saved validation recording macro-F1: 0.4026

ICBHI test recordings: 394
ICBHI test windows: 1667
Fraiwan external-test recordings: 28
Fraiwan external-test windows: 114

Test datasets are loaded; evaluation has not run yet.


In [91]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ---------------------------------------------------------
# 1. Output directory
# ---------------------------------------------------------
OUTPUT_DIR = B_CACHE_DIR / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Recording-level evaluation
# ---------------------------------------------------------
@torch.no_grad()
def evaluate_test_set(model, loader, dataset, metadata, dataset_name):
    model.eval()

    probs_list = []
    labels_list = []

    for logmel, mfcc, chroma, labels in loader:
        logits = model(
            logmel.to(DEVICE),
            mfcc.to(DEVICE),
            chroma.to(DEVICE),
        )
        probs_list.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels_list.append(labels.numpy())

    window_probs = np.concatenate(probs_list)
    window_labels = np.concatenate(labels_list)

    assert len(window_probs) == len(metadata)
    assert np.array_equal(window_labels, dataset.labels)

    keys = list(zip(
        metadata["source_dataset"].astype(str),
        metadata["source_recording_id"].astype(str),
    ))

    unique_keys = list(dict.fromkeys(keys))
    key_to_idx = {key: i for i, key in enumerate(unique_keys)}

    n_classes = len(CLASS_NAMES)
    recording_probs = np.zeros((len(unique_keys), n_classes), dtype=np.float64)
    recording_labels = np.full(len(unique_keys), -1, dtype=np.int64)
    window_counts = np.zeros(len(unique_keys), dtype=np.int64)

    for i, key in enumerate(keys):
        j = key_to_idx[key]
        recording_probs[j] += window_probs[i]
        window_counts[j] += 1

        label = int(window_labels[i])
        if recording_labels[j] == -1:
            recording_labels[j] = label
        else:
            assert recording_labels[j] == label, (
                f"Inconsistent labels for recording {key}"
            )

    recording_probs /= window_counts[:, None]
    predictions = recording_probs.argmax(axis=1)

    supported = np.unique(recording_labels)

    metrics = {
        "dataset": dataset_name,
        "recordings": int(len(unique_keys)),
        "windows": int(len(window_labels)),
        "accuracy": float(accuracy_score(recording_labels, predictions)),
        "balanced_accuracy_supported_classes": float(
            balanced_accuracy_score(recording_labels, predictions)
        ),
        "macro_f1_supported_classes": float(
            f1_score(
                recording_labels,
                predictions,
                labels=supported,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_8_classes": float(
            f1_score(
                recording_labels,
                predictions,
                labels=np.arange(n_classes),
                average="macro",
                zero_division=0,
            )
        ),
    }

    report = classification_report(
        recording_labels,
        predictions,
        labels=np.arange(n_classes),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).T

    cm = confusion_matrix(
        recording_labels,
        predictions,
        labels=np.arange(n_classes),
    )
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in CLASS_NAMES],
        columns=[f"Pred_{c}" for c in CLASS_NAMES],
    )

    predictions_df = pd.DataFrame({
        "source_dataset": [key[0] for key in unique_keys],
        "source_recording_id": [key[1] for key in unique_keys],
        "true_label_index": recording_labels,
        "true_diagnosis": [CLASS_NAMES[i] for i in recording_labels],
        "predicted_label_index": predictions,
        "predicted_diagnosis": [CLASS_NAMES[i] for i in predictions],
        "num_windows": window_counts,
    })

    for i, class_name in enumerate(CLASS_NAMES):
        predictions_df[f"prob_{class_name}"] = recording_probs[:, i]

    # Save results independently for each dataset.
    report_df.to_csv(
        OUTPUT_DIR / f"{dataset_name.lower()}_classification_report.csv"
    )
    cm_df.to_csv(
        OUTPUT_DIR / f"{dataset_name.lower()}_confusion_matrix.csv"
    )
    predictions_df.to_csv(
        OUTPUT_DIR / f"{dataset_name.lower()}_recording_predictions.csv",
        index=False,
    )

    return metrics, report_df, cm_df, predictions_df


# ---------------------------------------------------------
# 3. Evaluate each test set separately
# ---------------------------------------------------------
icbhi_metrics, icbhi_report, icbhi_cm, icbhi_predictions = (
    evaluate_test_set(
        model,
        icbhi_test_loader,
        icbhi_test_dataset,
        icbhi_test_meta,
        "ICBHI_TEST",
    )
)

fraiwan_metrics, fraiwan_report, fraiwan_cm, fraiwan_predictions = (
    evaluate_test_set(
        model,
        fraiwan_test_loader,
        fraiwan_test_dataset,
        fraiwan_test_meta,
        "FRAIWAN_EXTERNAL_TEST",
    )
)

metrics_df = pd.DataFrame([icbhi_metrics, fraiwan_metrics])
metrics_df.to_csv(OUTPUT_DIR / "experiment_b_test_summary.csv", index=False)

with open(OUTPUT_DIR / "experiment_b_test_summary.json", "w") as f:
    json.dump(
        {
            "ICBHI_TEST": icbhi_metrics,
            "FRAIWAN_EXTERNAL_TEST": fraiwan_metrics,
        },
        f,
        indent=2,
    )

# ---------------------------------------------------------
# 4. Display headline metrics and per-class reports
# ---------------------------------------------------------
print("=== HEADLINE TEST METRICS ===")
display(metrics_df)

print("\n=== ICBHI INTERNAL TEST: PER-CLASS REPORT ===")
display(icbhi_report)

print("\n=== ICBHI INTERNAL TEST: CONFUSION MATRIX ===")
display(icbhi_cm)

print("\n=== FRAIWAN EXTERNAL TEST: PER-CLASS REPORT ===")
display(fraiwan_report)

print("\n=== FRAIWAN EXTERNAL TEST: CONFUSION MATRIX ===")
display(fraiwan_cm)

print("\nSaved evaluation artifacts to:", OUTPUT_DIR)

/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


=== HEADLINE TEST METRICS ===


,dataset,recordings,windows,accuracy,balanced_accuracy_supported_classes,macro_f1_supported_classes,macro_f1_all_8_classes
0,ICBHI_TEST,394,1667,0.779188,0.392395,0.385319,0.240824
1,FRAIWAN_EXTERNAL_TEST,28,114,0.571429,0.280702,0.242424,0.090909



=== ICBHI INTERNAL TEST: PER-CLASS REPORT ===


,precision,recall,f1-score,support
Asthma,0.000000,0.000000,0.000000,0.000000
Bronchiectasis,0.333333,0.500000,0.400000,2.000000
Bronchiolitis,0.000000,0.000000,0.000000,6.000000
COPD,1.000000,0.814917,0.898021,362.000000
Healthy,0.611111,0.647059,0.628571,17.000000
LRTI,0.000000,0.000000,0.000000,0.000000
Pneumonia,0.000000,0.000000,0.000000,0.000000
URTI,0.000000,0.000000,0.000000,7.000000
accuracy,0.779188,0.779188,0.779188,0.779188
macro avg,0.243056,0.245247,0.240824,394.000000


...


=== FRAIWAN EXTERNAL TEST: PER-CLASS REPORT ===


,precision,recall,f1-score,support
Asthma,0.640000,0.842105,0.727273,19.000000
Bronchiectasis,0.000000,0.000000,0.000000,0.000000
Bronchiolitis,0.000000,0.000000,0.000000,0.000000
COPD,0.000000,0.000000,0.000000,6.000000
Healthy,0.000000,0.000000,0.000000,0.000000
LRTI,0.000000,0.000000,0.000000,0.000000
Pneumonia,0.000000,0.000000,0.000000,3.000000
URTI,0.000000,0.000000,0.000000,0.000000
accuracy,0.571429,0.571429,0.571429,0.571429
macro avg,0.080000,0.105263,0.090909,28.000000



=== FRAIWAN EXTERNAL TEST: CONFUSION MATRIX ===


,Pred_Asthma,Pred_Bronchiectasis,Pred_Bronchiolitis,Pred_COPD,Pred_Healthy,Pred_LRTI,Pred_Pneumonia,Pred_URTI
True_Asthma,16,0,0,3,0,0,0,0
True_Bronchiectasis,0,0,0,0,0,0,0,0
True_Bronchiolitis,0,0,0,0,0,0,0,0
True_COPD,6,0,0,0,0,0,0,0
True_Healthy,0,0,0,0,0,0,0,0
True_LRTI,0,0,0,0,0,0,0,0
True_Pneumonia,3,0,0,0,0,0,0,0
True_URTI,0,0,0,0,0,0,0,0



Saved evaluation artifacts to: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/evaluation


In [92]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Use the existing Experiment B evaluation artifacts
B_CACHE_DIR = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

EVALUATION_DIR = B_CACHE_DIR / "evaluation"

icbhi_pred_path = (
    EVALUATION_DIR / "icbhi_test_recording_predictions.csv"
)
fraiwan_pred_path = (
    EVALUATION_DIR / "fraiwan_external_test_recording_predictions.csv"
)

assert icbhi_pred_path.exists(), f"Missing file: {icbhi_pred_path}"
assert fraiwan_pred_path.exists(), f"Missing file: {fraiwan_pred_path}"

icbhi_pred = pd.read_csv(icbhi_pred_path)
fraiwan_pred = pd.read_csv(fraiwan_pred_path)

# Combine only for auditing, retaining the source as a separate column.
audit_df = pd.concat(
    [icbhi_pred, fraiwan_pred],
    ignore_index=True
)

# ---------------------------------------------------------
# 1. Actual versus predicted class counts
# ---------------------------------------------------------
print("=== ICBHI: TRUE VS PREDICTED ===")
display(pd.crosstab(
    icbhi_pred["true_diagnosis"],
    icbhi_pred["predicted_diagnosis"],
    margins=True
))

print("\n=== FRAIWAN: TRUE VS PREDICTED ===")
display(pd.crosstab(
    fraiwan_pred["true_diagnosis"],
    fraiwan_pred["predicted_diagnosis"],
    margins=True
))

# ---------------------------------------------------------
# 2. Per-class recall and precision, computed from recordings
# ---------------------------------------------------------
def make_class_audit(df, dataset_name):
    rows = []

    for class_name in CLASS_NAMES:
        true_mask = df["true_diagnosis"].eq(class_name)
        pred_mask = df["predicted_diagnosis"].eq(class_name)

        support = int(true_mask.sum())
        predicted_count = int(pred_mask.sum())
        true_positive = int(
            (true_mask & pred_mask).sum()
        )

        rows.append({
            "dataset": dataset_name,
            "class": class_name,
            "true_recordings": support,
            "predicted_recordings": predicted_count,
            "correct_recordings": true_positive,
            "recall": (
                true_positive / support if support else np.nan
            ),
            "precision": (
                true_positive / predicted_count
                if predicted_count else np.nan
            ),
        })

    return pd.DataFrame(rows)

class_audit = pd.concat([
    make_class_audit(icbhi_pred, "ICBHI"),
    make_class_audit(fraiwan_pred, "Fraiwan external"),
], ignore_index=True)

print("\n=== PER-CLASS ERROR AUDIT ===")
display(class_audit.round(4))

# ---------------------------------------------------------
# 3. Confidence of correct versus incorrect predictions
# ---------------------------------------------------------
prob_cols = [f"prob_{c}" for c in CLASS_NAMES]
missing_cols = [c for c in prob_cols if c not in audit_df.columns]
assert not missing_cols, f"Missing probability columns: {missing_cols}"

def confidence_audit(df, dataset_name):
    out = df.copy()
    probabilities = out[prob_cols].to_numpy()
    out["top_probability"] = probabilities.max(axis=1)
    out["correct"] = (
        out["true_diagnosis"] == out["predicted_diagnosis"]
    )

    summary = out.groupby("correct")["top_probability"].agg(
        ["count", "mean", "median", "min", "max"]
    )
    summary.index = summary.index.map({
        True: "Correct prediction",
        False: "Incorrect prediction"
    })
    print(f"\n=== {dataset_name}: PREDICTION CONFIDENCE ===")
    display(summary.round(4))

    return out

icbhi_confidence = confidence_audit(icbhi_pred, "ICBHI")
fraiwan_confidence = confidence_audit(
    fraiwan_pred, "Fraiwan external"
)

print("\nAudit completed. No model was retrained or modified.")

=== ICBHI: TRUE VS PREDICTED ===


predicted_diagnosis,Asthma,Bronchiectasis,COPD,Healthy,Pneumonia,URTI,All
true_diagnosis,,,,,,,
Bronchiectasis,1,1,0,0,0,0,2
Bronchiolitis,1,0,0,1,1,3,6
COPD,58,1,295,0,8,0,362
Healthy,2,0,0,11,3,1,17
URTI,0,1,0,6,0,0,7
All,62,3,295,18,12,4,394



=== FRAIWAN: TRUE VS PREDICTED ===


predicted_diagnosis,Asthma,COPD,All
true_diagnosis,,,
Asthma,16,3,19
COPD,6,0,6
Pneumonia,3,0,3
All,25,3,28



=== PER-CLASS ERROR AUDIT ===


...


=== ICBHI: PREDICTION CONFIDENCE ===


,count,mean,median,min,max
correct,,,,,
Incorrect prediction,87,0.6133,0.6684,0.2339,0.8935
Correct prediction,307,0.9429,0.9988,0.2787,1.0000



=== Fraiwan external: PREDICTION CONFIDENCE ===


,count,mean,median,min,max
correct,,,,,
Incorrect prediction,12,0.5610,0.5161,0.4176,0.7446
Correct prediction,16,0.6339,0.6324,0.4009,0.7867



Audit completed. No model was retrained or modified.


In [94]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "deep_learning"

print("Project root:", PROJECT_ROOT)
print("\nSearching for Experiment B manifests...")

matches = list(DATA_ROOT.rglob("experiment_b_*manifest.csv"))

if matches:
    for path in matches:
        print(path.relative_to(PROJECT_ROOT))
else:
    print("No matching manifests found.")
    print("\nAvailable CSV files in the combined dataset directory:")

    combined_dir = (
        DATA_ROOT / "combined_icbhi_fraiwan_hf_lung_v1"
    )

    if combined_dir.exists():
        for path in sorted(combined_dir.rglob("*.csv")):
            print(path.relative_to(PROJECT_ROOT))
    else:
        print("Combined dataset directory not found:", combined_dir)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification

Searching for Experiment B manifests...
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_icbhi_test_manifest.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_train_manifest.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_fraiwan_test_manifest.csv
data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_window_manifest.csv


In [95]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

COMBINED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

train_manifest = pd.read_csv(
    COMBINED_DIR / "experiment_b_train_manifest.csv"
)
icbhi_manifest = pd.read_csv(
    COMBINED_DIR / "experiment_b_icbhi_test_manifest.csv"
)
fraiwan_manifest = pd.read_csv(
    COMBINED_DIR / "experiment_b_fraiwan_test_manifest.csv"
)

# Normalize column values for a consistent audit.
for df in [train_manifest, icbhi_manifest, fraiwan_manifest]:
    df["source_dataset"] = df["source_dataset"].astype(str)
    df["diagnosis"] = df["diagnosis"].astype(str)

# ---------------------------------------------------------
# 1. Recording and patient support by training source
# ---------------------------------------------------------
print("=== TRAINING RECORDINGS BY SOURCE ===")
display(
    train_manifest.groupby("source_dataset")
    .agg(
        recordings=("source_recording_id", "nunique"),
        patients=("patient_id", "nunique"),
    )
    .reset_index()
)

# ---------------------------------------------------------
# 2. Source × diagnosis counts
# ---------------------------------------------------------
print("\n=== TRAINING RECORDINGS: SOURCE × DIAGNOSIS ===")
display(
    pd.crosstab(
        train_manifest["source_dataset"],
        train_manifest["diagnosis"],
        margins=True,
    )
)

print("\n=== TRAINING DIAGNOSIS PROPORTIONS WITHIN EACH SOURCE (%) ===")
display(
    pd.crosstab(
        train_manifest["source_dataset"],
        train_manifest["diagnosis"],
        normalize="index",
    ).mul(100).round(2)
)

# ---------------------------------------------------------
# 3. Test-set composition, kept separate
# ---------------------------------------------------------
test_manifest = pd.concat(
    [
        icbhi_manifest.assign(evaluation_split="ICBHI_test"),
        fraiwan_manifest.assign(
            evaluation_split="Fraiwan_external_test"
        ),
    ],
    ignore_index=True,
)

print("\n=== TEST RECORDINGS: SPLIT × SOURCE × DIAGNOSIS ===")
display(
    pd.crosstab(
        [
            test_manifest["evaluation_split"],
            test_manifest["source_dataset"],
        ],
        test_manifest["diagnosis"],
    )
)

# ---------------------------------------------------------
# 4. Which training diagnoses occur in only one source?
# ---------------------------------------------------------
source_sets = (
    train_manifest.groupby("diagnosis")["source_dataset"]
    .agg(lambda x: sorted(set(x)))
)

source_exclusivity = pd.DataFrame({
    "diagnosis": source_sets.index,
    "training_sources": source_sets.values,
})
source_exclusivity["number_of_sources"] = (
    source_exclusivity["training_sources"].map(len)
)

print("\n=== DIAGNOSIS COVERAGE ACROSS TRAINING SOURCES ===")
display(
    source_exclusivity.sort_values(
        ["number_of_sources", "diagnosis"]
    ).reset_index(drop=True)
)

# ---------------------------------------------------------
# 5. Integrity checks
# ---------------------------------------------------------
print("\n=== MANIFEST INTEGRITY ===")
for name, df in [
    ("Train", train_manifest),
    ("ICBHI test", icbhi_manifest),
    ("Fraiwan test", fraiwan_manifest),
]:
    print(
        f"{name}: {len(df)} rows, "
        f"{df['source_recording_id'].nunique()} unique recording IDs, "
        f"{df['patient_id'].nunique()} patient IDs"
    )

print("\nAudit complete. No model was retrained or modified.")

=== TRAINING RECORDINGS BY SOURCE ===


,source_dataset,recordings,patients
0,Fraiwan,107,36
1,ICBHI,525,77



=== TRAINING RECORDINGS: SOURCE × DIAGNOSIS ===


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI,All
source_dataset,,,,,,,,,
Fraiwan,74,0,0,21,0,0,12,0,107
ICBHI,1,14,7,431,18,2,36,16,525
All,75,14,7,452,18,2,48,16,632



=== TRAINING DIAGNOSIS PROPORTIONS WITHIN EACH SOURCE (%) ===


...


=== TEST RECORDINGS: SPLIT × SOURCE × DIAGNOSIS ===


,diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,Pneumonia,URTI
evaluation_split,source_dataset,,,,,,,
Fraiwan_external_test,Fraiwan,19,0,0,6,0,3,0
ICBHI_test,ICBHI,0,2,6,362,17,0,7



=== DIAGNOSIS COVERAGE ACROSS TRAINING SOURCES ===


,diagnosis,training_sources,number_of_sources
0,Bronchiectasis,[ICBHI],1
1,Bronchiolitis,[ICBHI],1
2,Healthy,[ICBHI],1
3,LRTI,[ICBHI],1
4,URTI,[ICBHI],1
5,Asthma,"[Fraiwan, ICBHI]",2
6,COPD,"[Fraiwan, ICBHI]",2
7,Pneumonia,"[Fraiwan, ICBHI]",2



=== MANIFEST INTEGRITY ===
Train: 632 rows, 632 unique recording IDs, 113 patient IDs
ICBHI test: 394 rows, 394 unique recording IDs, 49 patient IDs
Fraiwan test: 28 rows, 28 unique recording IDs, 10 patient IDs

Audit complete. No model was retrained or modified.


In [96]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

train_path = B_CACHE_DIR / "inner_train_recordings.csv"
val_path = B_CACHE_DIR / "validation_recordings.csv"

print("Strict cache:", B_CACHE_DIR)
print("Inner training manifest exists:", train_path.exists())
print("Validation manifest exists:", val_path.exists())

if train_path.exists() and val_path.exists():
    inner_train = pd.read_csv(train_path)
    validation = pd.read_csv(val_path)

    print("\n=== INNER TRAINING: SOURCE × DIAGNOSIS ===")
    display(pd.crosstab(
        inner_train["source_dataset"],
        inner_train["diagnosis"],
        margins=True,
    ))

    print("\n=== VALIDATION: SOURCE × DIAGNOSIS ===")
    display(pd.crosstab(
        validation["source_dataset"],
        validation["diagnosis"],
        margins=True,
    ))

    train_patients = set(inner_train["patient_id"].astype(str))
    val_patients = set(validation["patient_id"].astype(str))

    print("\nInner training recordings:", len(inner_train))
    print("Validation recordings:", len(validation))
    print("Inner training patients:", len(train_patients))
    print("Validation patients:", len(val_patients))
    print("Patient overlap:", len(train_patients & val_patients))

    assert len(train_patients & val_patients) == 0, (
        "Patient overlap detected; do not start another experiment."
    )
else:
    print(
        "\nThe split manifests are missing from the expected location. "
        "We will locate them before proceeding."
    )

Strict cache: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict
Inner training manifest exists: True
Validation manifest exists: True

=== INNER TRAINING: SOURCE × DIAGNOSIS ===


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI,All
source_dataset,,,,,,,,,
Fraiwan,59,0,0,18,0,0,12,0,89
ICBHI,1,12,5,344,14,1,27,13,417
All,60,12,5,362,14,1,39,13,506



=== VALIDATION: SOURCE × DIAGNOSIS ===


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI,All
source_dataset,,,,,,,,,
Fraiwan,15,0,0,3,0,0,0,0,18
ICBHI,0,2,2,87,4,1,9,3,108
All,15,2,2,90,4,1,9,3,126



Inner training recordings: 506
Validation recordings: 126
Inner training patients: 92
Validation patients: 21
Patient overlap: 0


In [97]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

for filename in [
    "train_window_metadata.csv",
    "validation_window_metadata.csv",
]:
    path = B_CACHE_DIR / filename
    print(f"\n=== {filename} ===")
    assert path.exists(), f"Missing file: {path}"

    df = pd.read_csv(path)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "source_dataset" in df.columns:
        print("\nSource counts:")
        display(df["source_dataset"].value_counts(dropna=False))

        if "diagnosis" in df.columns:
            print("\nSource × diagnosis window counts:")
            display(pd.crosstab(df["source_dataset"], df["diagnosis"]))

print("\nFeature-cache metadata inspection complete.")


=== train_window_metadata.csv ===
Rows: 2224
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'label_index', 'split', 'recording_path', 'window_index', 'start_seconds', 'end_seconds', 'valid_duration_seconds', 'requires_zero_padding', 'original_sample_rate', 'resampled_sample_rate']

Source counts:


source_dataset
ICBHI      1861
Fraiwan     363
Name: count, dtype: int64


Source × diagnosis window counts:


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
source_dataset,,,,,,,,
Fraiwan,237,0,0,78,0,0,48,0
ICBHI,4,60,20,1557,56,4,108,52



=== validation_window_metadata.csv ===
Rows: 536
Columns: ['source_dataset', 'source_recording_id', 'patient_id', 'diagnosis', 'label_index', 'split', 'recording_path', 'window_index', 'start_seconds', 'end_seconds', 'valid_duration_seconds', 'requires_zero_padding', 'original_sample_rate', 'resampled_sample_rate']

Source counts:


source_dataset
ICBHI      482
Fraiwan     54
Name: count, dtype: int64


Source × diagnosis window counts:


diagnosis,Asthma,Bronchiectasis,Bronchiolitis,COPD,Healthy,LRTI,Pneumonia,URTI
source_dataset,,,,,,,,
Fraiwan,45,0,0,9,0,0,0,0
ICBHI,0,10,8,396,16,4,36,12



Feature-cache metadata inspection complete.


In [98]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path.cwd().parent

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

# Load metadata and cached feature arrays
train_meta = pd.read_csv(
    B_CACHE_DIR / "train_window_metadata.csv"
).reset_index(drop=True)

val_meta = pd.read_csv(
    B_CACHE_DIR / "validation_window_metadata.csv"
).reset_index(drop=True)

train_npz = np.load(
    B_CACHE_DIR / "train_features.npz",
    allow_pickle=True
)

val_npz = np.load(
    B_CACHE_DIR / "validation_features.npz",
    allow_pickle=True
)

print("=== ARRAY KEYS ===")
print("Training:", train_npz.files)
print("Validation:", val_npz.files)

for name, archive, meta in [
    ("Training", train_npz, train_meta),
    ("Validation", val_npz, val_meta),
]:
    print(f"\n=== {name.upper()} ARRAY CHECK ===")

    for key in archive.files:
        arr = archive[key]
        print(
            f"{key}: shape={arr.shape}, dtype={arr.dtype}"
        )

    # Expected feature arrays and labels must align with metadata.
    for key in ["logmel", "mfcc", "chroma", "labels"]:
        assert key in archive.files, (
            f"{name}: expected key {key!r} not found"
        )
        assert len(archive[key]) == len(meta), (
            f"{name}: {key} length does not match metadata"
        )

    assert np.array_equal(
        archive["labels"].astype(int),
        meta["label_index"].to_numpy(dtype=int)
    ), f"{name}: labels do not match metadata order"

    print("Metadata and label ordering: verified")

# Verify original training-only normalization statistics are available.
stats_path = B_CACHE_DIR / "normalization_stats.json"
assert stats_path.exists(), f"Missing normalization stats: {stats_path}"

with open(stats_path, "r") as f:
    stats = json.load(f)

print("\n=== NORMALIZATION STATISTICS KEYS ===")
print(list(stats.keys()))
print("\nArray alignment checks passed.")
print("No model was trained or modified.")

=== ARRAY KEYS ===
Training: ['logmel', 'mfcc', 'chroma', 'labels', 'paths']
Validation: ['logmel', 'mfcc', 'chroma', 'labels', 'paths']

=== TRAINING ARRAY CHECK ===
logmel: shape=(2224, 1, 64, 157), dtype=float32
mfcc: shape=(2224, 1, 13, 157), dtype=float32
chroma: shape=(2224, 1, 12, 157), dtype=float32
labels: shape=(2224,), dtype=int64
paths: shape=(2224,), dtype=<U114
Metadata and label ordering: verified

=== VALIDATION ARRAY CHECK ===
logmel: shape=(536, 1, 64, 157), dtype=float32
mfcc: shape=(536, 1, 13, 157), dtype=float32
chroma: shape=(536, 1, 12, 157), dtype=float32
labels: shape=(536,), dtype=int64
paths: shape=(536,), dtype=<U110
Metadata and label ordering: verified

=== NORMALIZATION STATISTICS KEYS ===
['logmel', 'mfcc', 'chroma']

Array alignment checks passed.
No model was trained or modified.


In [99]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

C1_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c1_icbhi_only"
)

C1_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 1. Load existing arrays, metadata, and normalization stats
# ---------------------------------------------------------
train_meta = pd.read_csv(
    B_CACHE_DIR / "train_window_metadata.csv"
).reset_index(drop=True)

val_meta = pd.read_csv(
    B_CACHE_DIR / "validation_window_metadata.csv"
).reset_index(drop=True)

train_npz = np.load(
    B_CACHE_DIR / "train_features.npz",
    allow_pickle=True
)

val_npz = np.load(
    B_CACHE_DIR / "validation_features.npz",
    allow_pickle=True
)

with open(B_CACHE_DIR / "normalization_stats.json", "r") as f:
    old_stats = json.load(f)

FEATURE_KEYS = ["logmel", "mfcc", "chroma"]

# Verify metadata and arrays before filtering.
for name, archive, meta in [
    ("training", train_npz, train_meta),
    ("validation", val_npz, val_meta),
]:
    for key in FEATURE_KEYS + ["labels"]:
        assert len(archive[key]) == len(meta), (
            f"{name}: {key} is not aligned with metadata"
        )

    assert np.array_equal(
        archive["labels"].astype(int),
        meta["label_index"].to_numpy(dtype=int),
    ), f"{name}: label ordering mismatch"

# ---------------------------------------------------------
# 2. Select ICBHI windows only
# ---------------------------------------------------------
train_mask = train_meta["source_dataset"].eq("ICBHI").to_numpy()
val_mask = val_meta["source_dataset"].eq("ICBHI").to_numpy()

assert train_mask.any(), "No ICBHI training windows found"
assert val_mask.any(), "No ICBHI validation windows found"

train_meta_c1 = train_meta.loc[train_mask].reset_index(drop=True)
val_meta_c1 = val_meta.loc[val_mask].reset_index(drop=True)

# Recover approximate raw features from the existing normalized arrays.
# The recovery can introduce small floating-point differences.
raw_train = {}
raw_val = {}

for key in FEATURE_KEYS:
    mean = float(old_stats[key]["mean"])
    std = float(old_stats[key]["std"])

    raw_train[key] = (
        train_npz[key][train_mask].astype(np.float32) * std + mean
    )
    raw_val[key] = (
        val_npz[key][val_mask].astype(np.float32) * std + mean
    )

# ---------------------------------------------------------
# 3. Compute normalization statistics from ICBHI train only
# ---------------------------------------------------------
c1_stats = {}

for key in FEATURE_KEYS:
    values = raw_train[key].astype(np.float64)

    mean = float(values.mean())
    std = float(values.std(ddof=0))

    assert np.isfinite(mean) and np.isfinite(std)
    assert std > 0, f"Invalid standard deviation for {key}"

    c1_stats[key] = {
        "mean": mean,
        "std": std,
        "count": int(values.size),
        "normalization_source": "ICBHI inner-training windows only",
        "recovered_from_existing_normalized_cache": True,
    }

# ---------------------------------------------------------
# 4. Normalize and save the C1 datasets
# ---------------------------------------------------------
def normalize_features(raw_features, stats):
    return {
        key: (
            (raw_features[key] - stats[key]["mean"])
            / stats[key]["std"]
        ).astype(np.float32)
        for key in FEATURE_KEYS
    }

train_features_c1 = normalize_features(raw_train, c1_stats)
val_features_c1 = normalize_features(raw_val, c1_stats)

np.savez_compressed(
    C1_CACHE_DIR / "train_features.npz",
    **train_features_c1,
    labels=train_npz["labels"][train_mask].astype(np.int64),
    paths=train_npz["paths"][train_mask],
)

np.savez_compressed(
    C1_CACHE_DIR / "validation_features.npz",
    **val_features_c1,
    labels=val_npz["labels"][val_mask].astype(np.int64),
    paths=val_npz["paths"][val_mask],
)

train_meta_c1.to_csv(
    C1_CACHE_DIR / "train_window_metadata.csv", index=False
)
val_meta_c1.to_csv(
    C1_CACHE_DIR / "validation_window_metadata.csv", index=False
)

with open(C1_CACHE_DIR / "normalization_stats.json", "w") as f:
    json.dump(c1_stats, f, indent=2)

# Save the source-filtering and preprocessing description.
report = {
    "experiment": "C1_ICBHI_only",
    "training_source": "ICBHI",
    "validation_source": "ICBHI",
    "train_windows": int(len(train_meta_c1)),
    "validation_windows": int(len(val_meta_c1)),
    "train_recordings": int(
        train_meta_c1["source_recording_id"].nunique()
    ),
    "validation_recordings": int(
        val_meta_c1["source_recording_id"].nunique()
    ),
    "train_patients": int(train_meta_c1["patient_id"].nunique()),
    "validation_patients": int(val_meta_c1["patient_id"].nunique()),
    "normalization": (
        "Statistics computed from ICBHI inner-training windows only; "
        "raw features approximately reconstructed from existing normalized "
        "features, so small floating-point differences may occur."
    ),
    "test_sets_used_for_training_or_normalization": False,
}

with open(C1_CACHE_DIR / "preparation_report.json", "w") as f:
    json.dump(report, f, indent=2)

# ---------------------------------------------------------
# 5. Verify the new cache
# ---------------------------------------------------------
print("=== EXPERIMENT C1 CACHE ===")
print("Directory:", C1_CACHE_DIR)
print("Training windows:", len(train_meta_c1))
print("Validation windows:", len(val_meta_c1))
print("Training recordings:", train_meta_c1["source_recording_id"].nunique())
print("Validation recordings:", val_meta_c1["source_recording_id"].nunique())
print("Training patients:", train_meta_c1["patient_id"].nunique())
print("Validation patients:", val_meta_c1["patient_id"].nunique())

print("\n=== TRAINING CLASS SUPPORT (WINDOWS) ===")
display(
    train_meta_c1.groupby("diagnosis")
    .agg(
        windows=("label_index", "size"),
        recordings=("source_recording_id", "nunique"),
        patients=("patient_id", "nunique"),
    )
    .reindex(CLASS_NAMES, fill_value=0)
)

print("\n=== VALIDATION CLASS SUPPORT (WINDOWS) ===")
display(
    val_meta_c1.groupby("diagnosis")
    .agg(
        windows=("label_index", "size"),
        recordings=("source_recording_id", "nunique"),
        patients=("patient_id", "nunique"),
    )
    .reindex(CLASS_NAMES, fill_value=0)
)

print("\n=== NORMALIZATION CHECK ===")
for key in FEATURE_KEYS:
    x = train_features_c1[key]
    print(
        f"{key}: mean={x.mean():.5f}, std={x.std():.5f}, "
        f"finite={bool(np.isfinite(x).all())}"
    )

assert set(train_meta_c1["source_dataset"]) == {"ICBHI"}
assert set(val_meta_c1["source_dataset"]) == {"ICBHI"}
assert not (
    set(train_meta_c1["patient_id"].astype(str))
    & set(val_meta_c1["patient_id"].astype(str))
), "Patient overlap detected"

print("\nC1 cache created and verified. No model was trained.")

=== EXPERIMENT C1 CACHE ===
Directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only
Training windows: 1861
Validation windows: 482
Training recordings: 417
Validation recordings: 108
Training patients: 62
Validation patients: 15

=== TRAINING CLASS SUPPORT (WINDOWS) ===


,windows,recordings,patients
diagnosis,,,
Asthma,4,1,1
Bronchiectasis,60,12,5
Bronchiolitis,20,5,2
COPD,1557,344,31
Healthy,56,14,9
LRTI,4,1,1
Pneumonia,108,27,5
URTI,52,13,8



=== VALIDATION CLASS SUPPORT (WINDOWS) ===


,windows,recordings,patients
diagnosis,,,
Asthma,0,0,0
Bronchiectasis,10,2,1
Bronchiolitis,8,2,1
COPD,396,87,6
Healthy,16,4,3
LRTI,4,1,1
Pneumonia,36,9,1
URTI,12,3,2



=== NORMALIZATION CHECK ===
logmel: mean=0.00000, std=1.00000, finite=True
mfcc: mean=0.00000, std=1.00000, finite=True
chroma: mean=-0.00000, std=1.00000, finite=True

C1 cache created and verified. No model was trained.


In [100]:
# Verify the training components from Experiment B are available.
required_objects = [
    "LightweightMultiFeatureCNN",
    "RespiratoryFeatureDataset",
    "evaluate_recording_level",
    "DEVICE",
    "CLASS_NAMES",
    "MAX_EPOCHS",
    "PATIENCE",
    "MIN_DELTA",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

print("=== C1 TRAINING READINESS ===")
print("Missing objects:", missing)

if not missing:
    print("Device:", DEVICE)
    print("Classes:", CLASS_NAMES)
    print("Maximum epochs:", MAX_EPOCHS)
    print("Early-stopping patience:", PATIENCE)
    print("Minimum improvement:", MIN_DELTA)
    print("\nTraining components are available.")
else:
    print(
        "\nDo not start training yet. We need to restore or redefine "
        "the missing components first."
    )

=== C1 TRAINING READINESS ===
Missing objects: []
Device: mps
Classes: ['Asthma', 'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'LRTI', 'Pneumonia', 'URTI']
Maximum epochs: 60
Early-stopping patience: 10
Minimum improvement: 0.0001

Training components are available.


In [101]:
import json
import numpy as np
import torch
import pandas as pd

from pathlib import Path
from torch.utils.data import DataLoader
from collections import Counter

# ---------------------------------------------------------
# 1. C1 paths and configuration
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

C1_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c1_icbhi_only"
)

C1_OUTPUT_DIR = C1_CACHE_DIR / "checkpoints"
C1_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

C1_BEST_MODEL_PATH = (
    C1_OUTPUT_DIR / "experiment_c1_icbhi_only_best.pt"
)
C1_HISTORY_PATH = (
    C1_OUTPUT_DIR / "experiment_c1_icbhi_only_history.csv"
)
C1_CONFIG_PATH = (
    C1_OUTPUT_DIR / "experiment_c1_icbhi_only_config.json"
)

# ---------------------------------------------------------
# 2. Load C1 datasets
# ---------------------------------------------------------
c1_train_npz = C1_CACHE_DIR / "train_features.npz"
c1_val_npz = C1_CACHE_DIR / "validation_features.npz"

assert c1_train_npz.exists()
assert c1_val_npz.exists()

c1_train_dataset = RespiratoryFeatureDataset(
    c1_train_npz,
    augment=True,
)
c1_val_dataset = RespiratoryFeatureDataset(
    c1_val_npz,
    augment=False,
)

C1_BATCH_SIZE = 32

c1_train_loader = DataLoader(
    c1_train_dataset,
    batch_size=C1_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
c1_val_loader = DataLoader(
    c1_val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
)

# ---------------------------------------------------------
# 3. Calculate class weights from C1 training labels only
# ---------------------------------------------------------
c1_labels = np.asarray(c1_train_dataset.labels, dtype=np.int64)
c1_counts = np.bincount(
    c1_labels,
    minlength=len(CLASS_NAMES),
)

# Inverse-square-root weighting, matching Experiment B's method.
c1_weights = 1.0 / np.sqrt(np.maximum(c1_counts, 1))
c1_weights = c1_weights / (
    (c1_counts * c1_weights).sum() / max(c1_counts.sum(), 1)
)

# Keep absent classes at zero weight. They contribute no training examples.
c1_weights[c1_counts == 0] = 0.0

c1_class_weights = torch.tensor(
    c1_weights,
    dtype=torch.float32,
    device=DEVICE,
)

# ---------------------------------------------------------
# 4. Initialize a separate C1 model and optimizer
# ---------------------------------------------------------
c1_model = LightweightMultiFeatureCNN(
    num_classes=len(CLASS_NAMES)
).to(DEVICE)

C1_LEARNING_RATE = 0.001

c1_criterion = torch.nn.CrossEntropyLoss(
    weight=c1_class_weights
)

c1_optimizer = torch.optim.AdamW(
    c1_model.parameters(),
    lr=C1_LEARNING_RATE,
    weight_decay=1e-4,
)

c1_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    c1_optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-5,
)

# ---------------------------------------------------------
# 5. Save the configuration
# ---------------------------------------------------------
c1_config = {
    "experiment": "C1_ICBHI_only",
    "architecture": "LightweightMultiFeatureCNN",
    "trainable_parameters": int(sum(
        p.numel() for p in c1_model.parameters()
        if p.requires_grad
    )),
    "training_windows": len(c1_train_dataset),
    "validation_windows": len(c1_val_dataset),
    "class_names": CLASS_NAMES,
    "training_class_counts_windows": {
        CLASS_NAMES[i]: int(c1_counts[i])
        for i in range(len(CLASS_NAMES))
    },
    "class_weighting": "inverse square root of training-window class counts",
    "batch_size": C1_BATCH_SIZE,
    "learning_rate": C1_LEARNING_RATE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "normalization": "ICBHI inner-training windows only; reconstructed raw features",
    "validation_selection": "recording-level macro-F1 on supported classes",
    "device": str(DEVICE),
}

with open(C1_CONFIG_PATH, "w") as f:
    json.dump(c1_config, f, indent=2)

# ---------------------------------------------------------
# 6. Readiness summary
# ---------------------------------------------------------
print("=== C1 INITIALIZATION COMPLETE ===")
print("Device:", DEVICE)
print("Trainable parameters:", c1_config["trainable_parameters"])
print("Training windows:", len(c1_train_dataset))
print("Validation windows:", len(c1_val_dataset))
print("Training batches per epoch:", len(c1_train_loader))
print("\nClass counts and weights:")

display(pd.DataFrame({
    "class": CLASS_NAMES,
    "training_windows": c1_counts,
    "class_weight": c1_weights,
}))

print("\nCheckpoint:", C1_BEST_MODEL_PATH)
print("History:", C1_HISTORY_PATH)
print("Config:", C1_CONFIG_PATH)
print("\nModel initialized only; training has not started.")

=== C1 INITIALIZATION COMPLETE ===
Device: mps
Trainable parameters: 5408
Training windows: 1861
Validation windows: 482
Training batches per epoch: 59

Class counts and weights:


,class,training_windows,class_weight
0,Asthma,4,11.521270
1,Bronchiectasis,60,2.974779
2,Bronchiolitis,20,5.152469
3,COPD,1557,0.583964
4,Healthy,56,3.079189
5,LRTI,4,11.521270
6,Pneumonia,108,2.217269
7,URTI,52,3.195425



Checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/checkpoints/experiment_c1_icbhi_only_best.pt
History: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/checkpoints/experiment_c1_icbhi_only_history.csv
Config: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/checkpoints/experiment_c1_icbhi_only_config.json

Model initialized only; training has not started.


In [103]:
import inspect

print(inspect.getsource(evaluate_recording_level))

@torch.no_grad()
def evaluate_recording_level(model, loader, metadata, device):
    model.eval()

    all_probs = []
    all_labels = []

    for logmel, mfcc, chroma, labels in loader:
        logmel = logmel.to(device)
        mfcc = mfcc.to(device)
        chroma = chroma.to(device)

        logits = model(logmel, mfcc, chroma)
        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

    window_probs = np.concatenate(all_probs, axis=0)
    window_labels = np.concatenate(all_labels, axis=0)

    assert len(window_probs) == len(metadata)
    assert np.array_equal(window_labels, validation_dataset.labels)

    # Average window probabilities for each recording.
    keys = list(zip(
        metadata["source_dataset"].astype(str),
        metadata["source_recording_id"].astype(str),
    ))
    unique_keys = list(dict.fromkeys(keys))
    key_to_idx = {key: i for i, key in enumerate(unique_keys)}

    recording_pr

In [104]:
import numpy as np
import torch

from sklearn.metrics import accuracy_score, f1_score

@torch.no_grad()
def evaluate_recording_level(model, loader, metadata, device):
    model.eval()

    all_probs = []
    all_labels = []

    for logmel, mfcc, chroma, labels in loader:
        logits = model(
            logmel.to(device),
            mfcc.to(device),
            chroma.to(device),
        )

        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    assert all_probs, "The evaluation loader is empty."

    window_probs = np.concatenate(all_probs, axis=0)
    window_labels = np.concatenate(all_labels, axis=0)

    assert len(window_probs) == len(metadata), (
        "Window probabilities and metadata have different lengths."
    )
    assert len(window_labels) == len(metadata), (
        "Window labels and metadata have different lengths."
    )
    assert np.array_equal(
        window_labels,
        metadata["label_index"].to_numpy(dtype=int),
    ), "Loader labels do not match the supplied metadata."

    keys = list(zip(
        metadata["source_dataset"].astype(str),
        metadata["source_recording_id"].astype(str),
    ))

    unique_keys = list(dict.fromkeys(keys))
    key_to_idx = {key: i for i, key in enumerate(unique_keys)}

    recording_probs = np.zeros(
        (len(unique_keys), len(CLASS_NAMES)),
        dtype=np.float64,
    )
    recording_labels = np.full(len(unique_keys), -1, dtype=np.int64)
    counts = np.zeros(len(unique_keys), dtype=np.int64)

    for i, key in enumerate(keys):
        group_idx = key_to_idx[key]
        label = int(window_labels[i])

        if recording_labels[group_idx] == -1:
            recording_labels[group_idx] = label
        else:
            assert recording_labels[group_idx] == label, (
                f"Inconsistent labels for recording {key}"
            )

        recording_probs[group_idx] += window_probs[i]
        counts[group_idx] += 1

    assert np.all(counts > 0)
    recording_probs /= counts[:, None]

    predictions = recording_probs.argmax(axis=1)
    supported_classes = np.unique(recording_labels)

    return {
        "macro_f1_recording": float(
            f1_score(
                recording_labels,
                predictions,
                labels=supported_classes,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy_recording": float(
            accuracy_score(recording_labels, predictions)
        ),
        "recording_labels": recording_labels,
        "recording_predictions": predictions,
        "recording_probabilities": recording_probs,
        "recording_keys": unique_keys,
    }


print("Corrected evaluator defined.")
print("No training has run in this cell.")

Corrected evaluator defined.
No training has run in this cell.


In [105]:
# Verify the corrected evaluator on the current C1 model.
c1_val_result = evaluate_recording_level(
    c1_model,
    c1_val_loader,
    c1_val_meta,
    DEVICE,
)

print("C1 validation recordings:", len(c1_val_result["recording_labels"]))
print(
    "Initial validation macro-F1:",
    round(c1_val_result["macro_f1_recording"], 4),
)
print(
    "Initial validation accuracy:",
    round(c1_val_result["accuracy_recording"], 4),
)
print("Verification passed; training has not started.")

C1 validation recordings: 108
Initial validation macro-F1: 0.1275
Initial validation accuracy: 0.8056
Verification passed; training has not started.


In [107]:
import time
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score

# ---------------------------------------------------------
# 1. Load validation metadata
# ---------------------------------------------------------
c1_val_meta = pd.read_csv(
    C1_CACHE_DIR / "validation_window_metadata.csv"
).reset_index(drop=True)

assert len(c1_val_meta) == len(c1_val_dataset)

# Confirm the validation evaluator's expected inputs.
# This function was used for Experiment B validation.
assert callable(evaluate_recording_level)

# ---------------------------------------------------------
# 2. Training configuration
# ---------------------------------------------------------
c1_best_val_f1 = -np.inf
c1_best_epoch = 0
c1_epochs_without_improvement = 0
c1_history = []

C1_MAX_EPOCHS = MAX_EPOCHS
C1_PATIENCE = PATIENCE
C1_MIN_DELTA = MIN_DELTA

print("Starting C1 training...")
print("Device:", DEVICE)
print("Maximum epochs:", C1_MAX_EPOCHS)
print("Early stopping patience:", C1_PATIENCE)
print("-" * 78)

# ---------------------------------------------------------
# 3. Training loop
# ---------------------------------------------------------
for epoch in range(1, C1_MAX_EPOCHS + 1):
    epoch_start = time.time()
    c1_model.train()

    total_loss = 0.0
    total_examples = 0
    train_true = []
    train_pred = []

    for logmel, mfcc, chroma, labels in c1_train_loader:
        logmel = logmel.to(DEVICE)
        mfcc = mfcc.to(DEVICE)
        chroma = chroma.to(DEVICE)
        labels = labels.to(DEVICE)

        c1_optimizer.zero_grad(set_to_none=True)

        logits = c1_model(logmel, mfcc, chroma)
        loss = c1_criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            c1_model.parameters(), max_norm=5.0
        )
        c1_optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        train_true.extend(labels.detach().cpu().numpy().tolist())
        train_pred.extend(
            logits.detach().argmax(dim=1).cpu().numpy().tolist()
        )

    train_loss = total_loss / max(total_examples, 1)

    train_macro_f1 = f1_score(
        train_true,
        train_pred,
        labels=np.unique(train_true),
        average="macro",
        zero_division=0,
    )

    # Recording-level validation using the existing evaluator.
    c1_val_result = evaluate_recording_level(
        c1_model,
        c1_val_loader,
        c1_val_meta,
        DEVICE,
    )

    # The existing Experiment B evaluator returns a dictionary.
    # Extract its recorded validation metrics.
    if isinstance(c1_val_result, dict):
        c1_val_f1 = c1_val_result.get(
            "macro_f1_recording",
            c1_val_result.get("macro_f1")
        )
        c1_val_accuracy = c1_val_result.get(
            "accuracy_recording",
            c1_val_result.get("accuracy")
        )
    else:
        raise TypeError(
            "Unexpected return type from evaluate_recording_level: "
            f"{type(c1_val_result)}. Stop and inspect the evaluator."
        )

    if c1_val_f1 is None or not np.isfinite(c1_val_f1):
        raise ValueError(
            "Could not obtain a valid recording-level validation macro-F1. "
            f"Returned result: {c1_val_result}"
        )

    c1_val_f1 = float(c1_val_f1)
    c1_val_accuracy = (
        float(c1_val_accuracy)
        if c1_val_accuracy is not None
        else float("nan")
    )

    c1_scheduler.step(c1_val_f1)

    improved = c1_val_f1 > c1_best_val_f1 + C1_MIN_DELTA

    if improved:
        c1_best_val_f1 = c1_val_f1
        c1_best_epoch = epoch
        c1_epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": c1_model.state_dict(),
                "epoch": epoch,
                "best_validation_macro_f1_recording": c1_best_val_f1,
                "validation_accuracy_recording": c1_val_accuracy,
                "class_names": CLASS_NAMES,
                "experiment": "C1_ICBHI_only",
                "config_path": str(C1_CONFIG_PATH),
            },
            C1_BEST_MODEL_PATH,
        )
    else:
        c1_epochs_without_improvement += 1

    elapsed = time.time() - epoch_start

    c1_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_macro_f1_window": float(train_macro_f1),
        "validation_macro_f1_recording": c1_val_f1,
        "validation_accuracy_recording": c1_val_accuracy,
        "learning_rate": c1_optimizer.param_groups[0]["lr"],
        "seconds": elapsed,
        "best_checkpoint_updated": improved,
    })

    pd.DataFrame(c1_history).to_csv(
        C1_HISTORY_PATH, index=False
    )

    print(
        f"Epoch {epoch:02d}/{C1_MAX_EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"train-window macro-F1={train_macro_f1:.4f} | "
        f"val-recording macro-F1={c1_val_f1:.4f} | "
        f"val-accuracy={c1_val_accuracy:.4f} | "
        f"lr={c1_optimizer.param_groups[0]['lr']:.6f} | "
        f"{elapsed:.1f}s"
        + (" | BEST" if improved else "")
    )

    if c1_epochs_without_improvement >= C1_PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch}. "
            f"Best epoch: {c1_best_epoch}."
        )
        break

# ---------------------------------------------------------
# 4. Restore best C1 checkpoint
# ---------------------------------------------------------
assert C1_BEST_MODEL_PATH.exists(), (
    "No C1 checkpoint was saved."
)

c1_checkpoint = torch.load(
    C1_BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

c1_model.load_state_dict(c1_checkpoint["model_state_dict"])
c1_model.eval()

print("\n=== C1 TRAINING SUMMARY ===")
print("Best epoch:", c1_checkpoint["epoch"])
print(
    "Best validation recording macro-F1:",
    round(
        c1_checkpoint["best_validation_macro_f1_recording"], 4
    ),
)
print(
    "Best validation recording accuracy:",
    round(c1_checkpoint["validation_accuracy_recording"], 4),
)
print("Checkpoint:", C1_BEST_MODEL_PATH)
print("History:", C1_HISTORY_PATH)

Starting C1 training...
Device: mps
Maximum epochs: 60
Early stopping patience: 10
------------------------------------------------------------------------------
Epoch 01/60 | loss=1.4688 | train-window macro-F1=0.1204 | val-recording macro-F1=0.1275 | val-accuracy=0.8056 | lr=0.001000 | 4.0s | BEST
Epoch 02/60 | loss=1.2460 | train-window macro-F1=0.1692 | val-recording macro-F1=0.2195 | val-accuracy=0.7315 | lr=0.001000 | 3.6s | BEST
Epoch 03/60 | loss=1.1345 | train-window macro-F1=0.2016 | val-recording macro-F1=0.2210 | val-accuracy=0.7685 | lr=0.001000 | 3.3s | BEST
Epoch 04/60 | loss=1.0308 | train-window macro-F1=0.2389 | val-recording macro-F1=0.1730 | val-accuracy=0.7500 | lr=0.001000 | 3.1s
Epoch 05/60 | loss=0.9829 | train-window macro-F1=0.2690 | val-recording macro-F1=0.1803 | val-accuracy=0.6204 | lr=0.001000 | 3.2s
Epoch 06/60 | loss=0.9163 | train-window macro-F1=0.3011 | val-recording macro-F1=0.1921 | val-accuracy=0.7685 | lr=0.001000 | 3.3s
Epoch 07/60 | loss=0.8527

In [112]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from torch.utils.data import DataLoader

# ---------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

C1_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c1_icbhi_only"
)

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

C1_EVAL_DIR = C1_CACHE_DIR / "evaluation"
C1_EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the original, held-out test features.
icbhi_test_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "icbhi_test_features.npz",
    augment=False,
)
fraiwan_test_dataset = RespiratoryFeatureDataset(
    B_CACHE_DIR / "fraiwan_test_features.npz",
    augment=False,
)

icbhi_test_loader = DataLoader(
    icbhi_test_dataset, batch_size=64, shuffle=False, num_workers=0
)
fraiwan_test_loader = DataLoader(
    fraiwan_test_dataset, batch_size=64, shuffle=False, num_workers=0
)

icbhi_test_meta = pd.read_csv(
    B_CACHE_DIR / "icbhi_test_window_metadata.csv"
).reset_index(drop=True)

fraiwan_test_meta = pd.read_csv(
    B_CACHE_DIR / "fraiwan_test_window_metadata.csv"
).reset_index(drop=True)

# ---------------------------------------------------------
# 2. Generic recording-level test evaluation
# ---------------------------------------------------------
def evaluate_c1_test_set(model, loader, metadata, dataset_name):
    result = evaluate_recording_level(
        model, loader, metadata, DEVICE
    )

    y_true = result["recording_labels"]
    y_pred = result["recording_predictions"]
    probabilities = result["recording_probabilities"]
    recording_keys = result["recording_keys"]

    supported = np.unique(y_true)

    metrics = {
        "dataset": dataset_name,
        "recordings": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy_supported_classes": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1_supported_classes": float(
            f1_score(
                y_true, y_pred,
                labels=supported,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_8_classes": float(
            f1_score(
                y_true, y_pred,
                labels=np.arange(len(CLASS_NAMES)),
                average="macro",
                zero_division=0,
            )
        ),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(len(CLASS_NAMES)),
    )

    report_df = pd.DataFrame(report).T
    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{c}" for c in CLASS_NAMES],
        columns=[f"Pred_{c}" for c in CLASS_NAMES],
    )

    # Recover metadata per recording from the window metadata.
    key_columns = ["source_dataset", "source_recording_id"]
    recording_metadata = metadata[key_columns + [
        "patient_id", "diagnosis"
    ]].copy()

    recording_metadata["source_dataset"] = (
        recording_metadata["source_dataset"].astype(str)
    )
    recording_metadata["source_recording_id"] = (
        recording_metadata["source_recording_id"].astype(str)
    )

    recording_metadata = recording_metadata.drop_duplicates(
        subset=key_columns
    ).set_index(key_columns)

    rows = []
    for i, key in enumerate(recording_keys):
        source, recording_id = key
        meta_row = recording_metadata.loc[(source, recording_id)]

        row = {
            "source_dataset": source,
            "source_recording_id": recording_id,
            "patient_id": meta_row["patient_id"],
            "true_diagnosis": CLASS_NAMES[int(y_true[i])],
            "predicted_diagnosis": CLASS_NAMES[int(y_pred[i])],
        }

        for j, class_name in enumerate(CLASS_NAMES):
            row[f"prob_{class_name}"] = float(probabilities[i, j])

        rows.append(row)

    predictions_df = pd.DataFrame(rows)

    stem = dataset_name.lower()
    report_df.to_csv(
        C1_EVAL_DIR / f"{stem}_classification_report.csv"
    )
    cm_df.to_csv(
        C1_EVAL_DIR / f"{stem}_confusion_matrix.csv"
    )
    predictions_df.to_csv(
        C1_EVAL_DIR / f"{stem}_recording_predictions.csv",
        index=False,
    )

    return metrics, report_df, cm_df, predictions_df


# ---------------------------------------------------------
# 3. Run C1 test evaluation
# ---------------------------------------------------------
c1_icbhi_metrics, c1_icbhi_report, c1_icbhi_cm, c1_icbhi_preds = (
    evaluate_c1_test_set(
        c1_test_model,
        icbhi_test_loader,
        icbhi_test_meta,
        "C1_ICBHI_TEST",
    )
)

c1_fraiwan_metrics, c1_fraiwan_report, c1_fraiwan_cm, c1_fraiwan_preds = (
    evaluate_c1_test_set(
        c1_test_model,
        fraiwan_test_loader,
        fraiwan_test_meta,
        "C1_FRAIWAN_EXTERNAL_TEST",
    )
)

c1_metrics_df = pd.DataFrame([
    c1_icbhi_metrics,
    c1_fraiwan_metrics,
])

c1_metrics_df.to_csv(
    C1_EVAL_DIR / "c1_test_summary.csv",
    index=False,
)

with open(C1_EVAL_DIR / "c1_test_summary.json", "w") as f:
    json.dump({
        "ICBHI_TEST": c1_icbhi_metrics,
        "FRAIWAN_EXTERNAL_TEST": c1_fraiwan_metrics,
    }, f, indent=2)

# ---------------------------------------------------------
# 4. Display results
# ---------------------------------------------------------
print("=== C1 TEST SUMMARY ===")
display(c1_metrics_df)

print("\n=== C1 ICBHI TEST PER-CLASS REPORT ===")
display(c1_icbhi_report)

print("\n=== C1 ICBHI TEST CONFUSION MATRIX ===")
display(c1_icbhi_cm)

print("\n=== C1 FRAIWAN EXTERNAL TEST PER-CLASS REPORT ===")
display(c1_fraiwan_report)

print("\n=== C1 FRAIWAN EXTERNAL TEST CONFUSION MATRIX ===")
display(c1_fraiwan_cm)

print("\nC1 results saved to:", C1_EVAL_DIR)
print("Experiment B evaluation artifacts were not modified.")

=== C1 TEST SUMMARY ===


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


,dataset,recordings,accuracy,balanced_accuracy_supported_classes,macro_f1_supported_classes,macro_f1_all_8_classes
0,C1_ICBHI_TEST,394,0.763959,0.450713,0.374572,0.234107
1,C1_FRAIWAN_EXTERNAL_TEST,28,0.000000,0.000000,0.000000,0.000000



=== C1 ICBHI TEST PER-CLASS REPORT ===


,precision,recall,f1-score,support
Asthma,0.000000,0.000000,0.000000,0.000000
Bronchiectasis,0.076923,0.500000,0.133333,2.000000
Bronchiolitis,0.000000,0.000000,0.000000,6.000000
COPD,1.000000,0.795580,0.886154,362.000000
Healthy,0.642857,0.529412,0.580645,17.000000
LRTI,0.000000,0.000000,0.000000,0.000000
Pneumonia,0.000000,0.000000,0.000000,0.000000
URTI,0.200000,0.428571,0.272727,7.000000
accuracy,0.763959,0.763959,0.763959,0.763959
macro avg,0.239973,0.281695,0.234107,394.000000


...


=== C1 FRAIWAN EXTERNAL TEST PER-CLASS REPORT ===


,precision,recall,f1-score,support
Asthma,0.0,0.0,0.0,19.0
Bronchiectasis,0.0,0.0,0.0,0.0
Bronchiolitis,0.0,0.0,0.0,0.0
COPD,0.0,0.0,0.0,6.0
Healthy,0.0,0.0,0.0,0.0
LRTI,0.0,0.0,0.0,0.0
Pneumonia,0.0,0.0,0.0,3.0
URTI,0.0,0.0,0.0,0.0
accuracy,0.0,0.0,0.0,0.0
macro avg,0.0,0.0,0.0,28.0



=== C1 FRAIWAN EXTERNAL TEST CONFUSION MATRIX ===


,Pred_Asthma,Pred_Bronchiectasis,Pred_Bronchiolitis,Pred_COPD,Pred_Healthy,Pred_LRTI,Pred_Pneumonia,Pred_URTI
True_Asthma,0,4,5,4,2,0,2,2
True_Bronchiectasis,0,0,0,0,0,0,0,0
True_Bronchiolitis,0,0,0,0,0,0,0,0
True_COPD,0,0,3,0,0,0,3,0
True_Healthy,0,0,0,0,0,0,0,0
True_LRTI,0,0,0,0,0,0,0,0
True_Pneumonia,0,0,0,0,0,0,0,3
True_URTI,0,0,0,0,0,0,0,0



C1 results saved to: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/evaluation
Experiment B evaluation artifacts were not modified.


In [113]:
import json
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

B_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_b_feature_cache"
    / "experiment_b_v1_strict"
)

C1_CACHE_DIR = (
    PROJECT_ROOT
    / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c1_icbhi_only"
)

C1_TEST_CACHE_DIR = C1_CACHE_DIR / "test_features_c1_normalized"
C1_TEST_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_KEYS = ["logmel", "mfcc", "chroma"]

# Load combined-source and C1 normalization statistics.
with open(B_CACHE_DIR / "normalization_stats.json", "r") as f:
    combined_stats = json.load(f)

with open(C1_CACHE_DIR / "normalization_stats.json", "r") as f:
    c1_stats = json.load(f)

for split in ["icbhi_test", "fraiwan_test"]:
    source_path = B_CACHE_DIR / f"{split}_features.npz"
    destination_path = C1_TEST_CACHE_DIR / f"{split}_features.npz"

    source = np.load(source_path, allow_pickle=True)

    normalized_features = {}

    for key in FEATURE_KEYS:
        # Recover approximate raw values from combined normalization.
        raw = (
            source[key].astype(np.float32)
            * float(combined_stats[key]["std"])
            + float(combined_stats[key]["mean"])
        )

        # Apply C1 training-only normalization.
        normalized_features[key] = (
            (
                raw - float(c1_stats[key]["mean"])
            ) / float(c1_stats[key]["std"])
        ).astype(np.float32)

        assert np.isfinite(normalized_features[key]).all()

    np.savez_compressed(
        destination_path,
        **normalized_features,
        labels=source["labels"].astype(np.int64),
        paths=source["paths"],
    )

    print(
        f"{split}: saved {len(source['labels'])} windows to "
        f"{destination_path}"
    )

print("\nC1-consistent test feature caches created.")
print("Existing test results were not overwritten.")
print(
    "Note: raw features were approximately reconstructed from "
    "the existing normalized arrays."
)

icbhi_test: saved 1667 windows to /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/test_features_c1_normalized/icbhi_test_features.npz
fraiwan_test: saved 114 windows to /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/test_features_c1_normalized/fraiwan_test_features.npz

C1-consistent test feature caches created.
Existing test results were not overwritten.
Note: raw features were approximately reconstructed from the existing normalized arrays.


In [115]:
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

BASE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

B_CACHE_DIR = (
    BASE_DIR / "experiment_b_feature_cache" / "experiment_b_v1_strict"
)

C1_DIR = BASE_DIR / "experiment_c1_icbhi_only"
C1_TEST_CACHE_DIR = C1_DIR / "test_features_c1_normalized"

OUTPUT_DIR = C1_DIR / "evaluation_c1_normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)
print("Output directory:", OUTPUT_DIR)

# --------------------------------------------------
# 2. Load the best C1 checkpoint
# --------------------------------------------------
checkpoint_path = (
    C1_DIR / "checkpoints" / "experiment_c1_icbhi_only_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

# Reuse the architecture already defined in your notebook.
c1_test_model = LightweightMultiFeatureCNN().to(DEVICE)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    c1_test_model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    c1_test_model.load_state_dict(checkpoint["state_dict"])
else:
    c1_test_model.load_state_dict(checkpoint)

c1_test_model.eval()
print("Loaded C1 best checkpoint:", checkpoint_path)

# --------------------------------------------------
# 3. Evaluate each independent test set
# --------------------------------------------------
summary_rows = []

for split in ["icbhi_test", "fraiwan_test"]:
    print("\n" + "=" * 70)
    print("Evaluating:", split)
    print("=" * 70)

    feature_path = C1_TEST_CACHE_DIR / f"{split}_features.npz"
    metadata_path = B_CACHE_DIR / f"{split}_metadata.csv"

    if not feature_path.exists():
        raise FileNotFoundError(f"Missing features: {feature_path}")
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing metadata: {metadata_path}")

    test_dataset = RespiratoryFeatureDataset(
        str(feature_path),
        augment=False,
    )
    test_metadata = pd.read_csv(metadata_path)

    if len(test_dataset) != len(test_metadata):
        raise ValueError(
            f"Window/metadata mismatch for {split}: "
            f"{len(test_dataset)} vs {len(test_metadata)}"
        )

    # Use the notebook's corrected recording-level evaluator.
    result = evaluate_recording_level(
        c1_test_model,
        test_dataset,
        test_metadata,
        DEVICE,
    )

    y_true = np.asarray(result["recording_labels"], dtype=int)
    y_pred = np.asarray(result["recording_predictions"], dtype=int)

    if len(y_true) != len(y_pred):
        raise ValueError("Recording labels and predictions have different lengths.")

    present_labels = sorted(np.unique(y_true).tolist())

    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1_supported = f1_score(
        y_true,
        y_pred,
        labels=present_labels,
        average="macro",
        zero_division=0,
    )
    macro_f1_all = f1_score(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        average="macro",
        zero_division=0,
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        zero_division=0,
    )
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
    )

    summary_rows.append({
        "experiment": "C1_ICBHI_only",
        "test_split": split,
        "recordings": len(y_true),
        "accuracy": acc,
        "balanced_accuracy_supported_classes": balanced_acc,
        "macro_f1_supported_classes": macro_f1_supported,
        "macro_f1_all_8_classes": macro_f1_all,
    })

    print("Recordings:", len(y_true))
    print(f"Accuracy: {acc:.4f}")
    print(f"Balanced accuracy (supported classes): {balanced_acc:.4f}")
    print(f"Macro-F1 (supported classes): {macro_f1_supported:.4f}")
    print(f"Macro-F1 (all 8 classes): {macro_f1_all:.4f}")

    print("\nClassification report:")
    print(report_text)

    print("Confusion matrix (true rows, predicted columns):")
    print(pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ))

    # Save recording-level predictions.
    prediction_df = pd.DataFrame({
        "recording_key": result["recording_keys"],
        "true_label_index": y_true,
        "predicted_label_index": y_pred,
        "true_class": [CLASS_NAMES[i] for i in y_true],
        "predicted_class": [CLASS_NAMES[i] for i in y_pred],
    })

    if "recording_probabilities" in result:
        probabilities = np.asarray(result["recording_probabilities"])
        if probabilities.ndim == 2 and probabilities.shape[0] == len(y_true):
            for class_index, class_name in enumerate(CLASS_NAMES):
                prediction_df[f"prob_{class_name}"] = probabilities[:, class_index]

    prediction_df.to_csv(
        OUTPUT_DIR / f"{split}_recording_predictions.csv",
        index=False,
    )

    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    ).to_csv(OUTPUT_DIR / f"{split}_confusion_matrix.csv")

    with open(OUTPUT_DIR / f"{split}_classification_report.json", "w") as f:
        json.dump(report_dict, f, indent=2)

# --------------------------------------------------
# 4. Save overall summary and evaluation caveat
# --------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)

with open(OUTPUT_DIR / "evaluation_notes.txt", "w") as f:
    f.write(
        "Experiment C1 ICBHI-only evaluation.\n"
        "Test features were transformed using C1-specific normalization statistics.\n"
        "Important caveat: raw features were approximately reconstructed from the "
        "existing combined-normalized cache, then re-normalized. This is not a "
        "fresh extraction from original audio and may introduce small numerical "
        "differences. Treat results accordingly and document this in the research.\n"
    )

print("\n" + "=" * 70)
print("C1 NORMALIZED TEST SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nSaved corrected evaluation artifacts to:")
print(OUTPUT_DIR)
print("\nCaveat: feature recovery was approximate; these results are not yet a fresh-audio validation.")

Device: mps
Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/evaluation_c1_normalized
Loaded C1 best checkpoint: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only/checkpoints/experiment_c1_icbhi_only_best.pt

Evaluating: icbhi_test


FileNotFoundError: Missing metadata: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict/icbhi_test_metadata.csv

In [116]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

BASE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
)

for folder in [
    BASE_DIR / "experiment_b_feature_cache" / "experiment_b_v1_strict",
    BASE_DIR / "experiment_c1_icbhi_only",
]:
    print(f"\n{'=' * 70}\n{folder}")
    if folder.exists():
        for path in sorted(folder.iterdir()):
            if path.is_file() and (
                "metadata" in path.name.lower()
                or "manifest" in path.name.lower()
                or "recording" in path.name.lower()
                or "feature" in path.name.lower()
            ):
                print(path.name)
            elif path.is_dir() and (
                "metadata" in path.name.lower()
                or "recording" in path.name.lower()
            ):
                print(f"[DIR] {path.name}")
    else:
        print("Folder not found.")

print("\nSearching for test metadata CSV files...")
for path in BASE_DIR.rglob("*.csv"):
    name = path.name.lower()
    if "metadata" in name and ("test" in name or "icbhi" in name or "fraiwan" in name):
        print(path.relative_to(BASE_DIR))


/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_b_feature_cache/experiment_b_v1_strict
fraiwan_test_features.npz
fraiwan_test_window_metadata.csv
icbhi_test_features.npz
icbhi_test_window_metadata.csv
inner_train_recordings.csv
train_features.npz
train_window_metadata.csv
validation_features.npz
validation_recordings.csv
validation_window_metadata.csv

/Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/combined_icbhi_fraiwan_hf_lung_v1/experiment_c1_icbhi_only
train_features.npz
train_window_metadata.csv
validation_features.npz
validation_window_metadata.csv

Searching for test metadata CSV files...
experiment_b_feature_cache/fraiwan_test_window_metadata.csv
experiment_b_feature_cache/icbhi_test_window_metadata.csv
experiment_b_feature_cache/experiment_b_v1_strict/fraiwan_test_window_metadata.csv
experiment_b_feature_cache/experiment_b_v1_strict/icbhi_test_window_metadata.csv


In [118]:
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

C1_TEST_CACHE_DIR = (
    PROJECT_ROOT / "data" / "processed" / "deep_learning"
    / "combined_icbhi_fraiwan_hf_lung_v1"
    / "experiment_c1_icbhi_only"
    / "test_features_c1_normalized"
)

test_path = C1_TEST_CACHE_DIR / "icbhi_test_features.npz"

test_dataset = RespiratoryFeatureDataset(
    str(test_path),
    augment=False,
)

print("Dataset length:", len(test_dataset))

sample = test_dataset[0]

print("\nShapes returned by test_dataset[0]:")
for name, value in zip(
    ["Log-Mel", "MFCC", "Chroma", "Label"],
    sample,
):
    if torch.is_tensor(value):
        print(f"{name}: shape={tuple(value.shape)}, dtype={value.dtype}")
    else:
        print(f"{name}: type={type(value)}, value={value}")

print("\nModel branch architecture:")
print(c1_test_model.logmel_branch)
print("\nExpected input format: [batch, channel, frequency, time]")

Dataset length: 1667

Shapes returned by test_dataset[0]:
Log-Mel: shape=(1, 64, 157), dtype=torch.float32
MFCC: shape=(1, 13, 157), dtype=torch.float32
Chroma: shape=(1, 12, 157), dtype=torch.float32
Label: shape=(), dtype=torch.int64

Model branch architecture:
CompactFeatureBranch(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): DepthwiseSeparableBlock(
      (block): Sequential(
        (0): Conv2d(8, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8, bias=False)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2

In [119]:
import inspect

print(inspect.getsource(evaluate_recording_level))

@torch.no_grad()
def evaluate_recording_level(model, loader, metadata, device):
    model.eval()

    all_probs = []
    all_labels = []

    for logmel, mfcc, chroma, labels in loader:
        logits = model(
            logmel.to(device),
            mfcc.to(device),
            chroma.to(device),
        )

        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    assert all_probs, "The evaluation loader is empty."

    window_probs = np.concatenate(all_probs, axis=0)
    window_labels = np.concatenate(all_labels, axis=0)

    assert len(window_probs) == len(metadata), (
        "Window probabilities and metadata have different lengths."
    )
    assert len(window_labels) == len(metadata), (
        "Window labels and metadata have different lengths."
    )
    assert np.array_equal(
        window_labels,
        metadata["label_index"].to_numpy(dtype=int),
    ), "Loader labels do not match the

In [120]:
import numpy as np
import torch

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score


@torch.no_grad()
def evaluate_recording_level_batched(model, loader, metadata, device):
    model.eval()

    all_probs = []
    all_labels = []

    for logmel, mfcc, chroma, labels in loader:
        logits = model(
            logmel.to(device),
            mfcc.to(device),
            chroma.to(device),
        )

        probs = torch.softmax(logits, dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    assert all_probs, "The evaluation loader is empty."

    window_probs = np.concatenate(all_probs, axis=0)
    window_labels = np.concatenate(all_labels, axis=0)

    assert len(window_probs) == len(metadata), (
        "Window probabilities and metadata have different lengths."
    )
    assert len(window_labels) == len(metadata), (
        "Window labels and metadata have different lengths."
    )
    assert np.array_equal(
        window_labels,
        metadata["label_index"].to_numpy(dtype=int),
    ), "Loader labels do not match the supplied metadata."

    keys = list(zip(
        metadata["source_dataset"].astype(str),
        metadata["source_recording_id"].astype(str),
    ))

    unique_keys = list(dict.fromkeys(keys))
    key_to_idx = {key: i for i, key in enumerate(unique_keys)}

    recording_probs = np.zeros(
        (len(unique_keys), len(CLASS_NAMES)),
        dtype=np.float64,
    )
    recording_labels = np.full(len(unique_keys), -1, dtype=np.int64)
    counts = np.zeros(len(unique_keys), dtype=np.int64)

    for i, key in enumerate(keys):
        group_idx = key_to_idx[key]
        label = int(window_labels[i])

        if recording_labels[group_idx] == -1:
            recording_labels[group_idx] = label
        else:
            assert recording_labels[group_idx] == label, (
                f"Inconsistent labels for recording {key}"
            )

        recording_probs[group_idx] += window_probs[i]
        counts[group_idx] += 1

    assert np.all(counts > 0)
    recording_probs /= counts[:, None]

    predictions = recording_probs.argmax(axis=1)
    supported_classes = np.unique(recording_labels)

    return {
        "macro_f1_recording": float(
            f1_score(
                recording_labels,
                predictions,
                labels=supported_classes,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy_recording": float(
            accuracy_score(recording_labels, predictions)
        ),
        "recording_labels": recording_labels,
        "recording_predictions": predictions,
        "recording_probabilities": recording_probs,
        "recording_keys": unique_keys,
    }


print("Separate batched evaluator defined successfully.")
print("Original evaluator remains unchanged.")

Separate batched evaluator defined successfully.
Original evaluator remains unchanged.


In [121]:
from torch.utils.data import DataLoader
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

OUTPUT_DIR = C1_DIR / "evaluation_c1_normalized"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []

for split in ["icbhi_test", "fraiwan_test"]:
    print("\n" + "=" * 70)
    print("Evaluating:", split)
    print("=" * 70)

    feature_path = C1_TEST_CACHE_DIR / f"{split}_features.npz"
    metadata_path = B_CACHE_DIR / f"{split}_window_metadata.csv"

    test_dataset = RespiratoryFeatureDataset(
        str(feature_path),
        augment=False,
    )
    test_metadata = pd.read_csv(metadata_path)

    if len(test_dataset) != len(test_metadata):
        raise ValueError(
            f"{split}: {len(test_dataset)} windows but "
            f"{len(test_metadata)} metadata rows."
        )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=0,
    )

    result = evaluate_recording_level_batched(
        c1_test_model,
        test_loader,
        test_metadata,
        DEVICE,
    )

    y_true = np.asarray(result["recording_labels"], dtype=int)
    y_pred = np.asarray(result["recording_predictions"], dtype=int)
    present_labels = sorted(np.unique(y_true).tolist())

    metrics = {
        "experiment": "C1_ICBHI_only",
        "test_split": split,
        "recordings": len(y_true),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy_supported_classes": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1_supported_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_8_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(len(CLASS_NAMES))),
                average="macro",
                zero_division=0,
            )
        ),
    }
    summary_rows.append(metrics)

    print("Recordings:", len(y_true))
    for metric_name, metric_value in metrics.items():
        if isinstance(metric_value, float):
            print(f"{metric_name}: {metric_value:.4f}")

    report_text = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        zero_division=0,
    )
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(CLASS_NAMES))),
    )

    print("\nClassification report:")
    print(report_text)
    print("Confusion matrix (true rows, predicted columns):")
    print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

    prediction_df = pd.DataFrame({
        "recording_key": [
            f"{source}::{recording_id}"
            for source, recording_id in result["recording_keys"]
        ],
        "true_label_index": y_true,
        "predicted_label_index": y_pred,
        "true_class": [CLASS_NAMES[i] for i in y_true],
        "predicted_class": [CLASS_NAMES[i] for i in y_pred],
    })

    probabilities = np.asarray(result["recording_probabilities"])
    for i, class_name in enumerate(CLASS_NAMES):
        prediction_df[f"prob_{class_name}"] = probabilities[:, i]

    prediction_df.to_csv(
        OUTPUT_DIR / f"{split}_recording_predictions.csv",
        index=False,
    )

    pd.DataFrame(
        cm, index=CLASS_NAMES, columns=CLASS_NAMES
    ).to_csv(OUTPUT_DIR / f"{split}_confusion_matrix.csv")

    with open(
        OUTPUT_DIR / f"{split}_classification_report.json", "w"
    ) as f:
        json.dump(report_dict, f, indent=2)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)

with open(OUTPUT_DIR / "evaluation_notes.txt", "w") as f:
    f.write(
        "C1 ICBHI-only evaluation with C1-specific test normalization.\n"
        "Recording predictions are calculated by averaging window probabilities.\n"
        "Caveat: test features were approximately reconstructed from existing "
        "combined-normalized features and then re-normalized, rather than "
        "freshly extracted from original audio. Interpret results accordingly.\n"
    )

print("\n" + "=" * 70)
print("FINAL C1 NORMALIZED TEST SUMMARY")
print("=" * 70)
print(summary_df.to_string(index=False))
print("\nSaved evaluation artifacts to:", OUTPUT_DIR)


Evaluating: icbhi_test


/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Recordings: 394
accuracy: 0.7183
balanced_accuracy_supported_classes: 0.3077
macro_f1_supported_classes: 0.3025
macro_f1_all_8_classes: 0.1891

Classification report:
                precision    recall  f1-score   support

        Asthma       0.00      0.00      0.00         0
Bronchiectasis       0.00      0.00      0.00         2
 Bronchiolitis       0.00      0.00      0.00         6
          COPD       1.00      0.76      0.86       362
       Healthy       0.55      0.35      0.43        17
          LRTI       0.00      0.00      0.00         0
     Pneumonia       0.00      0.00      0.00         0
          URTI       0.15      0.43      0.22         7

      accuracy                           0.72       394
     macro avg       0.21      0.19      0.19       394
  weighted avg       0.94      0.72      0.81       394

Confusion matrix (true rows, predicted columns):
                Asthma  Bronchiectasis  Bronchiolitis  COPD  Healthy  LRTI  \
Asthma               0         

/Users/vs/Sem 5/Respiratory-Disease-Classification/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
